# functions

In [1]:
import pandas as pd
import glob
import os
# Set option to display all columns
pd.set_option('display.max_columns', None)


In [2]:
import pandas as pd
import numpy as np
from urllib.parse import urlparse
import re
import unicodedata

DOI_CORE_RE = re.compile(r"(10\.\d{4,9}/\S+)", re.IGNORECASE)

def _normalize_doi_raw(x: str) -> str | None:
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    s = str(x).strip()
    s = unicodedata.normalize("NFKC", s).lower()
    m = DOI_CORE_RE.search(s)  # extract from URL/“doi:” etc.
    if m:
        return m.group(1)
    if s.startswith("10.") and "/" in s:
        return s
    return None

def clean_preprint_fields(df: pd.DataFrame, *, numeric_keep: int = 2, add_bucket: bool = True) -> pd.DataFrame:
    """Clean + enrich preprint fields.
    - Builds doi_prefix_first_token
    - If first token is purely numeric, keeps only its first `numeric_keep` digits
    - Optionally adds a bucketing column for grouping
    """
    df = df.drop_duplicates().copy()

    # # --- gold_server_name
    # df["gold_server_name"] = (
    #     df.get("institution_name")
    #       .fillna(df.get("group_title"))
    #       .fillna(df.get("publisher"))
    # )

    # --- Normalize DOI
    if "doi" in df.columns:
        df["doi_lc"] = df["doi"].map(_normalize_doi_raw)
    elif "parent_doi" in df.columns:
        df["doi_lc"] = df["parent_doi"].map(_normalize_doi_raw)
    else:
        df["doi_lc"] = pd.Series(pd.NA, index=df.index, dtype="object")

    # --- prefix as given
    if "prefix" in df.columns:
        df["prefix_lc"] = df["prefix"].astype(str).str.strip().str.lower()
        df.loc[df["prefix_lc"].isin(["", "nan", "none"]), "prefix_lc"] = pd.NA
    else:
        df["prefix_lc"] = pd.Series(pd.NA, index=df.index, dtype="object")

    # --- Extract prefix/suffix from normalized DOI
    doi_parts = df["doi_lc"].str.extract(r"^(10\.\d{4,9})/(.+)$")
    df["doi_prefix_from_text"] = doi_parts[0]
    df["doi_suffix"] = doi_parts[1]
    df["prefix_lc"] = df["prefix_lc"].where(df["prefix_lc"].notna(), df["doi_prefix_from_text"])

    # --- Build first segment of suffix
    starts_with_letter = df["doi_suffix"].str.match(r"^[a-z]", na=False)

    # letters-case: take only leading letters/hyphens; stop before digits or separators
    first_seg_letters = df["doi_suffix"].str.extract(
        r"^([a-z\-]+)(?=\d|[.\-_/:]|$)", expand=False
    )

    # default-case: first chunk before separators (keeps digits)
    first_seg_default = df["doi_suffix"].str.split(r"[.\-_/:\s]", n=1, regex=True).str[0]

    first_seg = pd.Series(
        np.where(starts_with_letter, first_seg_letters, first_seg_default),
        index=df.index,
        dtype="object"
    )

    # fallback: permissive token if still NA
    need_fallback = first_seg.isna() & df["doi_suffix"].notna()
    first_seg.loc[need_fallback] = df.loc[need_fallback, "doi_suffix"].str.extract(r"^([a-z0-9\-]+)", expand=False)

    # --- NEW: compress purely numeric first tokens to first `numeric_keep` digits
    if numeric_keep and numeric_keep > 0:
        numeric_only = first_seg.str.fullmatch(r"\d+", na=False)
        first_seg.loc[numeric_only] = first_seg.loc[numeric_only].str[:numeric_keep]

    # --- Assemble final token
    df["doi_prefix_first_token"] = pd.Series(pd.NA, index=df.index, dtype="object")
    ok = df["prefix_lc"].notna() & first_seg.notna() & (first_seg.astype(str) != "")
    df.loc[ok, "doi_prefix_first_token"] = df.loc[ok, "prefix_lc"].astype(str) + "/" + first_seg.loc[ok].astype(str)

    # --- Optional bucket for grouping/plots (mirrors the compressed numeric rule)
    if add_bucket:
        df["doi_prefix_bucket_2d"] = df["doi_prefix_first_token"]

    # --- Domains
    def domain_and_first_path(u):
        try:
            parsed = urlparse(str(u).lower())
            host = parsed.netloc
            if host.startswith("www."):
                host = host[4:]
            parts = re.split(r"[/=]", parsed.path)
            first_part = parts[1] if len(parts) > 1 and parts[1] else None
            return f"{host}/{first_part}" if host and first_part else (host or None)
        except Exception:
            return None

    if "landing_page_url" in df.columns:
        df["primary_domain"] = df["landing_page_url"].apply(lambda u: urlparse(str(u)).netloc.lower().replace("www.", "") if pd.notna(u) else None)
        df["primary_domain_extend"] = df["landing_page_url"].apply(domain_and_first_path)
    elif "parent_url" in df.columns:
        df["primary_domain"] = df["parent_url"].apply(lambda u: urlparse(str(u)).netloc.lower().replace("www.", "") if pd.notna(u) else None)
        df["primary_domain_extend"] = df["parent_url"].apply(domain_and_first_path)
    else:
        df["primary_domain"] = pd.Series(pd.NA, index=df.index, dtype="object")
        df["primary_domain_extend"] = pd.Series(pd.NA, index=df.index, dtype="object")

    # --- Dates → year
    if "posted_date" in df.columns:
        df["posted_date"] = pd.to_datetime(df["posted_date"], errors="coerce")
        df["year"] = df["posted_date"].dt.year

    return df

# # ----- usage
# df = clean_preprint_fields(df, numeric_keep=2, add_bucket=True)
# print(df.shape)

In [3]:
import pandas as pd
import glob
import os

def get_server_data(server_name, base_path=r"/mnt/c/SCHOLCOMMLAB/APPs/preprint-harvester/data/by_server/"):
    """
    Loads, cleans, and summarizes server metadata with clear visual formatting.
    """
    
    # 1. CONSTRUCTION & LOADING
    folder_path = os.path.join(base_path, server_name)
    parquet_files = glob.glob(os.path.join(folder_path, "*.parquet"))
    
    if not parquet_files:
        print(f"\n[!] ERROR: No parquet files found for '{server_name}'")
        print(f"    Path searched: {folder_path}\n")
        return None, None

    raw_df = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)
    raw_df = raw_df.drop_duplicates('record_id')
    
    # 2. STATUS HEADER
    print("\n" + "="*50)
    print(f" SERVER ANALYSIS: {server_name.upper()}")
    print("="*50)
    print(f"  > Files found:    {len(parquet_files)}")
    print(f"  > Raw records:    {len(raw_df)}")
    
    
    # 3. CLEANING STEP
    df = clean_preprint_fields(raw_df, numeric_keep=2, add_bucket=True)
    print(f"  > Cleaned shape:  {df.shape}")
    print(f"  > Unique DOIs:    {df['doi'].nunique()}")
    print("-" * 50)


    # 4. SUMMARY GENERATION
    cols_to_summarize = [
        "doi_prefix_first_token", "primary_domain", "prefix", "member_id", 
        "publisher", "container_title", "institution_name", "group_title", 
        "issn", "type_backend_raw", "subtype_backend_raw"
    ]
    
    summary = {}
    
    for col in cols_to_summarize:
        if col in df.columns:
            # Store the data
            summary[col] = df[col].value_counts(dropna=False)
            
            # Print with plenty of space
            print(f"\nTOP VALUES FOR: {col.upper()}")
            print("-" * 30)
            print(summary[col].head(10))
            print("\n")
        else:
            summary[col] = "Column not found"
            print(f"\n[!] Column '{col}' not found in DataFrame.\n")

    print("="*50)
    print(f" END OF SUMMARY FOR {server_name.upper()}")
    print("="*50 + "\n")

    return df, summary

# ----- Execution example
# elife_df, elife_summary = get_server_data("eLife")

In [4]:
import pandas as pd
import os

def analyze_parent_data(server_name, parent_df):
    """
    Filters the consolidated parent DataFrame for a specific server 
    and provides a summary of versioning and metadata.
    """
    
    # 1. FILTERING
    # Filter first to minimize memory usage during cleaning
    df_filtered = parent_df[parent_df['parent_server_name'] == server_name].copy()
    
    # Check if empty BEFORE running cleaning logic
    if df_filtered.empty:
        print(f"\n" + "!"*60)
        print(f" [!] ERROR: No records found for '{server_name}'")
        print(f" Check if the name matches 'parent_server_name' exactly.")
        print("!"*60 + "\n")
        return None, None

    # 2. CLEANING
    # Running cleaning on the filtered subset
    df = clean_preprint_fields(df_filtered, numeric_keep=2, add_bucket=True)
    
    # 3. STATUS HEADER
    print("\n" + "="*65)
    print(f" CONSOLIDATED ANALYSIS: {server_name.upper()}")
    print("="*65)
    print(f"  > Total Parent Groups:     {len(df):,}")
    print(f"  > Total Versions Found:    {int(df['total_versions'].sum()):,}")
    print(f"  > Unique Parent DOIs:      {df['parent_doi'].nunique():,}")
    print("-" * 65)

    # 4. VERSIONING SUMMARY
    print(f"\n VERSIONING DISTRIBUTION for {server_name}:")
    print("-" * 35)
    v_counts = df['total_versions'].value_counts().sort_index()
    for version_num, count in v_counts.items():
        print(f"  {int(version_num)} version(s):".ljust(20) + f"{count:,} groups")
    
    print(f"\n  Average versions per parent: {df['total_versions'].mean():.2f}")

    # 5. CROSS-SERVER MAPPING
    print(f"\n MOST RECENT DESTINATIONS (Migration):")
    print("-" * 35)
    destinations = df['most_recent_version_servers'].value_counts().head(10)
    print(destinations.to_string())

    # 6. METADATA SUMMARY
    cols_to_summarize = [
        "doi_prefix_first_token", "primary_domain"] # "servers_with_counts", "parent_year_first_seen", 
    summary = {}
    
    for col in cols_to_summarize:
        if col in df.columns:
            summary[col] = df[col].value_counts(dropna=False)
            print(f"\n TOP VALUES FOR: {col.upper()}")
            print("-" * 35)
            print(summary[col].head(10).to_string())
            print("\n")

    print("="*65)
    print(f" COMPLETED: {server_name.upper()}")
    print("="*65 + "\n")

    return df, summary

# ----- Execution Example
# parent_path = "outputs_new/parent/parent_version_index_full.parquet"
# full_parent_df = pd.read_parquet(parent_path)

# elife_parent_df, elife_parent_summary = analyze_parent_data("eLife", full_parent_df)

# Load parent data

In [5]:
# ----- Execution Example
# 1. Load the big file once
long_path = "outputs_new/parent/dedupe_clusters_long_full.parquet"
parent_path = "outputs_new/parent/parent_version_index_full.parquet"

full_parent_df = pd.read_parquet(parent_path)
full_dataset_df = pd.read_parquet(long_path)
# 2. Analyze a specific server within that file
# arxiv_parent_df, arxiv_parent_summary = analyze_parent_data("arXiv", full_parent_df)
# advance_parent_df, advance_parent_summary = analyze_parent_data("Advance", full_parent_df)

# Advance

In [6]:
Advance_df, Advance_summary = get_server_data("Advance")


 SERVER ANALYSIS: ADVANCE
  > Files found:    1
  > Raw records:    4401
  > Cleaned shape:  (4401, 90)
  > Unique DOIs:    4401
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31124/advance    4401
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
advance.sagepub.com    4392
authorea.com              7
techrxiv.org              2
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31124    4401
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
179    4401
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
SAGE Publications    4401
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    4401
Name: count, dtype: int64



TOP VALUES FOR: INSTITU

In [7]:
Advance_df

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
0,crossref::10.31124/advance.7037639,Advance,crossref,10.31124/advance.7037639,10.31124/advance.7037639,https://doi.org/10.31124/advance.7037639,https://advance.sagepub.com/articles/The_Priva...,https://advance.sagepub.com/articles/The_Priva...,10.31124,179,None,None,crossref,SAGE Publications,None,None,None,None,The Privatization of Security and the Emergenc...,None,None,None,None,posted-content,preprint,preprint,None,True,2018-09-04,2018-09-04,2018-09-04,2022-03-30,None,2018-09-04,None,2018-09-04,None,2018,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<jats:p>&lt;p&gt;This study interrogates the p...,This study interrogates the participation of p...,"[{""URL"": ""https://ndownloader.figshare.com/fil...",None,"Chinwokwu, Eke; Igbo, Emmanuel",None,None,"[{""affiliation"": [], ""family"": ""Chinwokwu"", ""g...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.7037639"", ""URL"": ""ht...",10.31124/advance.7037639,10.31124,10.31124,advance.7037639,10.31124/advance,10.31124/advance,advance.sagepub.com,advance.sagepub.com/articles
1,crossref::10.31124/advance.7038500,Advance,crossref,10.31124/advance.7038500,10.31124/advance.7038500,https://doi.org/10.31124/advance.7038500,https://advance.sagepub.com/articles/Unmet_nee...,https://advance.sagepub.com/articles/Unmet_nee...,10.31124,179,None,None,crossref,SAGE Publications,None,None,None,None,Unmet needs of ageing transgender and non-bina...,None,None,None,None,posted-content,preprint,preprint,None,True,2018-09-04,2018-09-04,2018-09-04,2025-02-21,None,2018-09-04,None,2018-09-04,None,2018,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<jats:p>An view of literature on transgender a...,An view of literature on transgender and non-b...,"[{""URL"": ""https://ndownloader.figshare.com/fil...",None,"Broadway-Horner, Matt",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-8834-7...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.7038500"", ""URL"": ""ht...",10.31124/advance.7038500,10.31124,10.31124,advance.7038500,10.31124/advance,10.31124/advance,advance.sagepub.com,advance.sagepub.com/articles
2,crossref::10.31124/advance.7038500.v1,Advance,crossref,10.31124/advance.7038500.v1,10.31124/advance.7038500.v1,https://doi.org/10.31124/advance.7038500.v1,https://advance.sagepub.com/articles/Unmet_nee...,https://advance.sa

In [8]:
Advance_df.columns

Index(['record_id', 'server_name', 'backend', 'source_work_id', 'doi',
       'doi_url', 'landing_page_url', 'url_best', 'prefix', 'member_id',
       'client_id', 'provider_id', 'source_registry', 'publisher',
       'container_title', 'institution_name', 'group_title', 'issn', 'title',
       'original_title', 'short_title', 'subtitle', 'language',
       'type_backend_raw', 'subtype_backend_raw', 'type_canonical',
       'is_paratext', 'is_preprint_candidate', 'date_created', 'date_posted',
       'date_deposited', 'date_indexed', 'date_updated', 'date_issued',
       'date_registered', 'date_published', 'date_published_online',
       'publication_year', 'date_published_source', 'date_posted_source',
       'is_oa', 'oa_status', 'license', 'license_url_best', 'abstract_raw',
       'abstract_text', 'links_json_best', 'fulltext_pdf_url', 'authors_flat',
       'institutions_flat', 'countries_flat', 'authors_json',
       'contributors_json', 'editors_json', 'funders_json', 'funders_

In [9]:
df=Advance_df.copy()

In [10]:
df[df['primary_domain']=='authorea.com']

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
3980,crossref::10.31124/advance.172975247.76617035/v1,Advance,crossref,10.31124/advance.172975247.76617035/v1,10.31124/advance.172975247.76617035/v1,https://doi.org/10.31124/advance.172975247.766...,https://www.authorea.com/users/749101/articles...,https://www.authorea.com/users/749101/articles...,10.31124,179,None,None,crossref,SAGE Publications,None,Advance,Preprints,None,The Relationships between the Perception of Ph...,None,None,None,None,posted-content,preprint,preprint,None,True,2024-10-24,2024-10-24,2024-10-24,2024-10-25,None,2024-10-24,None,2024-10-24,None,2024,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,None,None,None,None,"Perlman, Amotz",0,None,"[{""affiliation"": [{""name"": ""0""}], ""family"": ""P...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.172975247.76617035/v...",10.31124/advance.172975247.76617035/v1,10.31124,10.31124,advance.172975247.76617035/v1,10.31124/advance,10.31124/advance,authorea.com,authorea.com/users
3987,crossref::10.31124/advance.173086931.17011759/v1,Advance,crossref,10.31124/advance.173086931.17011759/v1,10.31124/advance.173086931.17011759/v1,https://doi.org/10.31124/advance.173086931.170...,https://www.authorea.com/users/811231/articles...,https://www.authorea.com/users/811231/articles...,10.31124,179,None,None,crossref,SAGE Publications,None,Advance,Preprints,None,Segment-Level Traffic Volume Estimation Incorp...,None,None,None,None,posted-content,preprint,preprint,None,True,2024-11-06,2024-11-06,2024-11-06,2025-05-14,None,2024-11-06,None,2024-11-06,None,2024,issued_date,posted_date,None,None,None,None,None,None,None,None,"Morshed, Syed Ahnaf; Amine, Kamar; Hadi, Mohammed",None,None,"[{""ORCID"": ""https://orcid.org/0000-0003-3193-3...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.173086931.17011759/v...",10.31124/advance.173086931.17011759/v1,10.31124,10.31124,advance.173086931.17011759/v1,10.31124/advance,10.31124/advance,authorea.com,authorea.com/users
4029,crossref::10.31124/advance.173873930.08099093/v1,Advance,crossref,10.31124/advance.173873930.08099093/v1,10.31124/advance.173873930.08099093/v1,https://doi.org/10.31124/advance.173873930.080...,https://www.authorea.com/users/887234/articles...,https://www.authorea.com/users/887234/articles...,10.31124,179,None,None,crossref,SAGE Publications,None,Advance,Preprints,None,Defining Martial Law: Introducing the EmPower ...,None,None

In [11]:
df[df['primary_domain']=='authorea.com']['doi'].tolist()

['10.31124/advance.172975247.76617035/v1',
 '10.31124/advance.173086931.17011759/v1',
 '10.31124/advance.173873930.08099093/v1',
 '10.31124/advance.173883835.54601691/v1',
 '10.31124/advance.173892076.63256740/v1',
 '10.31124/advance.173952912.28451956/v1',
 '10.31124/advance.173883514.46750785/v1']

In [12]:
df[df['primary_domain']=='techrxiv.org']

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
3821,crossref::10.31124/advance.171330081.13982256/v1,Advance,crossref,10.31124/advance.171330081.13982256/v1,10.31124/advance.171330081.13982256/v1,https://doi.org/10.31124/advance.171330081.139...,https://www.techrxiv.org/users/678900/articles...,https://www.techrxiv.org/users/678900/articles...,10.31124,179,None,None,crossref,SAGE Publications,None,Advance,Preprints,None,Test document,None,None,None,None,posted-content,preprint,preprint,None,True,2024-04-16,2024-04-16,2024-04-16,2025-05-14,None,2024-04-16,None,2024-04-16,None,2024,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,None,None,None,None,"Meyer, Carol Anne",None,None,"[{""ORCID"": ""https://orcid.org/0000-0003-2443-2...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.171330081.13982256/v...",10.31124/advance.171330081.13982256/v1,10.31124,10.31124,advance.171330081.13982256/v1,10.31124/advance,10.31124/advance,techrxiv.org,techrxiv.org/users
3822,crossref::10.31124/advance.171330568.89328065/v1,Advance,crossref,10.31124/advance.171330568.89328065/v1,10.31124/advance.171330568.89328065/v1,https://doi.org/10.31124/advance.171330568.893...,https://www.techrxiv.org/users/678900/articles...,https://www.techrxiv.org/users/678900/articles...,10.31124,179,None,None,crossref,SAGE Publications,None,Advance,Preprints,None,This is a test preprint,None,None,None,None,posted-content,preprint,preprint,None,True,2024-04-16,2024-04-16,2024-04-16,2025-05-14,None,2024-04-16,None,2024-04-16,None,2024,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,"<jats:p id=""p1"">This is the Abstract of my tes...",This is the Abstract of my test submission,None,None,"Meyer, Carol Anne",None,None,"[{""ORCID"": ""https://orcid.org/0000-0003-2443-2...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.171330568.89328065/v...",10.31124/advance.171330568.89328065/v1,10.31124,10.31124,advance.171330568.89328065/v1,10.31124/advance,10.31124/advance,techrxiv.org,techrxiv.org/users


In [13]:
df[df['primary_domain']=='techrxiv.org']['doi'].tolist()

['10.31124/advance.171330081.13982256/v1',
 '10.31124/advance.171330568.89328065/v1']

In [14]:
df[df['institution_name']=='Authorea, Inc.']['doi'].tolist()

['10.31124/advance.24454624.v1', '10.31124/advance.170921771.12975902/v1']

In [15]:
df[df['institution_name'].isna()]

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
0,crossref::10.31124/advance.7037639,Advance,crossref,10.31124/advance.7037639,10.31124/advance.7037639,https://doi.org/10.31124/advance.7037639,https://advance.sagepub.com/articles/The_Priva...,https://advance.sagepub.com/articles/The_Priva...,10.31124,179,None,None,crossref,SAGE Publications,None,None,None,None,The Privatization of Security and the Emergenc...,None,None,None,None,posted-content,preprint,preprint,None,True,2018-09-04,2018-09-04,2018-09-04,2022-03-30,None,2018-09-04,None,2018-09-04,None,2018,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<jats:p>&lt;p&gt;This study interrogates the p...,This study interrogates the participation of p...,"[{""URL"": ""https://ndownloader.figshare.com/fil...",None,"Chinwokwu, Eke; Igbo, Emmanuel",None,None,"[{""affiliation"": [], ""family"": ""Chinwokwu"", ""g...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.7037639"", ""URL"": ""ht...",10.31124/advance.7037639,10.31124,10.31124,advance.7037639,10.31124/advance,10.31124/advance,advance.sagepub.com,advance.sagepub.com/articles
1,crossref::10.31124/advance.7038500,Advance,crossref,10.31124/advance.7038500,10.31124/advance.7038500,https://doi.org/10.31124/advance.7038500,https://advance.sagepub.com/articles/Unmet_nee...,https://advance.sagepub.com/articles/Unmet_nee...,10.31124,179,None,None,crossref,SAGE Publications,None,None,None,None,Unmet needs of ageing transgender and non-bina...,None,None,None,None,posted-content,preprint,preprint,None,True,2018-09-04,2018-09-04,2018-09-04,2025-02-21,None,2018-09-04,None,2018-09-04,None,2018,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<jats:p>An view of literature on transgender a...,An view of literature on transgender and non-b...,"[{""URL"": ""https://ndownloader.figshare.com/fil...",None,"Broadway-Horner, Matt",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-8834-7...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.7038500"", ""URL"": ""ht...",10.31124/advance.7038500,10.31124,10.31124,advance.7038500,10.31124/advance,10.31124/advance,advance.sagepub.com,advance.sagepub.com/articles
2,crossref::10.31124/advance.7038500.v1,Advance,crossref,10.31124/advance.7038500.v1,10.31124/advance.7038500.v1,https://doi.org/10.31124/advance.7038500.v1,https://advance.sagepub.com/articles/Unmet_nee...,https://advance.sa

In [16]:
df[df['institution_name'].isna()]['landing_page_url'].tolist()

['https://advance.sagepub.com/articles/The_Privatization_of_Security_and_the_Emergence_of_Private_Security_Companies_in_Crime_Control_in_Nigeria_docx/7037639',
 'https://advance.sagepub.com/articles/Unmet_needs_of_ageing_transgender_and_non-binary_population_An_Overview/7038500',
 'https://advance.sagepub.com/articles/Unmet_needs_of_ageing_transgender_and_non-binary_population_An_Overview/7038500/1',
 'https://advance.sagepub.com/articles/A_TROG_masked_docUsing_T_R_O_G_to_improve_psychotherapy_engagement_when_working_with_Mild_Learning_Disability_Populations/7038503',
 'https://advance.sagepub.com/articles/A_TROG_masked_docUsing_T_R_O_G_to_improve_psychotherapy_engagement_when_working_with_Mild_Learning_Disability_Populations/7038503/1',
 'https://advance.sagepub.com/articles/Are_We_There_Yet_Understanding_Cultural_Issues_and_Making_them_Central_in_Psychology_and_Psychiatry/7038506/1',
 'https://advance.sagepub.com/articles/Are_We_There_Yet_Understanding_Cultural_Issues_and_Making_them

In [17]:
df[df['institution_name']=='Advance']['landing_page_url'].tolist()

['https://advance.sagepub.com/articles/preprint/Stress_Scale_in_the_Context_of_Online_Learning_among_Junior_High_School_Students_ages_11-17_Development_Validity_and_Reliability/15020103',
 'https://advance.sagepub.com/articles/preprint/Gifted_Programming_Identification_Procedures_A_Hidden_Curriculum/15040629',
 'https://advance.sagepub.com/articles/preprint/Gendered_Justice_The_Impact_of_Gender_on_Criminal_Justice_Policies_and_Legislation_throughout_the_United_Kingdom/15042804',
 'https://advance.sagepub.com/articles/preprint/Maximization_of_Female_Contribution_in_Global_Workforce/15043020',
 'https://advance.sagepub.com/articles/preprint/_Only_I_have_to_help_myself_Indian_Migrant_Workers_Plight_During_COVID-19_Lockdown/15047967',
 'https://advance.sagepub.com/articles/preprint/_Only_I_have_to_help_myself_Indian_Migrant_Workers_Plight_During_COVID-19_Lockdown/15047967/1',
 'https://advance.sagepub.com/articles/preprint/Changes_in_General_and_Specific_Teacher_Self-Efficacy_Related_to_Pr

In [18]:
df['doi'].sort_values().head(60)#.tolist()

354        10.31124/advance.10005662
1779    10.31124/advance.10005662.v1
1780    10.31124/advance.10005662.v2
340        10.31124/advance.10005884
1781    10.31124/advance.10005884.v1
346        10.31124/advance.10007411
1782    10.31124/advance.10007411.v1
342        10.31124/advance.10010381
1783    10.31124/advance.10010381.v1
343        10.31124/advance.10012031
1785    10.31124/advance.10012031.v1
355        10.31124/advance.10026860
1784    10.31124/advance.10026860.v1
390        10.31124/advance.10048160
1768    10.31124/advance.10048160.v1
1787    10.31124/advance.10048160.v2
1786    10.31124/advance.10048160.v3
3715    10.31124/advance.10048160.v4
653        10.31124/advance.10050131
1789    10.31124/advance.10050131.v1
345        10.31124/advance.10050497
1788    10.31124/advance.10050497.v1
347        10.31124/advance.10055363
1790    10.31124/advance.10055363.v1
344        10.31124/advance.10055399
1791    10.31124/advance.10055399.v1
348        10.31124/advance.10096058
1

In [19]:
df[df['subtype_backend_raw']=='other']

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
12,crossref::10.31124/advance.7038497,Advance,crossref,10.31124/advance.7038497,10.31124/advance.7038497,https://doi.org/10.31124/advance.7038497,https://advance.sagepub.com/articles/A_Snapsho...,https://advance.sagepub.com/articles/A_Snapsho...,10.31124,179,None,None,crossref,SAGE Publications,None,None,None,None,A Snapshot of Psycho-social issues in Camp Liv...,None,None,None,None,posted-content,other,other,None,True,2018-09-06,2018-09-06,2018-09-06,2025-02-21,None,2018-09-06,None,2018-09-06,None,2018,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<jats:p>Using survey data to measure the impac...,Using survey data to measure the impact of a d...,"[{""URL"": ""https://ndownloader.figshare.com/fil...",None,"Broadway-Horner, Matt",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-8834-7...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.7038497"", ""URL"": ""ht...",10.31124/advance.7038497,10.31124,10.31124,advance.7038497,10.31124/advance,10.31124/advance,advance.sagepub.com,advance.sagepub.com/articles
13,crossref::10.31124/advance.7038497.v1,Advance,crossref,10.31124/advance.7038497.v1,10.31124/advance.7038497.v1,https://doi.org/10.31124/advance.7038497.v1,https://advance.sagepub.com/articles/A_Snapsho...,https://advance.sagepub.com/articles/A_Snapsho...,10.31124,179,None,None,crossref,SAGE Publications,None,None,None,None,A Snapshot of Psycho-social issues in Camp Liv...,None,None,None,None,posted-content,other,other,None,True,2018-09-06,2018-09-06,2018-09-06,2025-02-21,None,2018-09-06,None,2018-09-06,None,2018,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<jats:p>Using survey data to measure the impac...,Using survey data to measure the impact of a d...,"[{""URL"": ""https://ndownloader.figshare.com/fil...",None,"Broadway-Horner, Matt",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-8834-7...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.7038497.v1"", ""URL"": ...",10.31124/advance.7038497.v1,10.31124,10.31124,advance.7038497.v1,10.31124/advance,10.31124/advance,advance.sagepub.com,advance.sagepub.com/articles


In [20]:
df[df['subtype_backend_raw']=='other']['doi'].tolist()

['10.31124/advance.7038497', '10.31124/advance.7038497.v1']

## observation

it seems like their create the new doi with versions patterns and the old records without version patterns are not available on the main page

In [21]:
df['doi'].sort_values().head(60)#.tolist()


354        10.31124/advance.10005662
1779    10.31124/advance.10005662.v1
1780    10.31124/advance.10005662.v2
340        10.31124/advance.10005884
1781    10.31124/advance.10005884.v1
346        10.31124/advance.10007411
1782    10.31124/advance.10007411.v1
342        10.31124/advance.10010381
1783    10.31124/advance.10010381.v1
343        10.31124/advance.10012031
1785    10.31124/advance.10012031.v1
355        10.31124/advance.10026860
1784    10.31124/advance.10026860.v1
390        10.31124/advance.10048160
1768    10.31124/advance.10048160.v1
1787    10.31124/advance.10048160.v2
1786    10.31124/advance.10048160.v3
3715    10.31124/advance.10048160.v4
653        10.31124/advance.10050131
1789    10.31124/advance.10050131.v1
345        10.31124/advance.10050497
1788    10.31124/advance.10050497.v1
347        10.31124/advance.10055363
1790    10.31124/advance.10055363.v1
344        10.31124/advance.10055399
1791    10.31124/advance.10055399.v1
348        10.31124/advance.10096058
1

In [22]:
# [./] matches either a dot or a slash
# v\d+ matches 'v' followed by one or more digits
# $ ensures this pattern is at the very end of the string
pattern = r'[./]v\d+$'
# pattern = r'v\d+$'
mask = df['doi'].str.contains(pattern, regex=True, na=False)
result = df[mask]
result

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
2,crossref::10.31124/advance.7038500.v1,Advance,crossref,10.31124/advance.7038500.v1,10.31124/advance.7038500.v1,https://doi.org/10.31124/advance.7038500.v1,https://advance.sagepub.com/articles/Unmet_nee...,https://advance.sagepub.com/articles/Unmet_nee...,10.31124,179,None,None,crossref,SAGE Publications,None,None,None,None,Unmet needs of ageing transgender and non-bina...,None,None,None,None,posted-content,preprint,preprint,None,True,2018-09-04,2018-09-04,2018-09-04,2025-02-21,None,2018-09-04,None,2018-09-04,None,2018,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<jats:p>An view of literature on transgender a...,An view of literature on transgender and non-b...,"[{""URL"": ""https://ndownloader.figshare.com/fil...",None,"Broadway-Horner, Matt",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-8834-7...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.7038500.v1"", ""URL"": ...",10.31124/advance.7038500.v1,10.31124,10.31124,advance.7038500.v1,10.31124/advance,10.31124/advance,advance.sagepub.com,advance.sagepub.com/articles
4,crossref::10.31124/advance.7038503.v1,Advance,crossref,10.31124/advance.7038503.v1,10.31124/advance.7038503.v1,https://doi.org/10.31124/advance.7038503.v1,https://advance.sagepub.com/articles/A_TROG_ma...,https://advance.sagepub.com/articles/A_TROG_ma...,10.31124,179,None,None,crossref,SAGE Publications,None,None,None,None,A TROG masked.docUsing T.R.O.G to improve psyc...,None,None,None,None,posted-content,preprint,preprint,None,True,2018-09-04,2018-09-04,2018-09-04,2025-02-21,None,2018-09-04,None,2018-09-04,None,2018,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<jats:p>A discussion on language from a theore...,A discussion on language from a theoretical pe...,"[{""URL"": ""https://ndownloader.figshare.com/fil...",None,"Broadway-Horner, Matt",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-8834-7...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.7038503.v1"", ""URL"": ...",10.31124/advance.7038503.v1,10.31124,10.31124,advance.7038503.v1,10.31124/advance,10.31124/advance,advance.sagepub.com,advance.sagepub.com/articles
5,crossref::10.31124/advance.7038506.v1,Advance,crossref,10.31124/advance.7038506.v1,10.31124/advance.7038506.v1,https://doi.org/10.31124/advance.7038506.v1,https://advance.sagepub.com/articles/Are_We_Th.

In [23]:
# [./] matches either a dot or a slash
# v\d+ matches 'v' followed by one or more digits
# $ ensures this pattern is at the very end of the string
pattern = r'[./]v\d+$'
# pattern = r'v\d+$'
mask = ~df['doi'].str.contains(pattern, regex=True, na=False)
result = df[mask]
result

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
0,crossref::10.31124/advance.7037639,Advance,crossref,10.31124/advance.7037639,10.31124/advance.7037639,https://doi.org/10.31124/advance.7037639,https://advance.sagepub.com/articles/The_Priva...,https://advance.sagepub.com/articles/The_Priva...,10.31124,179,None,None,crossref,SAGE Publications,None,None,None,None,The Privatization of Security and the Emergenc...,None,None,None,None,posted-content,preprint,preprint,None,True,2018-09-04,2018-09-04,2018-09-04,2022-03-30,None,2018-09-04,None,2018-09-04,None,2018,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<jats:p>&lt;p&gt;This study interrogates the p...,This study interrogates the participation of p...,"[{""URL"": ""https://ndownloader.figshare.com/fil...",None,"Chinwokwu, Eke; Igbo, Emmanuel",None,None,"[{""affiliation"": [], ""family"": ""Chinwokwu"", ""g...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.7037639"", ""URL"": ""ht...",10.31124/advance.7037639,10.31124,10.31124,advance.7037639,10.31124/advance,10.31124/advance,advance.sagepub.com,advance.sagepub.com/articles
1,crossref::10.31124/advance.7038500,Advance,crossref,10.31124/advance.7038500,10.31124/advance.7038500,https://doi.org/10.31124/advance.7038500,https://advance.sagepub.com/articles/Unmet_nee...,https://advance.sagepub.com/articles/Unmet_nee...,10.31124,179,None,None,crossref,SAGE Publications,None,None,None,None,Unmet needs of ageing transgender and non-bina...,None,None,None,None,posted-content,preprint,preprint,None,True,2018-09-04,2018-09-04,2018-09-04,2025-02-21,None,2018-09-04,None,2018-09-04,None,2018,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<jats:p>An view of literature on transgender a...,An view of literature on transgender and non-b...,"[{""URL"": ""https://ndownloader.figshare.com/fil...",None,"Broadway-Horner, Matt",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-8834-7...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.7038500"", ""URL"": ""ht...",10.31124/advance.7038500,10.31124,10.31124,advance.7038500,10.31124/advance,10.31124/advance,advance.sagepub.com,advance.sagepub.com/articles
3,crossref::10.31124/advance.7038503,Advance,crossref,10.31124/advance.7038503,10.31124/advance.7038503,https://doi.org/10.31124/advance.7038503,https://advance.sagepub.com/articles/A_TROG_ma...,https://advance.sagepub.com/ar

In [24]:
pattern = "10.31124/advance.14132306"

mask = df['doi'].str.contains(pattern, regex=False, na=False)
result = df[mask]
result

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
2473,crossref::10.31124/advance.14132306,Advance,crossref,10.31124/advance.14132306,10.31124/advance.14132306,https://doi.org/10.31124/advance.14132306,https://advance.sagepub.com/doi/full/10.31124/...,https://advance.sagepub.com/doi/full/10.31124/...,10.31124,179,None,None,crossref,SAGE Publications,None,None,None,None,The Effect of Healthcare Education on Future D...,None,None,None,None,posted-content,preprint,preprint,None,True,2021-03-04,2021-03-09,2024-02-22,2024-07-17,None,2021-03-09,None,2021-03-09,None,2021,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<jats:p>&lt;p&gt;In competitive education test...,In competitive education test scores and scien...,"[{""URL"": ""https://ndownloader.figshare.com/fil...",None,"Nowak, Ewa; Barciszewska, Anna-Maria; Lind, Ge...",None,None,"[{""affiliation"": [], ""family"": ""Nowak"", ""given...",None,None,None,None,None,None,None,None,1,None,None,1,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.14132306"", ""URL"": ""h...",10.31124/advance.14132306,10.31124,10.31124,advance.14132306,10.31124/advance,10.31124/advance,advance.sagepub.com,advance.sagepub.com/doi
2475,crossref::10.31124/advance.14132306.v1,Advance,crossref,10.31124/advance.14132306.v1,10.31124/advance.14132306.v1,https://doi.org/10.31124/advance.14132306.v1,https://advance.sagepub.com/doi/full/10.31124/...,https://advance.sagepub.com/doi/full/10.31124/...,10.31124,179,None,None,crossref,SAGE Publications,None,None,None,None,The effect of healthcare education on students...,None,None,None,None,posted-content,preprint,preprint,None,True,2021-03-04,2021-03-04,2024-02-22,2024-02-23,None,2021-03-04,None,2021-03-04,None,2021,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<jats:p>&lt;p&gt;In competitive education test...,In competitive education test scores and scien...,"[{""URL"": ""https://ndownloader.figshare.com/fil...",None,"Nowak, Ewa; Barciszewska, Anna-Maria; Lind, Ge...",None,None,"[{""affiliation"": [], ""family"": ""Nowak"", ""given...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.14132306.v1"", ""URL"":...",10.31124/advance.14132306.v1,10.31124,10.31124,advance.14132306.v1,10.31124/advance,10.31124/advance,advance.sagepub.com,advance.sagepub.com/doi
2477,crossref::10.31124/advance.14132306.v2,Advance,crossref,10.31124/advance.14132306.v2,10.31124/advance.14132306.v2,https://doi.org/10.31124/advan

In [25]:
advance_parent_df, advance_parent_summary = analyze_parent_data("Advance", full_parent_df)


 CONSOLIDATED ANALYSIS: ADVANCE
  > Total Parent Groups:     2,336
  > Total Versions Found:    4,308
  > Unique Parent DOIs:      2,336
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Advance:
-----------------------------------
  1 version(s):     653 groups
  2 version(s):     1,477 groups
  3 version(s):     161 groups
  4 version(s):     33 groups
  5 version(s):     9 groups
  7 version(s):     1 groups
  8 version(s):     1 groups
  26 version(s):    1 groups

  Average versions per parent: 1.84

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Advance                  2271
SSRN                       28
Research Square             7
SocArXiv                    6
Preprints.org               5
Advance; ResearchGate       4
ResearchGate                3
AgEcon Search               2
arXiv                       2
Authorea Inc.               2

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKE

# AfricArXiv

In [26]:
AfricArXiv_df, AfricArXiv_summary = get_server_data("AfricArXiv")


 SERVER ANALYSIS: AFRICARXIV
  > Files found:    2
  > Raw records:    2190
  > Cleaned shape:  (2190, 90)
  > Unique DOIs:    2190
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.60763/africarxiv    1689
10.31730/osf            496
10.31235/osf              5
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
africarxiv.ubuntunet.net    1689
osf.io                       501
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.60763    1689
10.31730     496
10.31235       5
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
None     1689
15934     501
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
My University                                                  1423
Center for Open Science              

In [27]:
AfricArXiv_parent_df, AfricArXiv_parent_summary = analyze_parent_data("AfricArXiv", full_parent_df)


 CONSOLIDATED ANALYSIS: AFRICARXIV
  > Total Parent Groups:     1,578
  > Total Versions Found:    2,107
  > Unique Parent DOIs:      1,578
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for AfricArXiv:
-----------------------------------
  1 version(s):     1,142 groups
  2 version(s):     391 groups
  3 version(s):     28 groups
  4 version(s):     10 groups
  6 version(s):     1 groups
  8 version(s):     3 groups
  9 version(s):     2 groups
  11 version(s):    1 groups

  Average versions per parent: 1.34

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
AfricArXiv                 1566
Open Science Framework        3
SSRN                          3
ResearchGate                  2
SocArXiv                      1
Advance                       1
Humanities Commons CORE       1
Zenodo                        1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
d

# AgEcon Search

In [28]:
AgEcon_Search_df, AgEcon_Search_summary = get_server_data("AgEcon_Search")


 SERVER ANALYSIS: AGECON_SEARCH
  > Files found:    2
  > Raw records:    188173
  > Cleaned shape:  (188173, 90)
  > Unique DOIs:    188173
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.22004/ag      188172
10.22004/tind         1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
ageconsearch.umn.edu    188173
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.22004    188173
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
None    188173
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Unknown    188173
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    188173
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
---------

In [29]:
# for x in AgEcon_Search_df.sample(1).iloc[0]:
#     print(x)

In [30]:
AgEcon_parent_df, AgEcon_parent_summary = analyze_parent_data("AgEcon Search", full_parent_df)


 CONSOLIDATED ANALYSIS: AGECON SEARCH
  > Total Parent Groups:     153,387
  > Total Versions Found:    167,422
  > Unique Parent DOIs:      153,387
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for AgEcon Search:
-----------------------------------
  1 version(s):     145,430 groups
  2 version(s):     6,904 groups
  3 version(s):     598 groups
  4 version(s):     142 groups
  5 version(s):     63 groups
  6 version(s):     49 groups
  7 version(s):     30 groups
  8 version(s):     21 groups
  9 version(s):     16 groups
  10 version(s):    14 groups
  11 version(s):    7 groups
  12 version(s):    11 groups
  13 version(s):    4 groups
  14 version(s):    9 groups
  15 version(s):    10 groups
  16 version(s):    3 groups
  17 version(s):    3 groups
  18 version(s):    7 groups
  19 version(s):    3 groups
  20 version(s):    2 groups
  21 version(s):    2 groups
  22 version(s):    4 groups
  23 version(s):    3 groups
  24 version(s

In [31]:
AgEcon_Search_df[AgEcon_Search_df['type_backend_raw']=='Other']

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
166567,datacite::10.22004/ag.econ.333722,AgEcon Search,datacite,10.22004/ag.econ.333722,10.22004/ag.econ.333722,https://doi.org/10.22004/ag.econ.333722,https://ageconsearch.umn.edu/record/333722,https://ageconsearch.umn.edu/record/333722,10.22004,None,tind.agecon,tind,datacite,Unknown,None,None,None,None,Farmers' risk exposition and its drivers,None,None,None,en,Other,Working or Discussion Paper,Other,None,None,2023-04-06,None,None,None,2025-09-30,None,2023-04-06,None,None,2019,published_year,None,None,None,None,None,"[{""description"": ""The analysis of income risk ...",The analysis of income risk is the basis for f...,None,None,"Duden, C.; Offermann, F.",None,None,"[{""affiliation"": [], ""familyName"": ""Duden"", ""g...",[],None,[],None,None,"[{""subject"": ""Farm Management""}, {""subject"": ""...",None,None,0,0,None,None,0,None,[],,,False,None,,None,None,,None,None,None,client_id,0,"{""client"": {""data"": {""id"": ""tind.agecon"", ""typ...","{""citationCount"": 0, ""container"": {}, ""content...",10.22004/ag.econ.333722,10.22004,10.22004,ag.econ.333722,10.22004/ag,10.22004/ag,ageconsearch.umn.edu,ageconsearch.umn.edu/record
166568,datacite::10.22004/ag.econ.333733,AgEcon Search,datacite,10.22004/ag.econ.333733,10.22004/ag.econ.333733,https://doi.org/10.22004/ag.econ.333733,https://ageconsearch.umn.edu/record/333733,https://ageconsearch.umn.edu/record/333733,10.22004,None,tind.agecon,tind,datacite,Unknown,None,None,None,None,Front matter,None,None,None,en,Other,Journal Article,Other,None,None,2023-04-06,None,None,None,2023-04-06,None,2023-04-06,None,None,2018,published_year,None,None,None,None,None,[],None,None,None,Australian Journal Of Agricultural And Resourc...,None,None,"[{""affiliation"": [], ""name"": ""Australian Journ...",[],None,[],None,None,"[{""subject"": ""Agricultural and Food Policy""}]",None,None,0,0,None,None,0,None,[],,,False,None,,None,None,,None,None,None,client_id,0,"{""client"": {""data"": {""id"": ""tind.agecon"", ""typ...","{""citationCount"": 0, ""container"": {}, ""content...",10.22004/ag.econ.333733,10.22004,10.22004,ag.econ.333733,10.22004/ag,10.22004/ag,ageconsearch.umn.edu,ageconsearch.umn.edu/record
166569,datacite::10.22004/ag.econ.333734,AgEcon Search,datacite,10.22004/ag.econ.333734,10.22004/ag.econ.333734,https://doi.org/10.22004/ag.econ.333734,https://ageconsearch.umn.edu/record/333734,https://ageconsearch.umn.edu/record/333734,10.22004,None,tind.agecon,tind,datacite,Unknown,None,None,None,None,Effects of supermarket monopsony pricing on ag...,None,None,None,en,Other,Journal Article,Other,None,None,2023-04-06,None,None,None,2023-04-06,None,2023-04-06,None,None,2018,published_year,None,None,None,None,None,"[{""de

# AgriRxiv

In [32]:
AgriRxiv_df, AgriRxiv_summary = get_server_data("AgriRxiv")


 SERVER ANALYSIS: AGRIRXIV
  > Files found:    1
  > Raw records:    818
  > Cleaned shape:  (818, 90)
  > Unique DOIs:    818
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31220/osf         391
10.31220/agrirxiv    380
10.31227/osf          22
10.31219/osf          10
10.31235/osf           9
10.31234/osf           5
10.31225/osf           1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
cabidigitallibrary.org    462
osf.io                    356
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31220    771
10.31227     22
10.31219     10
10.31235      9
10.31234      5
10.31225      1
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
242      771
15934     47
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
-----------------------

In [33]:
AgriRxiv_parent_df, AgriRxiv_parent_summary = analyze_parent_data("AgriRxiv", full_parent_df)


 CONSOLIDATED ANALYSIS: AGRIRXIV
  > Total Parent Groups:     777
  > Total Versions Found:    909
  > Unique Parent DOIs:      777
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for AgriRxiv:
-----------------------------------
  1 version(s):     672 groups
  2 version(s):     90 groups
  3 version(s):     8 groups
  4 version(s):     4 groups
  5 version(s):     1 groups
  6 version(s):     2 groups

  Average versions per parent: 1.17

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
AgriRxiv                            713
Open Science Framework               18
AgriRxiv; IndiaRxiv                  10
IndiaRxiv                             5
AgriRxiv; Open Science Framework      5
AgEcon Search                         4
INA-Rxiv                              4
SSRN                                  3
Research Square                       3
PeerJ Preprints                       1

 TOP V

# AIJR Preprints

In [34]:
AIJR_Preprints_df, AIJR_Preprints_summary = get_server_data("AIJR_Preprints")


 SERVER ANALYSIS: AIJR_PREPRINTS
  > Files found:    1
  > Raw records:    143
  > Cleaned shape:  (143, 90)
  > Unique DOIs:    143
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.21467/preprints    143
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
preprints.aijr.org    143
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.21467    143
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
8901    143
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
AIJR Publisher    143
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    143
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
institution_name
N

In [35]:
AIJR_parent_df, AIJR_parent_summary = analyze_parent_data("AIJR Preprints", full_parent_df)


 CONSOLIDATED ANALYSIS: AIJR PREPRINTS
  > Total Parent Groups:     141
  > Total Versions Found:    150
  > Unique Parent DOIs:      141
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for AIJR Preprints:
-----------------------------------
  1 version(s):     134 groups
  2 version(s):     5 groups
  3 version(s):     2 groups

  Average versions per parent: 1.06

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
AIJR Preprints                  134
ResearchGate                      3
ScienceOpen Preprints             1
AIJR Preprints; ResearchGate      1
AfricArXiv                        1
arXiv                             1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.21467/preprints    141



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
preprints.aijr.org    141


 COMPLETED: AIJR PREPRINTS


# AMRC Open Research

In [36]:
AMRC_Open_Research_df, AMRC_Open_Research_summary = get_server_data("AMRC_Open_Research")


 SERVER ANALYSIS: AMRC_OPEN_RESEARCH
  > Files found:    2
  > Raw records:    102
  > Cleaned shape:  (102, 90)
  > Unique DOIs:    102
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.12688/amrcopenres      52
10.12688/healthopenres    50
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
healthopenresearch.org    62
amrcopenresearch.org      40
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.12688    102
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
2560    102
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
F1000 Research Ltd    102
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
Health Open Research                             61

In [37]:
AMRC_Open_Research_df.sort_values(by='title')

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
98,crossref::10.12688/healthopenres.13924.1,AMRC Open Research,crossref,10.12688/healthopenres.13924.1,10.12688/healthopenres.13924.1,https://doi.org/10.12688/healthopenres.13924.1,https://healthopenresearch.org/articles/7-17/v1,https://healthopenresearch.org/articles/7-17/v1,10.12688,2560,None,None,crossref,F1000 Research Ltd,Health Open Research,None,None,2753-6416,A Protocol for Systematic Review of Prognostic...,None,None,None,en,journal-article,None,journal-article,None,False,2025-11-21,None,2025-11-21,2025-11-21,None,2025-10-10,None,2025-10-10,2025-10-10,2025.0,issued_date,None,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<ns3:p>Background Survivors of adolescent and ...,Background Survivors of adolescent and young a...,"[{""URL"": ""https://healthopenresearch.org/artic...",https://healthopenresearch.org/articles/7-17/v...,"Guolla, Louise; Mbuagbaw, Lawrence; Ma, Jinhui...",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-2199-6...",None,None,"[{""award"": [""CVG-186971""], ""award-info"": [{""aw...",CIHR Vanier Canada Graduate Scholarship,1.0,None,None,None,0,None,None,0,58,"[{""DOI"": ""10.1161/CIRCULATIONAHA.119.041403"", ...",None,,,False,None,,None,None,,None,None,https://doi.org/10.12688/healthopenres.crossma...,issn,0,None,"{""DOI"": ""10.12688/healthopenres.13924.1"", ""ISS...",10.12688/healthopenres.13924.1,10.12688,10.12688,healthopenres.13924.1,10.12688/healthopenres,10.12688/healthopenres,healthopenresearch.org,healthopenresearch.org/articles
17,crossref::10.12688/amrcopenres.12936.2,AMRC Open Research,crossref,10.12688/amrcopenres.12936.2,10.12688/amrcopenres.12936.2,https://doi.org/10.12688/amrcopenres.12936.2,https://amrcopenresearch.org/articles/2-29/v2,https://amrcopenresearch.org/articles/2-29/v2,10.12688,2560,None,None,crossref,F1000 Research Ltd,AMRC Open Research,None,None,2517-6900,A collaborative approach to exercise provision...,None,None,None,en,journal-article,None,journal-article,None,False,2021-04-01,None,2021-04-26,2026-02-27,None,2021-04-01,None,2021-04-01,2021-04-01,2021.0,issued_date,None,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<ns3:p>\n <ns3:bold>Backgro...,Background: Exercise has been shown to be bene...,"[{""URL"": ""https://amrcopenresearch.org/article...",https://amrcopenresearch.org/articles/2-29/v2/pdf,"Jones, Julie; Alexander, Lyndsay; Hancock, Eli...",None,None,"[{""ORCID"": ""https://orcid.org/0000-0003-1943-1...",None,None,"[{""award"": [""F-1901""], ""award-info"": [{""award-...","Parkinson's UK; Chief Scientist Office, Scotland",2.0,None,None,None,1,None,None,1,62,"[{""DOI"": ""10.1002

In [38]:
AMRC_parent_df, AMRC_parent_summary = analyze_parent_data("AMRC Open Research", full_parent_df)


 CONSOLIDATED ANALYSIS: AMRC OPEN RESEARCH
  > Total Parent Groups:     69
  > Total Versions Found:    99
  > Unique Parent DOIs:      69
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for AMRC Open Research:
-----------------------------------
  1 version(s):     43 groups
  2 version(s):     22 groups
  3 version(s):     4 groups

  Average versions per parent: 1.43

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
AMRC Open Research    69

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.12688/healthopenres    35
10.12688/amrcopenres      34



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
healthopenresearch.org    41
amrcopenresearch.org      28


 COMPLETED: AMRC OPEN RESEARCH



# APSA Preprints

In [39]:
APSA_Preprints_df, APSA_Preprints_summary = get_server_data("APSA_Preprints")


 SERVER ANALYSIS: APSA_PREPRINTS
  > Files found:    1
  > Raw records:    1470
  > Cleaned shape:  (1470, 90)
  > Unique DOIs:    1470
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.33774/apsa-    1470
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
preprints.apsanet.org    1470
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.33774    1470
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
56    1470
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Cambridge University Press (CUP)    1470
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    1470
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
-------------------------

In [40]:
APSA_parent_df, APSA_parent_summary = analyze_parent_data("APSA Preprints", full_parent_df)


 CONSOLIDATED ANALYSIS: APSA PREPRINTS
  > Total Parent Groups:     1,130
  > Total Versions Found:    1,497
  > Unique Parent DOIs:      1,130
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for APSA Preprints:
-----------------------------------
  1 version(s):     908 groups
  2 version(s):     149 groups
  3 version(s):     45 groups
  4 version(s):     14 groups
  5 version(s):     4 groups
  6 version(s):     3 groups
  7 version(s):     3 groups
  8 version(s):     1 groups
  9 version(s):     1 groups
  10 version(s):    1 groups
  14 version(s):    1 groups

  Average versions per parent: 1.32

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
APSA Preprints                  1089
SSRN                              27
SocArXiv                           4
Open Science Framework             3
Qeios                              1
Cambridge Open Engage              1
APSA Preprints; Res

# Arabixiv

In [41]:
Arabixiv_df, Arabixiv_summary = get_server_data("Arabixiv")


 SERVER ANALYSIS: ARABIXIV
  > Files found:    1
  > Raw records:    502
  > Cleaned shape:  (502, 90)
  > Unique DOIs:    502
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31221/osf    412
10.31219/osf     34
10.31227/osf     25
10.31234/osf     14
10.31235/osf      6
10.31223/osf      5
10.31220/osf      3
10.31228/osf      2
10.31225/osf      1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    502
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31221    412
10.31219     34
10.31227     25
10.31234     14
10.31235      6
10.31223      5
10.31220      3
10.31228      2
10.31225      1
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    494
29705      5
242        3
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
---

In [42]:
Arabixiv_parent_df, Arabixiv_parent_summary = analyze_parent_data("Arabixiv", full_parent_df)


 CONSOLIDATED ANALYSIS: ARABIXIV
  > Total Parent Groups:     476
  > Total Versions Found:    585
  > Unique Parent DOIs:      476
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Arabixiv:
-----------------------------------
  1 version(s):     383 groups
  2 version(s):     78 groups
  3 version(s):     14 groups
  4 version(s):     1 groups

  Average versions per parent: 1.23

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Arabixiv                            457
Frenxiv                              11
Arabixiv; Open Science Framework      4
ResearchGate                          2
Arabixiv; Frenxiv; SocArXiv           1
Open Science Framework                1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.31221/osf    471
10.17605/osf      5



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domai

# ARPHA Preprints

In [43]:
ARPHA_df, ARPHA_summary = get_server_data("ARPHA_Preprints")


 SERVER ANALYSIS: ARPHA_PREPRINTS
  > Files found:    1
  > Raw records:    890
  > Cleaned shape:  (890, 90)
  > Unique DOIs:    890
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.3897/arphapreprints    890
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
preprints.arphahub.com    890
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.3897    890
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
2258    890
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Pensoft Publishers    890
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    890
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
instit

In [44]:
ARPHA_parent_df, ARPHA_parent_summary = analyze_parent_data("ARPHA Preprints", full_parent_df)


 CONSOLIDATED ANALYSIS: ARPHA PREPRINTS
  > Total Parent Groups:     855
  > Total Versions Found:    881
  > Unique Parent DOIs:      855
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for ARPHA Preprints:
-----------------------------------
  1 version(s):     830 groups
  2 version(s):     24 groups
  3 version(s):     1 groups

  Average versions per parent: 1.03

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
ARPHA Preprints    851
SSRN                 1
bioRxiv              1
Research Square      1
Zenodo               1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.3897/arphapreprints    855



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
preprints.arphahub.com    855


 COMPLETED: ARPHA PREPRINTS



# ART-Dok

In [45]:
ART_df, ART_summary = get_server_data("ART-Dok")


 SERVER ANALYSIS: ART-DOK
  > Files found:    2
  > Raw records:    9653
  > Cleaned shape:  (9653, 90)
  > Unique DOIs:    9653
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.11588/artdok    9653
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
archiv.ub.uni-heidelberg.de    9651
ub.uni-heidelberg.de              2
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.11588    9653
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
None    9653
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Heidelberg University Library        9644
None                                    5
Vandenhoeck & Ruprecht                  1
Muzeum Uniwersytetu Warszawskiego       1
Biblioteka Jagiellońska                 1
D

In [46]:
ART_parent_df, ART_parent_summary = analyze_parent_data("ART-Dok", full_parent_df)


 CONSOLIDATED ANALYSIS: ART-DOK
  > Total Parent Groups:     9,529
  > Total Versions Found:    9,654
  > Unique Parent DOIs:      9,529
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for ART-Dok:
-----------------------------------
  1 version(s):     9,426 groups
  2 version(s):     91 groups
  3 version(s):     10 groups
  4 version(s):     1 groups
  12 version(s):    1 groups

  Average versions per parent: 1.01

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
ART-Dok                    9527
Humanities Commons CORE       2

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.11588/artdok    9529



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
archiv.ub.uni-heidelberg.de    9527
ub.uni-heidelberg.de              2


 COMPLETED: ART-DOK



# arXiv

In [47]:
# arXiv_df, arXiv_summary = get_server_data("arXiv")

In [48]:
arXiv_parent_df, arXiv_parent_summary = analyze_parent_data("arXiv", full_parent_df)


 CONSOLIDATED ANALYSIS: ARXIV
  > Total Parent Groups:     2,857,685
  > Total Versions Found:    2,939,445
  > Unique Parent DOIs:      2,857,685
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for arXiv:
-----------------------------------
  1 version(s):     2,784,275 groups
  2 version(s):     66,944 groups
  3 version(s):     5,329 groups
  4 version(s):     772 groups
  5 version(s):     223 groups
  6 version(s):     74 groups
  7 version(s):     29 groups
  8 version(s):     13 groups
  9 version(s):     10 groups
  10 version(s):    2 groups
  11 version(s):    5 groups
  12 version(s):    4 groups
  13 version(s):    2 groups
  14 version(s):    1 groups
  19 version(s):    1 groups
  69 version(s):    1 groups

  Average versions per parent: 1.03

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
arXiv                                         2821925
SSRN                         

# Authorea Inc.

In [49]:
Authorea_df, Authorea_summary = get_server_data("Authorea_Inc.")


 SERVER ANALYSIS: AUTHOREA_INC.
  > Files found:    1
  > Raw records:    65450
  > Cleaned shape:  (65450, 90)
  > Unique DOIs:    65450
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.22541/au          61998
10.22541/essoar       3325
10.1002/essoar         105
10.22541/21docs         17
10.22541/techrxiv        2
10.31124/advance         2
10.15200/winn            1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
authorea.com                    61424
essopenarchive.org               3594
techrxiv.org                      374
advance.sagepub.com                46
journal.sketchingscience.org        4
21docs.com                          4
cise@computer.org                   3
doi.org                             1
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.22541    65342
10.1002  

In [50]:
Authorea_parent_df, Authorea_parent_summary = analyze_parent_data("Authorea Inc.", full_parent_df)


 CONSOLIDATED ANALYSIS: AUTHOREA INC.
  > Total Parent Groups:     56,167
  > Total Versions Found:    60,446
  > Unique Parent DOIs:      56,167
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Authorea Inc.:
-----------------------------------
  1 version(s):     52,683 groups
  2 version(s):     2,976 groups
  3 version(s):     375 groups
  4 version(s):     90 groups
  5 version(s):     17 groups
  6 version(s):     10 groups
  7 version(s):     4 groups
  8 version(s):     2 groups
  9 version(s):     2 groups
  11 version(s):    1 groups
  12 version(s):    1 groups
  14 version(s):    1 groups
  15 version(s):    2 groups
  16 version(s):    1 groups
  17 version(s):    1 groups
  19 version(s):    1 groups

  Average versions per parent: 1.08

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Authorea Inc.                           55050
Research Square                          

In [51]:
Authorea_parent_df[Authorea_parent_df['primary_domain']=='techrxiv.org']

,dup_group_id_full,parent_record_id,parent_server_name,parent_doi,parent_url,parent_title,parent_authors,parent_date_first_seen,parent_year_first_seen,version_record_ids,version_dois,servers_with_counts,total_versions,first_version_date,last_version_date,most_recent_version_servers,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
11183,crossref::10.22541/au.175649113.36750850/v1,crossref::10.22541/au.175649113.36750850/v1,Authorea Inc.,10.22541/au.175649113.36750850/v1,https://www.techrxiv.org/users/960820/articles...,Prediction of Overall Survival Status of Diffu...,"Ganapathy, Ananya",2025-08-29,2025,crossref::10.22541/au.175649113.36750850/v1,10.22541/au.175649113.36750850/v1,Authorea Inc. (1),1,2025-08-29,2025-08-29,Authorea Inc.,10.22541/au.175649113.36750850/v1,10.22541,10.22541,au.175649113.36750850/v1,10.22541/au,10.22541/au,techrxiv.org,techrxiv.org/users
68488,crossref::10.22541/au.175192051.19091421/v1,crossref::10.22541/au.175192051.19091421/v1,Authorea Inc.,10.22541/au.175192051.19091421/v1,https://www.techrxiv.org/users/940424/articles...,The Yanomami People and the Favela: A Comparat...,"Paula, Abdon de",2025-07-07,2025,crossref::10.22541/au.175192051.19091421/v1,10.22541/au.175192051.19091421/v1,Authorea Inc. (1),1,2025-07-07,2025-07-07,Authorea Inc.,10.22541/au.175192051.19091421/v1,10.22541,10.22541,au.175192051.19091421/v1,10.22541/au,10.22541/au,techrxiv.org,techrxiv.org/users
70326,crossref::10.22541/au.170379738.88140454/v1,crossref::10.22541/au.170379738.88140454/v1,Authorea Inc.,10.22541/au.170379738.88140454/v1,https://www.techrxiv.org/users/712087/articles...,Real-time Detection of Low-Rate DDoS Attacks i...,"Alashhab, Abdussalam Ahmed; Zahid, Mohd Soperi...",2023-12-28,2023,crossref::10.22541/au.170379738.88140454/v1,10.22541/au.170379738.88140454/v1,Authorea Inc. (1),1,2023-12-28,2023-12-28,Authorea Inc.,10.22541/au.170379738.88140454/v1,10.22541,10.22541,au.170379738.88140454/v1,10.22541/au,10.22541/au,techrxiv.org,techrxiv.org/users
79897,crossref::10.22541/au.170216685.50804614/v1,crossref::10.22541/au.170216685.50804614/v1,Authorea Inc.,10.22541/au.170216685.50804614/v1,https://techrxiv.org/users/684836/articles/692...,Ayurveda for One Health,"Bansal, Abhishek",2023-12-10,2023,crossref::10.22541/au.170216685.50804614/v1,10.22541/au.170216685.50804614/v1,Authorea Inc. (1),1,2023-12-10,2023-12-10,Authorea Inc.,10.22541/au.170216685.50804614/v1,10.22541,10.22541,au.170216685.50804614/v1,10.22541/au,10.22541/au,techrxiv.org,techrxiv.org/users
140826,crossref::10.22541/au.175950949.97637298/v1,crossref::10.22541/au.175950949.97637298/v1,Authorea Inc.,10.22541/au.175950949.97637298/v1,https://www.techrxiv.org/users/972385/articles...,Coordinated Aerial-Ground Swarm System for Pre...,"Mahmood, Ayan; Khalid, Ali; Saif, Muhammad",2025-10-03,2025,crossref::10.22541/au.175950949.97637298/v1,10.22541/au.175950949.97637298/v1,Authorea Inc. (1),1,2025-10-03,2025-10-03,Authorea Inc.,10.22541/au.175950949.97637298/v1,10.22541,10.22541,au.175950949.97637298/v1,10.22541/au,10.22541/au,techrxiv.org,techrxiv.org/users
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7226091,crossref::10.22541/au.171804670.01201352/v1,crossref::10.22541/au.171804670.01201352/v1,Authorea Inc.,10.22541/au.171804670.01201352/v1,https://www.techrxiv.org/users/714701/articles...,Machine Learning-assisted Partially Blind Hand...,"Yazici, İbrahim; Gures, Emre",2024-06-10,2024,crossref::10.22541/au.171804670.01201352/v1,10.22541/au.171804670.01201352/v1,Authorea Inc. (1),1,2024-06-10,2024-06-10,Authorea Inc.,10.22541/au.171804670.01201352/v1,10.22541,10.22541,au.171804670.01201352/v1,10.22541/au,10.22541/au,techrxiv.org,techrxiv.org/users
7243082,crossref::10.22541/au.171032701.12008685/v1,crossref::10.22541/au.171032701.12008685/v1,Authorea Inc.,10.22541/au.171032701.12008685/v1,https://www.techrxiv.org/users/

https://doi.org/10.22541/au.171804670.01201352/v1   ---> https://www.authorea.com/users/714701/articles/892429-machine-learning-assisted-partially-blind-handover-prediction-in-5g-network-systems

on Authorea


In [52]:
Authorea_parent_df[Authorea_parent_df['primary_domain']=='essopenarchive.org']

,dup_group_id_full,parent_record_id,parent_server_name,parent_doi,parent_url,parent_title,parent_authors,parent_date_first_seen,parent_year_first_seen,version_record_ids,version_dois,servers_with_counts,total_versions,first_version_date,last_version_date,most_recent_version_servers,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
61432,crossref::10.22541/au.170052235.59584468/v1,crossref::10.22541/au.170052235.59584468/v1,Authorea Inc.,10.22541/au.170052235.59584468/v1,https://essopenarchive.org/users/700416/articl...,Kinetics of melt-rock and melt-rock-fluid inte...,"Borisova, Anastassia Y.; Schott, Jacques; Topl...",2023-11-20,2023,crossref::10.22541/au.170052235.59584468/v1,10.22541/au.170052235.59584468/v1,Authorea Inc. (1),1,2023-11-20,2023-11-20,Authorea Inc.,10.22541/au.170052235.59584468/v1,10.22541,10.22541,au.170052235.59584468/v1,10.22541/au,10.22541/au,essopenarchive.org,essopenarchive.org/users
72467,crossref::10.22541/au.176583173.39004043/v1,crossref::10.22541/au.176583173.39004043/v1,Authorea Inc.,10.22541/au.176583173.39004043/v1,https://essopenarchive.org/users/995512/articl...,Europa Surface Spectrophotometry -- Preparing ...,"Benton, Zeina; Seelos, Frank; Itoh, Yuki; Step...",2025-12-15,2025,crossref::10.22541/au.176583173.39004043/v1,10.22541/au.176583173.39004043/v1,Authorea Inc. (1),1,2025-12-15,2025-12-15,Authorea Inc.,10.22541/au.176583173.39004043/v1,10.22541,10.22541,au.176583173.39004043/v1,10.22541/au,10.22541/au,essopenarchive.org,essopenarchive.org/users
74890,crossref::10.22541/au.176046418.80101496/v1,crossref::10.22541/au.176046418.80101496/v1,Authorea Inc.,10.22541/au.176046418.80101496/v1,https://essopenarchive.org/users/975696/articl...,Data Centers Water Footprint: The Need for Mor...,"Privette, Ana Pinheiro; Barros, Ana P.; Cai, X...",2025-10-14,2025,crossref::10.22541/au.176046418.80101496/v1,10.22541/au.176046418.80101496/v1,Authorea Inc. (1),1,2025-10-14,2025-10-14,Authorea Inc.,10.22541/au.176046418.80101496/v1,10.22541,10.22541,au.176046418.80101496/v1,10.22541/au,10.22541/au,essopenarchive.org,essopenarchive.org/users
75372,crossref::10.22541/au.168147732.23405579/v1,crossref::10.22541/au.168147732.23405579/v1,Authorea Inc.,10.22541/au.168147732.23405579/v1,https://essopenarchive.org/users/593062/articl...,Effect of seasonal freeze-thaw process on spat...,"Sun, Jingwei; Wang, Fugang; Ding, Lujiao; Wang...",2023-04-14,2023,crossref::10.22541/au.168147732.23405579/v1,10.22541/au.168147732.23405579/v1,Authorea Inc. (1),1,2023-04-14,2023-04-14,Authorea Inc.,10.22541/au.168147732.23405579/v1,10.22541,10.22541,au.168147732.23405579/v1,10.22541/au,10.22541/au,essopenarchive.org,essopenarchive.org/users
79898,crossref::10.22541/au.170258901.13051548/v1,crossref::10.22541/au.170258901.13051548/v1,Authorea Inc.,10.22541/au.170258901.13051548/v1,https://essopenarchive.org/users/709109/articl...,Signal to Noise Ratio and Spectral Sampling Co...,"Perez-Lopez, Sebastian Alonso; Kremer, Christo...",2023-12-14,2023,crossref::10.22541/au.170258901.13051548/v1,10.22541/au.170258901.13051548/v1,Authorea Inc. (1),1,2023-12-14,2023-12-14,Authorea Inc.,10.22541/au.170258901.13051548/v1,10.22541,10.22541,au.170258901.13051548/v1,10.22541/au,10.22541/au,essopenarchive.org,essopenarchive.org/users
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7274361,crossref::10.22541/au.176617599.93150350/v1,crossref::10.22541/au.176617599.93150350/v1,Authorea Inc.,10.22541/au.176617599.93150350/v1,https://essopenarchive.org/users/564073/articl...,Exploring\tWavefield-Based Location Imaging in...,"Komeazi, Abolfazl; Limberger, Fabian; Rümpker,...",2025-12-19,2025,crossref::10.22541/au.176617599.93150350/v1,10.22541/au.176617599.93150350/v1,Authorea Inc. (1),1,2025-12-19,2025-12-19,Authorea Inc.,10.22541/au.176617599.93150350/v1,10.22541,10.22541,au.176617599.93150350/v1,10.22541/au,10.2254

In [53]:
# Authorea_parent_df['parent_url'][7322807]

In [54]:
# https://doi.org/10.22541/au.176617599.93150350/v1  ---> https://www.authorea.com/users/564073/articles/1371863-exploring-wavefield-based-location-imaging-in-heterogeneous-media-a-borehole-das-example

# Beilstein Archives

In [55]:
Beilstein_df, Beilstein_summary = get_server_data("Beilstein_Archives")


 SERVER ANALYSIS: BEILSTEIN_ARCHIVES
  > Files found:    1
  > Raw records:    697
  > Cleaned shape:  (697, 90)
  > Unique DOIs:    697
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.3762/bxiv    697
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
beilstein-archives.org    697
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.3762    697
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
2086    697
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Beilstein Institut    697
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    697
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
institution_n

In [6]:
Beilstein_parent_df, Beilstein_parent_summary = analyze_parent_data("Beilstein Archives", full_parent_df)


 CONSOLIDATED ANALYSIS: BEILSTEIN ARCHIVES
  > Total Parent Groups:     694
  > Total Versions Found:    702
  > Unique Parent DOIs:      694
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Beilstein Archives:
-----------------------------------
  1 version(s):     687 groups
  2 version(s):     6 groups
  3 version(s):     1 groups

  Average versions per parent: 1.01

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Beilstein Archives    687
arXiv                   5
SSRN                    1
ChemRxiv                1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.3762/bxiv    694



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
beilstein-archives.org    694


 COMPLETED: BEILSTEIN ARCHIVES



# BioHackrXiv

In [7]:
BioHackrXiv_df, BioHackrXiv_summary = get_server_data("BioHackrXiv")


 SERVER ANALYSIS: BIOHACKRXIV
  > Files found:    1
  > Raw records:    139
  > Cleaned shape:  (139, 90)
  > Unique DOIs:    139
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.37044/osf    139
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    139
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.37044    139
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    139
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Center for Open Science    139
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    139
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
institution_name
None    139


In [8]:
BioHackrXiv_parent_df, BioHackrXiv_parent_summary = analyze_parent_data("BioHackrXiv", full_parent_df)


 CONSOLIDATED ANALYSIS: BIOHACKRXIV
  > Total Parent Groups:     136
  > Total Versions Found:    140
  > Unique Parent DOIs:      136
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for BioHackrXiv:
-----------------------------------
  1 version(s):     132 groups
  2 version(s):     4 groups

  Average versions per parent: 1.03

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
BioHackrXiv                  135
BioHackrXiv; ResearchGate      1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.37044/osf    136



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
osf.io    136


 COMPLETED: BIOHACKRXIV



# bioRxiv

In [ ]:
# bioRxiv_df, bioRxiv_summary = get_server_data("bioRxiv")

In [7]:
bioRxiv_parent_df, bioRxiv_parent_summary = analyze_parent_data("bioRxiv", full_parent_df)


 CONSOLIDATED ANALYSIS: BIORXIV
  > Total Parent Groups:     303,393
  > Total Versions Found:    331,893
  > Unique Parent DOIs:      303,393
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for bioRxiv:
-----------------------------------
  1 version(s):     285,068 groups
  2 version(s):     14,112 groups
  3 version(s):     1,318 groups
  4 version(s):     358 groups
  5 version(s):     2,055 groups
  6 version(s):     443 groups
  7 version(s):     34 groups
  8 version(s):     4 groups
  12 version(s):    1 groups

  Average versions per parent: 1.09

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
bioRxiv            285372
eLife                9497
Research Square      3755
SSRN                 2616
arXiv; bioRxiv        413
Authorea Inc.         395
arXiv                 393
F1000Research         163
HAL                    90
Preprints.org          85

 TOP VALUES FOR: DOI_PREFIX_

# BodoArXiv

In [8]:
BodoArXiv_df, BodoArXiv_summary = get_server_data("BodoArXiv")


 SERVER ANALYSIS: BODOARXIV
  > Files found:    1
  > Raw records:    165
  > Cleaned shape:  (165, 90)
  > Unique DOIs:    165
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.34055/osf    162
10.31219/osf      3
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    165
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.34055    162
10.31219      3
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    165
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Center for Open Science    165
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    165
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
--------------------------

In [9]:
BodoArXiv_parent_df, BodoArXiv_parent_summary = analyze_parent_data("BodoArXiv", full_parent_df)


 CONSOLIDATED ANALYSIS: BODOARXIV
  > Total Parent Groups:     149
  > Total Versions Found:    161
  > Unique Parent DOIs:      149
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for BodoArXiv:
-----------------------------------
  1 version(s):     139 groups
  2 version(s):     9 groups
  4 version(s):     1 groups

  Average versions per parent: 1.08

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
BodoArXiv      147
Law Archive      1
Zenodo           1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.34055/osf    146
10.31219/osf      3



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
osf.io    149


 COMPLETED: BODOARXIV



# Cambridge Open Engage

In [10]:
Cambridge_df, Cambridge_summary = get_server_data("Cambridge_Open_Engage")


 SERVER ANALYSIS: CAMBRIDGE_OPEN_ENGAGE
  > Files found:    1
  > Raw records:    3090
  > Cleaned shape:  (3090, 90)
  > Unique DOIs:    3090
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.33774/coe-    3090
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
cambridge.org    3090
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.33774    3090
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
56    3090
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Cambridge University Press (CUP)    3090
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    3090
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
---------------------------

In [11]:
Cambridge_parent_df, Cambridge_parent_summary = analyze_parent_data("Cambridge Open Engage", full_parent_df)


 CONSOLIDATED ANALYSIS: CAMBRIDGE OPEN ENGAGE
  > Total Parent Groups:     1,875
  > Total Versions Found:    2,976
  > Unique Parent DOIs:      1,875
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Cambridge Open Engage:
-----------------------------------
  1 version(s):     1,430 groups
  2 version(s):     266 groups
  3 version(s):     96 groups
  4 version(s):     29 groups
  5 version(s):     15 groups
  6 version(s):     6 groups
  7 version(s):     5 groups
  8 version(s):     4 groups
  9 version(s):     2 groups
  10 version(s):    1 groups
  11 version(s):    3 groups
  12 version(s):    5 groups
  14 version(s):    1 groups
  15 version(s):    2 groups
  18 version(s):    2 groups
  19 version(s):    1 groups
  20 version(s):    2 groups
  22 version(s):    1 groups
  26 version(s):    1 groups
  27 version(s):    1 groups
  32 version(s):    1 groups
  65 version(s):    1 groups

  Average versions per parent: 1.59

 MOST RE

# CERN document server

In [6]:
CERN_df, CERN_summary = get_server_data("CERN_document_server")


 SERVER ANALYSIS: CERN_DOCUMENT_SERVER
  > Files found:    2
  > Raw records:    2631
  > Cleaned shape:  (2631, 90)
  > Unique DOIs:    2026
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.48550/arxiv         776
<NA>                   605
10.5170/cern-          117
10.18429/jacow-ipac     71
10.17181/cern           49
10.17181/d              16
10.17181/q              15
10.17181/c              14
10.17181/m              13
10.17181/n              13
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
cds.cern.ch        1727
repository.cern     903
new-cds.cern.ch       1
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.17181    950
10.48550    776
None        605
10.5170     117
10.18429     96
10.1007      14
10.1393      10
10.18154      9
10.23732      8
10.1051       7
Name: count, d

In [7]:
CERN_df, CERN_summary = get_server_data("CERN_document_server")


 SERVER ANALYSIS: CERN_DOCUMENT_SERVER
  > Files found:    2
  > Raw records:    2631
  > Cleaned shape:  (2631, 90)
  > Unique DOIs:    2026
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.48550/arxiv         776
<NA>                   605
10.5170/cern-          117
10.18429/jacow-ipac     71
10.17181/cern           49
10.17181/d              16
10.17181/q              15
10.17181/c              14
10.17181/m              13
10.17181/n              13
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
cds.cern.ch        1727
repository.cern     903
new-cds.cern.ch       1
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.17181    950
10.48550    776
None        605
10.5170     117
10.18429     96
10.1007      14
10.1393      10
10.18154      9
10.23732      8
10.1051       7
Name: count, d

In [8]:
CERN_parent_df, CERN_parent_summary = analyze_parent_data("CERN document server", full_parent_df)


 CONSOLIDATED ANALYSIS: CERN DOCUMENT SERVER
  > Total Parent Groups:     975
  > Total Versions Found:    1,670
  > Unique Parent DOIs:      697
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for CERN document server:
-----------------------------------
  1 version(s):     433 groups
  2 version(s):     458 groups
  3 version(s):     52 groups
  4 version(s):     15 groups
  5 version(s):     7 groups
  6 version(s):     5 groups
  7 version(s):     3 groups
  8 version(s):     1 groups
  11 version(s):    1 groups

  Average versions per parent: 1.71

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
CERN document server                                   797
arXiv                                                  170
HAL                                                      2
Munich Personal RePEc Archive                            2
CERN document server; arXiv                            

In [9]:
CERN_df_cds = CERN_df[CERN_df['backend']=='cds']
CERN_df_cds

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend


In [10]:
pattern = r'preprint'
# pattern = r'v\d+$'
mask = CERN_df_cds['raw_json'].str.contains(pattern, regex=True, na=False)
result = CERN_df_cds[mask]
result

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend


In [11]:
# for x in result.sample(1).iloc[0]:
#     print(x)

In [12]:
pattern = r'cerncds:BOOK'
# pattern = r'v\d+$'
mask = CERN_df_cds['raw_json'].str.contains(pattern, regex=True, na=False)
result = CERN_df_cds[mask]
result

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend


In [13]:
# for x in result.sample(1).iloc[0]:
#     print(x)

In [14]:
import json
import pandas as pd

def extract_set_specs(raw_json_str):
    """
    Safely parses the raw OAI-PMH JSON string and extracts 
    set_specs elements into a clean, flat string.
    """
    # Handle missing or null records safely
    if pd.isna(raw_json_str) or not isinstance(raw_json_str, str):
        return None
        
    try:
        # 1. Parse the JSON string into a dictionary
        data = json.loads(raw_json_str)
        
        # 2. Get the 'set_specs' list (fallback to empty list if key is missing)
        specs_list = data.get("set_specs", [])
        
        if not specs_list:
            return None
            
        # 3. Clean each item (remove whitespace) and filter out any empties
        cleaned_specs = [str(spec).strip() for spec in specs_list if spec]
        
        # 4. Join multiple sets with a semicolon for flat-field consistency
        return "; ".join(cleaned_specs)
        
    except (json.JSONDecodeError, TypeError, AttributeError):
        # Fallback if the string isn't valid JSON or format is unexpected
        return None

# --- Application to your DataFrame
# Create the flat string column
CERN_df_cds["set_specs_flat"] = CERN_df_cds["raw_json"].apply(extract_set_specs)

# --- Check your results
print("Total CERN_df_cds records :", len(CERN_df_cds))
print("Null count in new column:", CERN_df_cds["set_specs_flat"].isna().sum())
print("\nTop 10 CERN Collections found:")
print("-" * 35)
print(CERN_df_cds["set_specs_flat"].value_counts(dropna=False).head(60))

Total CERN_df_cds records : 0
Null count in new column: 0

Top 10 CERN Collections found:
-----------------------------------
Series([], Name: count, dtype: int64)


In [15]:
# Count individual collection frequencies across the entire server
individual_sets = (
    CERN_df_cds["set_specs_flat"]
    .dropna()
    .str.split("; ")
    .explode()
    .value_counts()
)

print("\nIndividual Set Specification Frequencies:")
print("-" * 35)
print(individual_sets)


Individual Set Specification Frequencies:
-----------------------------------
Series([], Name: count, dtype: int64)


In [16]:
CERN_df_cds['publication_year'].value_counts().reset_index().sort_values(by='publication_year')

,publication_year,count


# ChemRxiv

In [25]:
ChemRxiv_df, ChemRxiv_summary = get_server_data("ChemRxiv")


 SERVER ANALYSIS: CHEMRXIV
  > Files found:    1
  > Raw records:    46475
  > Cleaned shape:  (46475, 90)
  > Unique DOIs:    46475
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.26434/chemrxiv-    35040
10.26434/chemrxiv     11435
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
chemrxiv.org    46475
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.26434    46475
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
316    46475
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
American Chemical Society (ACS)    46475
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    46475
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME

In [26]:
ChemRxiv_parent_df, ChemRxiv_parent_summary = analyze_parent_data("ChemRxiv", full_parent_df)


 CONSOLIDATED ANALYSIS: CHEMRXIV
  > Total Parent Groups:     36,128
  > Total Versions Found:    46,790
  > Unique Parent DOIs:      36,128
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for ChemRxiv:
-----------------------------------
  1 version(s):     28,095 groups
  2 version(s):     6,242 groups
  3 version(s):     1,341 groups
  4 version(s):     303 groups
  5 version(s):     83 groups
  6 version(s):     25 groups
  7 version(s):     14 groups
  8 version(s):     7 groups
  9 version(s):     5 groups
  10 version(s):    2 groups
  11 version(s):    2 groups
  12 version(s):    2 groups
  13 version(s):    2 groups
  14 version(s):    1 groups
  16 version(s):    1 groups
  23 version(s):    1 groups
  31 version(s):    1 groups
  36 version(s):    1 groups

  Average versions per parent: 1.30

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
ChemRxiv                  35656
Res

# CogPrints

In [27]:
CogPrints_df, CogPrints_summary = get_server_data("CogPrints")


 SERVER ANALYSIS: COGPRINTS
  > Files found:    1
  > Raw records:    1537
  > Cleaned shape:  (1537, 90)
  > Unique DOIs:    39
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
<NA>               1498
10.48550/arxiv       14
10.5281/zenodo        7
10.13140/rg           7
10.13140/2            2
10.60692/cs           1
10.6084/m             1
10.60692/frxsy-       1
10.60692/ny           1
10.48456/tr-          1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
None             1041
cogprints.org     471
doi.org            25
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
None        1498
10.48550      14
10.13140       9
10.5281        7
10.60692       4
10.6084        1
10.48456       1
10.71910       1
10.18267       1
10.13016       1
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_I

In [28]:
# CogPrints_df

In [29]:
CogPrints_parent_df, CogPrints_parent_summary = analyze_parent_data("CogPrints", full_parent_df)


 CONSOLIDATED ANALYSIS: COGPRINTS
  > Total Parent Groups:     1,487
  > Total Versions Found:    1,505
  > Unique Parent DOIs:      0
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for CogPrints:
-----------------------------------
  1 version(s):     1,470 groups
  2 version(s):     16 groups
  3 version(s):     1 groups

  Average versions per parent: 1.01

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
CogPrints          1472
arXiv                 9
SSRN                  2
PsyArXiv              1
viXra                 1
CogPrints; HAL        1
PhilSci-Archive       1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
<NA>    1487



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
None             1030
cogprints.org     457


 COMPLETED: COGPRINTS



In [30]:
pattern = r'10.48550/arxiv'
# pattern = r'v\d+$'
mask = CogPrints_parent_df['parent_doi'].str.contains(pattern, regex=True, na=False)
result = CogPrints_parent_df[mask]
result

,dup_group_id_full,parent_record_id,parent_server_name,parent_doi,parent_url,parent_title,parent_authors,parent_date_first_seen,parent_year_first_seen,version_record_ids,version_dois,servers_with_counts,total_versions,first_version_date,last_version_date,most_recent_version_servers,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend


In [31]:
pattern = r'10.48550/arxiv.physics/0001050'
# pattern = r'v\d+$'
mask = full_parent_df['parent_doi'].str.contains(pattern, regex=True, na=False)
result = full_parent_df[mask]
result

,dup_group_id_full,parent_record_id,parent_server_name,parent_doi,parent_url,parent_title,parent_authors,parent_date_first_seen,parent_year_first_seen,version_record_ids,version_dois,servers_with_counts,total_versions,first_version_date,last_version_date,most_recent_version_servers
4244908,datacite::10.48550/arxiv.physics/0001050,datacite::10.48550/arxiv.physics/0001050,arXiv,10.48550/arxiv.physics/0001050,https://arxiv.org/abs/physics/0001050,Statistical mechanics of neocortical interacti...,"Ingber, Lester",2000-01-23,2000,datacite::10.48550/arxiv.physics/0001050,10.48550/arxiv.physics/0001050,arXiv (1),1,2000-01-23,2000-01-23,arXiv


# CoP

In [32]:
CoP_df, CoP_summary = get_server_data("CoP")


 SERVER ANALYSIS: COP
  > Files found:    1
  > Raw records:    30
  > Cleaned shape:  (30, 90)
  > Unique DOIs:    30
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31219/osf    30
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    30
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31219    30
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    30
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Center for Open Science    30
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    30
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
institution_name
None    30
Name: count, dtype

In [33]:
CoP_parent_df, CoP_parent_summary = analyze_parent_data("CoP", full_parent_df)


 CONSOLIDATED ANALYSIS: COP
  > Total Parent Groups:     29
  > Total Versions Found:    30
  > Unique Parent DOIs:      29
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for CoP:
-----------------------------------
  1 version(s):     28 groups
  2 version(s):     1 groups

  Average versions per parent: 1.03

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
CoP    29

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.31219/osf    29



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
osf.io    29


 COMPLETED: COP



# Covid-19 Preprints

In [34]:
Covid_df, Covid_summary = get_server_data("Covid-19_Preprints")


 SERVER ANALYSIS: COVID-19_PREPRINTS
  > Files found:    1
  > Raw records:    647
  > Cleaned shape:  (647, 90)
  > Unique DOIs:    647
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.21055/preprints-    647
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
covid19-preprints.microbe.ru    647
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.21055    647
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
8634    647
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Russian Research Anti-Plague Institute Microbe    647
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    647
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
--

In [35]:
Covid_parent_df, Covid_parent_summary = analyze_parent_data("Covid-19 Preprints", full_parent_df)


 CONSOLIDATED ANALYSIS: COVID-19 PREPRINTS
  > Total Parent Groups:     293
  > Total Versions Found:    647
  > Unique Parent DOIs:      293
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Covid-19 Preprints:
-----------------------------------
  1 version(s):     262 groups
  2 version(s):     14 groups
  3 version(s):     2 groups
  4 version(s):     5 groups
  5 version(s):     1 groups
  6 version(s):     1 groups
  8 version(s):     1 groups
  9 version(s):     1 groups
  16 version(s):    1 groups
  21 version(s):    1 groups
  24 version(s):    1 groups
  33 version(s):    1 groups
  73 version(s):    1 groups
  136 version(s):   1 groups

  Average versions per parent: 2.21

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Covid-19 Preprints    292
PREPRINTS.RU            1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.

# CrimRxiv

In [36]:
CrimRxiv_df, CrimRxiv_summary = get_server_data("CrimRxiv")


 SERVER ANALYSIS: CRIMRXIV
  > Files found:    1
  > Raw records:    2838
  > Cleaned shape:  (2838, 90)
  > Unique DOIs:    2838
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.21428/cb          2827
10.21428/51bae76e      11
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
crimrxiv.com           2806
crimrxiv.pubpub.org      21
oqc.crimrxiv.com         11
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.21428    2838
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
9621    2838
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
PubPub    2838
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
CrimRxiv                        2678
None      

In [37]:
CrimRxiv_parent_df, CrimRxiv_parent_summary = analyze_parent_data("CrimRxiv", full_parent_df)


 CONSOLIDATED ANALYSIS: CRIMRXIV
  > Total Parent Groups:     2,570
  > Total Versions Found:    2,713
  > Unique Parent DOIs:      2,570
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for CrimRxiv:
-----------------------------------
  1 version(s):     2,443 groups
  2 version(s):     115 groups
  3 version(s):     10 groups
  4 version(s):     1 groups
  6 version(s):     1 groups

  Average versions per parent: 1.06

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
CrimRxiv                  2541
CrimRxiv; SocArXiv           8
CrimRxiv; arXiv              5
SSRN                         4
SocArXiv                     4
arXiv                        2
Open Science Framework       2
Research Square              2
CrimRxiv; ResearchGate       1
PsyArXiv                     1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.21428/cb     

In [38]:
CrimRxiv_df[CrimRxiv_df['type_backend_raw']=='component']

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
1707,crossref::10.21428/cb6ab371.b8929691/7cc1f2de,CrimRxiv,crossref,10.21428/cb6ab371.b8929691/7cc1f2de,10.21428/cb6ab371.b8929691/7cc1f2de,https://doi.org/10.21428/cb6ab371.b8929691/7cc...,https://www.crimrxiv.com/pub/pag3wtd8,https://www.crimrxiv.com/pub/pag3wtd8,10.21428,9621,None,None,crossref,PubPub,None,None,None,None,Open access to journal articles of the America...,None,None,None,None,component,None,component,None,False,2023-10-31,None,2023-10-31,2025-11-23,None,2023-10-31,None,2023-10-31,None,2023.0,issued_date,None,None,None,None,None,None,None,None,None,"Jacques, Scott",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-2089-4...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,prefix/primary_domain,25,None,"{""DOI"": ""10.21428/cb6ab371.b8929691/7cc1f2de"",...",10.21428/cb6ab371.b8929691/7cc1f2de,10.21428,10.21428,cb6ab371.b8929691/7cc1f2de,10.21428/cb,10.21428/cb,crimrxiv.com,crimrxiv.com/pub


In [39]:
CrimRxiv_df[CrimRxiv_df['type_backend_raw']=='book-chapter']

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
2162,crossref::10.21428/cb6ab371.a4e2c432,CrimRxiv,crossref,10.21428/cb6ab371.a4e2c432,10.21428/cb6ab371.a4e2c432,https://doi.org/10.21428/cb6ab371.a4e2c432,https://www.crimrxiv.com/pub/g5531dv0,https://www.crimrxiv.com/pub/g5531dv0,10.21428,9621,None,None,crossref,PubPub,Criminology's Public Domain,None,None,None,Community Organization and Juvenile Delinquenc...,None,None,None,None,book-chapter,None,book-chapter,None,False,2024-11-03,None,2024-11-03,2024-11-03,None,2024-11-03,None,2024-11-03,2024-11-03,2024.0,issued_date,None,None,None,None,None,None,None,None,None,"Park, Robert E.",None,None,"[{""affiliation"": [], ""family"": ""Park"", ""given""...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",,,False,None,,None,None,,None,NaN,None,prefix/primary_domain,25,None,"{""DOI"": ""10.21428/cb6ab371.a4e2c432"", ""URL"": ""...",10.21428/cb6ab371.a4e2c432,10.21428,10.21428,cb6ab371.a4e2c432,10.21428/cb,10.21428/cb,crimrxiv.com,crimrxiv.com/pub
2271,crossref::10.21428/cb6ab371.a03e79ee,CrimRxiv,crossref,10.21428/cb6ab371.a03e79ee,10.21428/cb6ab371.a03e79ee,https://doi.org/10.21428/cb6ab371.a03e79ee,https://www.crimrxiv.com/pub/1k60zvag,https://www.crimrxiv.com/pub/1k60zvag,10.21428,9621,None,None,crossref,PubPub,Criminology's Public Domain,None,None,None,An Essay [Excerpt] on Crimes and Punishments,None,None,None,None,book-chapter,None,book-chapter,None,False,2025-01-08,None,2025-01-08,2025-01-09,None,2025-01-08,None,2025-01-08,2025-01-08,2025.0,issued_date,None,None,None,None,None,None,None,None,None,"Beccaria, Cesare",None,None,"[{""affiliation"": [], ""family"": ""Beccaria"", ""gi...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",,,False,None,,None,None,,None,NaN,None,prefix/primary_domain,25,None,"{""DOI"": ""10.21428/cb6ab371.a03e79ee"", ""URL"": ""...",10.21428/cb6ab371.a03e79ee,10.21428,10.21428,cb6ab371.a03e79ee,10.21428/cb,10.21428/cb,crimrxiv.com,crimrxiv.com/pub


In [40]:
CrimRxiv_df[CrimRxiv_df['type_backend_raw']=='peer-review']

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
1014,crossref::10.21428/cb6ab371.33c9e126,CrimRxiv,crossref,10.21428/cb6ab371.33c9e126,10.21428/cb6ab371.33c9e126,https://doi.org/10.21428/cb6ab371.33c9e126,https://www.crimrxiv.com/pub/s4okkv64,https://www.crimrxiv.com/pub/s4okkv64,10.21428,9621,None,None,crossref,PubPub,None,None,None,None,"Review 1 of ""Treating Criminal Justice-Involve...",None,None,None,None,peer-review,None,peer-review,None,False,2022-05-24,None,2022-05-24,2024-03-03,None,2022-05-24,None,2022-05-24,None,2022.0,issued_date,None,None,None,None,None,None,None,None,None,"Windle, James",None,None,"[{""affiliation"": [], ""family"": ""Windle"", ""give...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,"{""is-review-of"": [{""asserted-by"": ""subject"", ""...",,,False,None,,None,None,,None,NaN,None,prefix/primary_domain,25,None,"{""DOI"": ""10.21428/cb6ab371.33c9e126"", ""URL"": ""...",10.21428/cb6ab371.33c9e126,10.21428,10.21428,cb6ab371.33c9e126,10.21428/cb,10.21428/cb,crimrxiv.com,crimrxiv.com/pub
1015,crossref::10.21428/cb6ab371.c4d02d0a,CrimRxiv,crossref,10.21428/cb6ab371.c4d02d0a,10.21428/cb6ab371.c4d02d0a,https://doi.org/10.21428/cb6ab371.c4d02d0a,https://www.crimrxiv.com/pub/frlqbegb,https://www.crimrxiv.com/pub/frlqbegb,10.21428,9621,None,None,crossref,PubPub,None,None,None,None,"Review 2 of ""Doing death work: A mixed method ...",None,None,None,None,peer-review,None,peer-review,None,False,2022-05-24,None,2022-05-24,2024-03-03,None,2022-05-24,None,2022-05-24,None,2022.0,issued_date,None,None,None,None,None,None,None,None,None,"Brondolo, Elizabeth",None,None,"[{""affiliation"": [], ""family"": ""Brondolo"", ""gi...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,"{""is-review-of"": [{""asserted-by"": ""subject"", ""...",,,False,None,,None,None,,None,NaN,None,prefix/primary_domain,25,None,"{""DOI"": ""10.21428/cb6ab371.c4d02d0a"", ""URL"": ""...",10.21428/cb6ab371.c4d02d0a,10.21428,10.21428,cb6ab371.c4d02d0a,10.21428/cb,10.21428/cb,crimrxiv.com,crimrxiv.com/pub
1321,crossref::10.21428/cb6ab371.f514a7ba,CrimRxiv,crossref,10.21428/cb6ab371.f514a7ba,10.21428/cb6ab371.f514a7ba,https://doi.org/10.21428/cb6ab371.f514a7ba,https://www.crimrxiv.com/pub/19r39l3l,https://www.crimrxiv.com/pub/19r39l3l,10.21428,9621,None,None,crossref,PubPub,None,None,None,None,"Review of ""Ranking the openness of criminology...",None,None,None,None,peer-review,None,peer-review,None,False,2022-10-06,None,2022-10-06,2025-05-14,None,2022-10-06,None,2022-10-06,None,2022.0,issued_date,None,None,None,None,None,None,None,None,None,"Wheeler, Andrew",None,None,"[{""ORCID"": ""https://orcid.org/0000-0003-2255-1...",None,None,None,None,None,None,None,None,1,None,None,1,0,None,"{""is-review-o

In [41]:
CrimRxiv_df[CrimRxiv_df['type_backend_raw']=='journal-article']

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
0,crossref::10.21428/cb6ab371.30868967,CrimRxiv,crossref,10.21428/cb6ab371.30868967,10.21428/cb6ab371.30868967,https://doi.org/10.21428/cb6ab371.30868967,https://crimrxiv.pubpub.org/pub/tn1vo8l2,https://crimrxiv.pubpub.org/pub/tn1vo8l2,10.21428,9621,None,None,crossref,PubPub,CrimRxiv,None,None,None,"Bentham, Not Epicurus: The Relevance of Pleasu...",None,None,None,en,journal-article,None,journal-article,None,False,2020-07-07,None,2020-07-07,2025-11-23,None,2020-07-07,None,2020-07-07,2020-07-07,2020.0,issued_date,None,None,None,None,None,None,None,None,None,"Jacques, Scott",Georgia State University,None,"[{""affiliation"": [{""name"": ""Georgia State Univ...",None,None,None,None,None,None,None,None,1,None,None,1,0,None,None,,,False,None,,None,None,,None,NaN,None,prefix/primary_domain,25,None,"{""DOI"": ""10.21428/cb6ab371.30868967"", ""URL"": ""...",10.21428/cb6ab371.30868967,10.21428,10.21428,cb6ab371.30868967,10.21428/cb,10.21428/cb,crimrxiv.pubpub.org,crimrxiv.pubpub.org/pub
1,crossref::10.21428/cb6ab371.aab5ffbe,CrimRxiv,crossref,10.21428/cb6ab371.aab5ffbe,10.21428/cb6ab371.aab5ffbe,https://doi.org/10.21428/cb6ab371.aab5ffbe,https://crimrxiv.pubpub.org/pub/oxbqg1op,https://crimrxiv.pubpub.org/pub/oxbqg1op,10.21428,9621,None,None,crossref,PubPub,CrimRxiv,None,None,None,Proterrence &amp; Rule Illegitimacy in an Age ...,None,None,None,en,journal-article,None,journal-article,None,False,2020-07-07,None,2020-07-07,2022-04-05,None,2020-07-07,None,2020-07-07,2020-07-07,2020.0,issued_date,None,None,None,None,None,None,None,None,None,"Jacobs, Bruce; Jacques, Scott","University of Texas, Dallas; Georgia State Uni...",None,"[{""affiliation"": [{""name"": ""University of Texa...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,prefix/primary_domain,25,None,"{""DOI"": ""10.21428/cb6ab371.aab5ffbe"", ""URL"": ""...",10.21428/cb6ab371.aab5ffbe,10.21428,10.21428,cb6ab371.aab5ffbe,10.21428/cb,10.21428/cb,crimrxiv.pubpub.org,crimrxiv.pubpub.org/pub
2,crossref::10.21428/cb6ab371.9e0bdd09,CrimRxiv,crossref,10.21428/cb6ab371.9e0bdd09,10.21428/cb6ab371.9e0bdd09,https://doi.org/10.21428/cb6ab371.9e0bdd09,https://crimrxiv.pubpub.org/pub/zk9k26ba,https://crimrxiv.pubpub.org/pub/zk9k26ba,10.21428,9621,None,None,crossref,PubPub,CrimRxiv,None,None,None,La cybercriminalité,None,None,None,en,journal-article,None,journal-article,None,False,2020-07-08,None,2020-07-08,2023-08-16,None,2020-07-08,None,2020-07-08,2020-07-08,2020.0,issued_date,None,None,None,None,None,None,None,None,None,"Décary-Hétu, David",None,None,"[{""affiliation"": [], ""family"": ""Décary-Hétu"", ...",None,None,None,None,None,None,None,None,1,None,None,1,0,

# CrossAsia-Repository

In [42]:
CrossAsia_df, CrossAsia_summary = get_server_data("CrossAsia-Repository")


 SERVER ANALYSIS: CROSSASIA-REPOSITORY
  > Files found:    2
  > Raw records:    479
  > Cleaned shape:  (479, 90)
  > Unique DOIs:    479
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.48796/20    479
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
repository.crossasia.org    479
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.48796    479
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
None    479
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Fachinformationsdienst (FID) Asien                                                             428
CrossAsia-eBooks                                                                                21
Iudicium                                          

In [43]:
CrossAsia_parent_df, CrossAsia_parent_summary = analyze_parent_data("CrossAsia-Repository", full_parent_df)


 CONSOLIDATED ANALYSIS: CROSSASIA-REPOSITORY
  > Total Parent Groups:     459
  > Total Versions Found:    479
  > Unique Parent DOIs:      459
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for CrossAsia-Repository:
-----------------------------------
  1 version(s):     455 groups
  2 version(s):     2 groups
  4 version(s):     1 groups
  16 version(s):    1 groups

  Average versions per parent: 1.04

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
CrossAsia-Repository    459

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.48796/20    459



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
repository.crossasia.org    459


 COMPLETED: CROSSASIA-REPOSITORY



In [44]:
CrossAsia_df[CrossAsia_df['subtype_backend_raw']=='blog_entry']

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
40,datacite::10.48796/20240116-000,CrossAsia-Repository,datacite,10.48796/20240116-000,10.48796/20240116-000,https://doi.org/10.48796/20240116-000,https://repository.crossasia.org/receive/cross...,https://repository.crossasia.org/receive/cross...,10.48796,None,shkl.lofhsj,shkl,datacite,Fachinformationsdienst (FID) Asien,None,None,None,None,Im Banne Chinas: Der Sinologe Wolfgang Franke ...,None,None,None,de,Text,blog_entry,Text,None,None,2024-01-18,None,None,None,2025-02-17,None,2024-01-18,None,None,2024,published_year,None,None,None,CrossAsia Open Access Repository-Lizenz,https://repository.crossasia.org/content/right...,"[{""description"": ""Der Sinologe und Historiker ...",Der Sinologe und Historiker Wolfgang Franke (*...,None,None,"Messingschlager, Stefan; Platzek, Antje",None,None,"[{""affiliation"": [], ""familyName"": ""Messingsch...","[{""affiliation"": [], ""contributorType"": ""Hosti...",None,[],None,None,"[{""subject"": ""Franke, Wolfgang""}, {""subject"": ...",None,None,0,0,None,None,0,None,"[{""relatedIdentifier"": ""https://repository.cro...",,,False,None,,None,None,,None,None,None,client_id,3,"{""client"": {""data"": {""id"": ""shkl.lofhsj"", ""typ...","{""citationCount"": 0, ""container"": {""identifier...",10.48796/20240116-000,10.48796,10.48796,20240116-000,10.48796/20,10.48796/20,repository.crossasia.org,repository.crossasia.org/receive
42,datacite::10.48796/20240326-000,CrossAsia-Repository,datacite,10.48796/20240326-000,10.48796/20240326-000,https://doi.org/10.48796/20240326-000,https://repository.crossasia.org/receive/cross...,https://repository.crossasia.org/receive/cross...,10.48796,None,shkl.lofhsj,shkl,datacite,Fachinformationsdienst (FID) Asien,None,None,None,None,Grundlagen und Konzept zur Erwerbung und Lizen...,None,None,None,de,Text,blog_entry,Text,None,None,2024-03-26,None,None,None,2025-02-17,None,2024-03-26,None,None,2024,published_year,None,None,None,CrossAsia Open Access Repository-Lizenz,https://repository.crossasia.org/content/right...,"[{""description"": ""Die Staatsbibliothek zu Berl...",Die Staatsbibliothek zu Berlin (SBB) betreut s...,None,None,FID Asien,None,None,"[{""affiliation"": [], ""familyName"": ""FID Asien""...","[{""affiliation"": [], ""contributorType"": ""Hosti...",None,[],None,None,"[{""subject"": ""Fachinformationsdienst""}, {""subj...",None,None,0,0,None,None,0,None,"[{""relatedIdentifier"": ""https://repository.cro...",,,False,None,,None,None,,None,None,None,client_id,3,"{""client"": {""data"": {""id"": ""shkl.lofhsj"", ""typ...","{""citationCount"": 0, ""container"": {}, ""content...",10.48796/20240326-000,10.48796,10.48796,20240326-000,10.48796/20,10.48796/20,repository.crossasia.org,repository.

# Digital Access to Scholarship at Harvard (DASH) (Harvard University)

In [45]:
DASH_df, DASH_summary = get_server_data("Digital_Access_to_Scholarship_at_Harvard_(DASH)_(H")


 SERVER ANALYSIS: DIGITAL_ACCESS_TO_SCHOLARSHIP_AT_HARVARD_(DASH)_(H
  > Files found:    1
  > Raw records:    9703
  > Cleaned shape:  (9703, 90)
  > Unique DOIs:    154
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
<NA>              9549
10.48550/arxiv      65
10.53901/tjbs        4
10.7916/d            4
10.15779/z           3
10.1901/jaba         2
10.1257/aer          2
10.1186/14           2
10.13140/rg          2
10.7448/ias          2
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
nrs.harvard.edu               8768
dash.harvard.edu               918
harvardlibrarybulletin.org       6
dissertations.umi.com            4
doi.org                          1
globalasia.org                   1
frstrategie.org                  1
iase-web.org                     1
ipres2023.us                     1
historytoday.com                

In [46]:
DASH_parent_df, DASH_parent_summary = analyze_parent_data("Digital Access to Scholarship at Harvard (DASH) (Harvard University)", full_parent_df)


 CONSOLIDATED ANALYSIS: DIGITAL ACCESS TO SCHOLARSHIP AT HARVARD (DASH) (HARVARD UNIVERSITY)
  > Total Parent Groups:     9,383
  > Total Versions Found:    9,554
  > Unique Parent DOIs:      0
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Digital Access to Scholarship at Harvard (DASH) (Harvard University):
-----------------------------------
  1 version(s):     9,224 groups
  2 version(s):     150 groups
  3 version(s):     6 groups
  4 version(s):     3 groups

  Average versions per parent: 1.02

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Digital Access to Scholarship at Harvard (DASH) (Harvard University)                                         9235
SSRN                                                                                                          117
arXiv                                                                                                          1

# DSpace@MIT

In [47]:
DSpace_df, DSpace_summary = get_server_data("DSpace@MIT")


 SERVER ANALYSIS: DSPACE@MIT
  > Files found:    1
  > Raw records:    12661
  > Cleaned shape:  (12661, 90)
  > Unique DOIs:    1308
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
<NA>               11353
10.48550/arxiv      1081
10.6084/m             17
10.3929/ethz-b-       16
10.17863/cam          13
10.5281/zenodo        12
10.1021/ja            10
10.13140/rg            8
10.1007/jhep           6
10.13016/m             6
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
hdl.handle.net                 12479
dspace.mit.edu                   117
mit.edu                           29
globalchange.mit.edu              20
orcid.org                          7
None                               3
jstor.org                          1
aclanthology.org                   1
arxiv.org                          1
convention2.allacademic.com    

In [48]:
DSpace_parent_df, DSpace_parent_summary = analyze_parent_data("DSpace@MIT", full_parent_df)


 CONSOLIDATED ANALYSIS: DSPACE@MIT
  > Total Parent Groups:     10,420
  > Total Versions Found:    10,686
  > Unique Parent DOIs:      0
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for DSpace@MIT:
-----------------------------------
  1 version(s):     10,165 groups
  2 version(s):     244 groups
  3 version(s):     11 groups

  Average versions per parent: 1.03

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
DSpace@MIT                                                              10244
arXiv                                                                     131
SSRN                                                                       26
RePEc: Research Papers in Economics                                         9
eLife                                                                       2
bioRxiv                                                                     2
DSpace@MIT; R

# E-LIS Repository

In [49]:
E_LIS_df, E_LIS_summary = get_server_data("E-LIS_Repository")


 SERVER ANALYSIS: E-LIS_REPOSITORY
  > Files found:    1
  > Raw records:    9128
  > Cleaned shape:  (9128, 90)
  > Unique DOIs:    264
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
<NA>              8864
10.13140/rg         75
10.5281/zenodo      54
10.13140/2          16
10.11575/prism      16
10.26268/heal       13
10.48550/arxiv      10
10.6084/m            9
10.4403/jlis         7
10.35050/jipm        5
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
None                   8116
eprints.rclis.org       759
doi.org                 232
dialnet.unirioja.es       3
icono14.net               2
elar.urfu.ru              2
jipm.irandoc.ac.ir        2
openaccess.uoc.edu        1
rmlconsultores.com        1
scientificia.com          1
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
None       

In [50]:
E_LIS_parent_df, E_LIS_parent_summary = analyze_parent_data("E-LIS Repository", full_parent_df)


 CONSOLIDATED ANALYSIS: E-LIS REPOSITORY
  > Total Parent Groups:     8,794
  > Total Versions Found:    8,884
  > Unique Parent DOIs:      0
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for E-LIS Repository:
-----------------------------------
  1 version(s):     8,718 groups
  2 version(s):     66 groups
  3 version(s):     6 groups
  4 version(s):     4 groups

  Average versions per parent: 1.01

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
E-LIS Repository                                           8773
SSRN                                                          6
Humanities Commons CORE                                       4
arXiv                                                         2
E-LIS Repository; Social Science Open Access Repository       2
Open Science Framework                                        1
engrXiv                                                      

# Earth and Space Science Open Archive

In [51]:
Earth_df, Earth_summary = get_server_data("Earth_and_Space_Science_Open_Archive")


 SERVER ANALYSIS: EARTH_AND_SPACE_SCIENCE_OPEN_ARCHIVE
  > Files found:    1
  > Raw records:    22748
  > Cleaned shape:  (22748, 90)
  > Unique DOIs:    22748
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.1002/essoar     12974
10.22541/essoar     9774
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
essopenarchive.org    22629
authorea.com             73
essoar.org               43
techrxiv.org              3
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.1002     12974
10.22541     9774
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
311    22748
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Wiley    22748
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------

In [52]:
Earth_parent_df, Earth_parent_summary = analyze_parent_data("Earth and Space Science Open Archive", full_parent_df)


 CONSOLIDATED ANALYSIS: EARTH AND SPACE SCIENCE OPEN ARCHIVE
  > Total Parent Groups:     19,185
  > Total Versions Found:    22,706
  > Unique Parent DOIs:      19,185
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Earth and Space Science Open Archive:
-----------------------------------
  1 version(s):     16,385 groups
  2 version(s):     2,251 groups
  3 version(s):     431 groups
  4 version(s):     88 groups
  5 version(s):     19 groups
  6 version(s):     7 groups
  7 version(s):     1 groups
  9 version(s):     1 groups
  10 version(s):    1 groups
  11 version(s):    1 groups

  Average versions per parent: 1.18

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Earth and Space Science Open Archive           18854
arXiv                                            105
EGUsphere                                         58
Research Square                                   47
SSRN

In [53]:
Earth_df[Earth_df['subtype_backend_raw']=='other']

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
0,crossref::10.1002/essoar.10502642.1,Earth and Space Science Open Archive,crossref,10.1002/essoar.10502642.1,10.1002/essoar.10502642.1,https://doi.org/10.1002/essoar.10502642.1,http://www.essoar.org/doi/10.1002/essoar.10502...,http://www.essoar.org/doi/10.1002/essoar.10502...,10.1002,311,None,None,crossref,Wiley,None,Earth and Space Science Open Archive,Atmospheric Sciences,None,Ionosphere vertical TEC calculated from GNSS r...,None,None,None,None,posted-content,other,other,None,True,2020-04-07,2020-04-07,2020-04-07,2025-02-21,None,2020-04-07,None,2020-04-07,None,2020,issued_date,posted_date,None,None,None,None,None,None,None,None,"Morozova, Anna; Barlyaeva, Tatiana; Barata, Te...",CITEUC,None,"[{""ORCID"": ""https://orcid.org/0000-0002-8552-8...",None,None,None,None,NaN,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,0,None,"{""DOI"": ""10.1002/essoar.10502642.1"", ""URL"": ""h...",10.1002/essoar.10502642.1,10.1002,10.1002,essoar.10502642.1,10.1002/essoar,10.1002/essoar,essoar.org,essoar.org/doi
1,crossref::10.1002/essoar.10503785.1,Earth and Space Science Open Archive,crossref,10.1002/essoar.10503785.1,10.1002/essoar.10503785.1,https://doi.org/10.1002/essoar.10503785.1,http://www.essoar.org/doi/10.1002/essoar.10503...,http://www.essoar.org/doi/10.1002/essoar.10503...,10.1002,311,None,None,crossref,Wiley,None,Earth and Space Science Open Archive,Education,None,Looking for technosignatures using the brain \...,None,None,None,None,posted-content,other,other,None,True,2020-07-31,2020-07-31,2020-07-31,2025-05-13,None,2020-07-31,None,2020-07-31,None,2020,issued_date,posted_date,None,None,None,None,None,None,None,None,"De la Torre, Gabriel",University of Cadiz,None,"[{""ORCID"": ""https://orcid.org/0000-0001-8636-5...",None,None,None,None,NaN,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,0,None,"{""DOI"": ""10.1002/essoar.10503785.1"", ""URL"": ""h...",10.1002/essoar.10503785.1,10.1002,10.1002,essoar.10503785.1,10.1002/essoar,10.1002/essoar,essoar.org,essoar.org/doi
3,crossref::10.1002/essoar.10504241.1,Earth and Space Science Open Archive,crossref,10.1002/essoar.10504241.1,10.1002/essoar.10504241.1,https://doi.org/10.1002/essoar.10504241.1,http://www.essoar.org/doi/10.1002/essoar.10504...,http://www.essoar.org/doi/10.1002/essoar.10504...,10.1002,311,None,None,crossref,Wiley,None,Earth and Space Science Open Archive,Oceanography,None,CMIP6 without the interpolation: Grid-native a...,None,None,None,None,posted-content,other,other,None,True,2020-09-10,2020-09-10,2020-09-10,2025-05-13,None,2020-09-10,None,2020-09-10,None,2020,issued_da

types others seems to be abstract and not article
https://essopenarchive.org/doi/full/10.1002/essoar.10511112.1 
https://essopenarchive.org/doi/full/10.1002/essoar.10509431.1

In [54]:
Earth_parent_df[Earth_parent_df['primary_domain']=='techrxiv.org']

,dup_group_id_full,parent_record_id,parent_server_name,parent_doi,parent_url,parent_title,parent_authors,parent_date_first_seen,parent_year_first_seen,version_record_ids,version_dois,servers_with_counts,total_versions,first_version_date,last_version_date,most_recent_version_servers,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
2467215,crossref::10.22541/essoar.170239691.10703830/v1,crossref::10.22541/essoar.170239691.10703830/v1,Earth and Space Science Open Archive,10.22541/essoar.170239691.10703830/v1,https://techrxiv.org/users/461179/articles/693...,bug test,"Asif, Usman",2023-12-12,2023,crossref::10.22541/essoar.170239691.10703830/v1,10.22541/essoar.170239691.10703830/v1,Earth and Space Science Open Archive (1),1,2023-12-12,2023-12-12,Earth and Space Science Open Archive,10.22541/essoar.170239691.10703830/v1,10.22541,10.22541,essoar.170239691.10703830/v1,10.22541/essoar,10.22541/essoar,techrxiv.org,techrxiv.org/users
6448262,crossref::10.22541/essoar.171007121.19572474/v1,crossref::10.22541/essoar.171007121.19572474/v1,Earth and Space Science Open Archive,10.22541/essoar.171007121.19572474/v1,https://www.techrxiv.org/users/715826/articles...,"Auto-WCEBleedGen Version V1 and V2: Challenge,...","Hub, Misa; Handa, Palak; Nautiyal, Divyansh; C...",2024-03-10,2024,crossref::10.22541/essoar.171007121.19572474/v1,10.22541/essoar.171007121.19572474/v1,Earth and Space Science Open Archive (1),1,2024-03-10,2024-03-10,Earth and Space Science Open Archive,10.22541/essoar.171007121.19572474/v1,10.22541,10.22541,essoar.171007121.19572474/v1,10.22541/essoar,10.22541/essoar,techrxiv.org,techrxiv.org/users


In [55]:
Earth_parent_df[Earth_parent_df['primary_domain']=='authorea.com']

,dup_group_id_full,parent_record_id,parent_server_name,parent_doi,parent_url,parent_title,parent_authors,parent_date_first_seen,parent_year_first_seen,version_record_ids,version_dois,servers_with_counts,total_versions,first_version_date,last_version_date,most_recent_version_servers,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
45930,crossref::10.22541/essoar.167458061.17750077/v1,crossref::10.22541/essoar.167458061.17750077/v1,Earth and Space Science Open Archive,10.22541/essoar.167458061.17750077/v1,https://www.authorea.com/users/354449/articles...,Convective self-compression of cratons and the...,"Paul, Jyotirmoy; Conrad, Clinton P; Becker, Th...",2023-01-24,2023,crossref::10.22541/essoar.167458061.17750077/v...,10.22541/essoar.167458061.17750077/v1; 10.2254...,Earth and Space Science Open Archive (2),2,2023-01-24,2023-03-28,Earth and Space Science Open Archive,10.22541/essoar.167458061.17750077/v1,10.22541,10.22541,essoar.167458061.17750077/v1,10.22541/essoar,10.22541/essoar,authorea.com,authorea.com/users
93451,crossref::10.22541/essoar.167048524.41438408/v1,crossref::10.22541/essoar.167048524.41438408/v1,Earth and Space Science Open Archive,10.22541/essoar.167048524.41438408/v1,https://www.authorea.com/users/549693/articles...,Quantifying site effects and their influence o...,"Chang, Hilary; Abercrombie, Rachel E.; Nakata,...",2022-12-08,2022,crossref::10.22541/essoar.167048524.41438408/v1,10.22541/essoar.167048524.41438408/v1,Earth and Space Science Open Archive (1),1,2022-12-08,2022-12-08,Earth and Space Science Open Archive,10.22541/essoar.167048524.41438408/v1,10.22541,10.22541,essoar.167048524.41438408/v1,10.22541/essoar,10.22541/essoar,authorea.com,authorea.com/users
542600,crossref::10.22541/essoar.169705252.27888438/v1,crossref::10.22541/essoar.169705252.27888438/v1,Earth and Space Science Open Archive,10.22541/essoar.169705252.27888438/v1,https://www.authorea.com/users/671644/articles...,Simulation and Optimization of Maize Phyllotax...,"Xiang, Zhaocheng; Ge, Yufeng",2023-10-11,2023,crossref::10.22541/essoar.169705252.27888438/v...,10.22541/essoar.169705252.27888438/v1; 10.2254...,Earth and Space Science Open Archive (2),2,2023-10-11,2023-10-18,Earth and Space Science Open Archive,10.22541/essoar.169705252.27888438/v1,10.22541,10.22541,essoar.169705252.27888438/v1,10.22541/essoar,10.22541/essoar,authorea.com,authorea.com/users
632649,crossref::10.22541/essoar.169705230.03901178/v1,crossref::10.22541/essoar.169705230.03901178/v1,Earth and Space Science Open Archive,10.22541/essoar.169705230.03901178/v1,https://www.authorea.com/users/671728/articles...,Low-Cost Photogrammetry Rig for 3D Crop Modell...,"Hrzich, Joe; Bidinosti, Christopher P.; Beck, ...",2023-10-11,2023,crossref::10.22541/essoar.169705230.03901178/v...,10.22541/essoar.169705230.03901178/v1; 10.2254...,Earth and Space Science Open Archive (2),2,2023-10-11,2023-10-14,Earth and Space Science Open Archive,10.22541/essoar.169705230.03901178/v1,10.22541,10.22541,essoar.169705230.03901178/v1,10.22541/essoar,10.22541/essoar,authorea.com,authorea.com/users
816136,crossref::10.22541/essoar.169699689.90164121/v1,crossref::10.22541/essoar.169699689.90164121/v1,Earth and Space Science Open Archive,10.22541/essoar.169699689.90164121/v1,https://www.authorea.com/users/672042/articles...,Exploring Maize Stress Response Phenotypes via...,"Lima, Leonardo W.; Murphy, Katie; Quinones, Al...",2023-10-11,2023,crossref::10.22541/essoar.169699689.90164121/v...,10.22541/essoar.169699689.90164121/v1; 10.2254...,Earth and Space Science Open Archive (2),2,2023-10-11,2023-10-11,Earth and Space Science Open Archive,10.22541/essoar.169699689.90164121/v1,10.22541,10.22541,essoar.169699689.90164121/v1,10.22541/essoar,10.22541/essoar,authorea.com,authorea.com/users
1087617,crossref::10.22541/essoar.169705269.92949948/v1,crossref::10.22541/essoar.169705269.92949948/v1,Earth and Space Science Open Archiv

# EarthArXiv

In [56]:
EarthArXiv_df, EarthArXiv_summary = get_server_data("EarthArXiv")


 SERVER ANALYSIS: EARTHARXIV
  > Files found:    2
  > Raw records:    6537
  > Cleaned shape:  (6537, 90)
  > Unique DOIs:    6537
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31223/x      4691
10.31223/osf    1777
10.31227/osf      26
10.31219/osf      18
10.31234/osf       6
10.31225/osf       6
10.31220/osf       6
10.15697/fk        2
10.31235/osf       2
10.31228/osf       2
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
eartharxiv.org        6373
osf.io                 163
dev.eartharxiv.org       1
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31223    6468
10.31227      26
10.31219      18
10.31234       6
10.31225       6
10.31220       6
10.15697       2
10.31235       2
10.31228       2
10.31230       1
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
----------

In [57]:
EarthArXiv_parent_df, EarthArXiv_parent_summary = analyze_parent_data("EarthArXiv", full_parent_df)


 CONSOLIDATED ANALYSIS: EARTHARXIV
  > Total Parent Groups:     6,256
  > Total Versions Found:    6,821
  > Unique Parent DOIs:      6,256
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for EarthArXiv:
-----------------------------------
  1 version(s):     5,718 groups
  2 version(s):     516 groups
  3 version(s):     18 groups
  4 version(s):     3 groups
  5 version(s):     1 groups

  Average versions per parent: 1.09

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
EarthArXiv                              6015
EarthArXiv; Open Science Framework       125
Research Square                           23
SSRN                                      23
Earth and Space Science Open Archive      13
EarthArXiv; arXiv                         12
EGUsphere                                 11
arXiv                                     10
EarthArXiv; ResearchGate                   7
ResearchGate     

# EasyChair preprint

In [58]:
EasyChair_df, EasyChair_summary = get_server_data("EasyChair_preprint")


 SERVER ANALYSIS: EASYCHAIR_PREPRINT
  > Files found:    1
  > Raw records:    620
  > Cleaned shape:  (620, 90)
  > Unique DOIs:    620
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.29007/g    12
10.29007/h     9
10.29007/k     9
10.29007/t     8
10.29007/c     8
10.29007/m     7
10.29007/p     7
10.29007/z     7
10.29007/v     6
10.29007/x     6
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
easychair.org    620
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.29007    620
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
11545    620
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
EasyChair    620
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------


In [59]:
EasyChair_df

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
0,crossref::10.29007/hsh2,EasyChair preprint,crossref,10.29007/hsh2,10.29007/hsh2,https://doi.org/10.29007/hsh2,https://easychair.org/publications/preprint/1,https://easychair.org/publications/preprint/1,10.29007,11545,None,None,crossref,EasyChair,EasyChair Preprints,None,None,2516-2314,Unification with Abstraction and Theory Instan...,None,None,None,None,report-series,None,report-series,None,False,2018-01-12,None,2018-01-22,2022-04-05,None,2017-09-13,None,2017-09-13,None,2017,issued_date,None,None,None,None,None,<jats:p>This paper explores two new inference ...,This paper explores two new inference rules fo...,None,None,"Reger, Giles; Suda, Martin; Voronkov, Andrei",None,None,"[{""affiliation"": [], ""family"": ""Reger"", ""given...",None,None,None,None,None,None,None,None,2,None,None,2,0,None,None,,,False,None,,None,None,,None,NaN,None,issn,14,None,"{""DOI"": ""10.29007/hsh2"", ""ISSN"": [""2516-2314""]...",10.29007/hsh2,10.29007,10.29007,hsh2,10.29007/hsh,10.29007/hsh,easychair.org,easychair.org/publications
1,crossref::10.29007/g4bq,EasyChair preprint,crossref,10.29007/g4bq,10.29007/g4bq,https://doi.org/10.29007/g4bq,https://easychair.org/publications/preprint/WjKW,https://easychair.org/publications/preprint/WjKW,10.29007,11545,None,None,crossref,EasyChair,EasyChair Preprints,None,None,2516-2314,"Reconstructing Turing's ""paper machine""",None,None,None,None,report-series,None,report-series,None,False,2018-01-12,None,2018-01-22,2022-04-04,None,2017-09-14,None,2017-09-14,None,2017,issued_date,None,None,None,None,None,<jats:p>It is an amazing fact that the very fi...,It is an amazing fact that the very first ches...,None,None,"Kasparov, Garry; Friedel, Frederic",None,None,"[{""affiliation"": [], ""family"": ""Kasparov"", ""gi...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,issn,14,None,"{""DOI"": ""10.29007/g4bq"", ""ISSN"": [""2516-2314""]...",10.29007/g4bq,10.29007,10.29007,g4bq,10.29007/g,10.29007/g,easychair.org,easychair.org/publications
2,crossref::10.29007/pjn4,EasyChair preprint,crossref,10.29007/pjn4,10.29007/pjn4,https://doi.org/10.29007/pjn4,https://easychair.org/publications/preprint/N2sl,https://easychair.org/publications/preprint/N2sl,10.29007,11545,None,None,crossref,EasyChair,EasyChair Preprints,None,None,2516-2314,Computation of Some Integer Sequences in Maple,None,None,None,None,report-series,None,report-series,None,False,2018-01-12,None,2018-01-22,2024-07-11,None,2017-11-20,None,2017-11-20,None,2017,issued_date,None,None,None,None,None,<jats:p>We consider some integer sequences con...,We consider some integer sequences connected w...,None,None,"Fan, W.L.; Jeffrey, David J

In [60]:
EasyChair_parent_df, EasyChair_parent_summary = analyze_parent_data("EasyChair preprint", full_parent_df)


 CONSOLIDATED ANALYSIS: EASYCHAIR PREPRINT
  > Total Parent Groups:     571
  > Total Versions Found:    606
  > Unique Parent DOIs:      571
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for EasyChair preprint:
-----------------------------------
  1 version(s):     539 groups
  2 version(s):     29 groups
  3 version(s):     3 groups

  Average versions per parent: 1.06

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
EasyChair preprint                  541
arXiv                                20
PeerJ Preprints                       3
ResearchGate                          2
EasyChair preprint; arXiv             2
EasyChair preprint; ResearchGate      2
HAL                                   1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.29007/g    10
10.29007/h     8
10.29007/k     8
10.29007/p     7
10.29007/z     7
10.29007/

# EcoEvoRxiv

In [61]:
EcoEvoRxiv_df, EcoEvoRxiv_summary = get_server_data("EcoEvoRxiv")


 SERVER ANALYSIS: ECOEVORXIV
  > Files found:    2
  > Raw records:    2886
  > Cleaned shape:  (2886, 90)
  > Unique DOIs:    2886
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.32942/x      1948
10.32942/osf     935
10.31219/osf       3
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
ecoevorxiv.org    2838
osf.io              48
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.32942    2883
10.31219       3
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
29705    2862
15934      24
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
California Digital Library (CDL)    2862
Center for Open Science               24
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
--------------------

In [62]:
EcoEvoRxiv_parent_df, EcoEvoRxiv_parent_summary = analyze_parent_data("EcoEvoRxiv", full_parent_df)


 CONSOLIDATED ANALYSIS: ECOEVORXIV
  > Total Parent Groups:     2,795
  > Total Versions Found:    2,896
  > Unique Parent DOIs:      2,795
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for EcoEvoRxiv:
-----------------------------------
  1 version(s):     2,721 groups
  2 version(s):     62 groups
  3 version(s):     6 groups
  4 version(s):     2 groups
  5 version(s):     3 groups
  10 version(s):    1 groups

  Average versions per parent: 1.04

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
EcoEvoRxiv                                         2752
Authorea Inc.                                        15
Research Square                                       7
SSRN                                                  5
eLife                                                 4
Open Science Framework                                2
EcoEvoRxiv; Zenodo                                    2
HAL

# EconStor Preprints

In [63]:
EconStor_df, EconStor_summary = get_server_data("EconStor_Preprints")

/tmp/ipykernel_3012/1196196344.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  raw_df = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)



 SERVER ANALYSIS: ECONSTOR_PREPRINTS
  > Files found:    3
  > Raw records:    71761
  > Cleaned shape:  (71761, 90)
  > Unique DOIs:    6867
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
<NA>                     64686
10.18723/diw               588
10.13140/rg                465
10.2373/18                 429
10.1007/s                  377
10.5282/ubm                376
10.34989/swp-              292
10.24406/publica-fhg-      262
10.3929/ethz-a-            242
10.17192/es                232
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
hdl.handle.net           71010
ideas.repec.org            371
econpapers.repec.org       176
econstor.eu                 71
budrich-journals.de         55
zenodo.org                  11
journal-alm.org              7
eprints.lincoln.ac.uk        4
bnarchives.yorku.ca          4
frankfurter-hefte

In [64]:
EconStor_parent_df, EconStor_parent_summary = analyze_parent_data("EconStor Preprints", full_parent_df)


 CONSOLIDATED ANALYSIS: ECONSTOR PREPRINTS
  > Total Parent Groups:     62,784
  > Total Versions Found:    69,018
  > Unique Parent DOIs:      0
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for EconStor Preprints:
-----------------------------------
  1 version(s):     57,156 groups
  2 version(s):     5,127 groups
  3 version(s):     422 groups
  4 version(s):     63 groups
  5 version(s):     14 groups
  8 version(s):     1 groups
  12 version(s):    1 groups

  Average versions per parent: 1.10

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
EconStor Preprints                                           57507
SSRN                                                          4269
RePEc: Research Papers in Economics                            544
AgEcon Search                                                  194
EconStor Preprints; RePEc: Research Papers in Economics        166
ResearchG

# ECSarXiv

In [65]:
ECSarXiv_df, ECSarXiv_summary = get_server_data("ECSarXiv")


 SERVER ANALYSIS: ECSARXIV
  > Files found:    1
  > Raw records:    314
  > Cleaned shape:  (314, 90)
  > Unique DOIs:    314
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.1149/osf    314
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    314
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.1149    314
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
77    314
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
The Electrochemical Society    314
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    314
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
institution_name
None    314
Name

In [66]:
ECSarXiv_parent_df, ECSarXiv_parent_summary = analyze_parent_data("ECSarXiv", full_parent_df)


 CONSOLIDATED ANALYSIS: ECSARXIV
  > Total Parent Groups:     299
  > Total Versions Found:    334
  > Unique Parent DOIs:      299
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for ECSarXiv:
-----------------------------------
  1 version(s):     265 groups
  2 version(s):     33 groups
  3 version(s):     1 groups

  Average versions per parent: 1.12

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
ECSarXiv                                     289
ECSarXiv; Open Science Framework               4
Preprints.org                                  1
SSRN                                           1
engrXiv                                        1
ECSarXiv; arXiv                                1
ECSarXiv; Open Science Framework; engrXiv      1
arXiv                                          1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10

# EdArXiv

In [67]:
EdArXiv_df, EdArXiv_summary = get_server_data("EdArXiv")


 SERVER ANALYSIS: EDARXIV
  > Files found:    1
  > Raw records:    2547
  > Cleaned shape:  (2547, 90)
  > Unique DOIs:    2547
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.35542/osf    2547
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    2547
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.35542    2547
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    2547
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Center for Open Science    2547
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    2547
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
institution_name
None   

In [68]:
EdArXiv_parent_df, EdArXiv_parent_summary = analyze_parent_data("EdArXiv", full_parent_df)


 CONSOLIDATED ANALYSIS: EDARXIV
  > Total Parent Groups:     2,153
  > Total Versions Found:    2,474
  > Unique Parent DOIs:      2,153
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for EdArXiv:
-----------------------------------
  1 version(s):     1,911 groups
  2 version(s):     192 groups
  3 version(s):     38 groups
  4 version(s):     2 groups
  5 version(s):     7 groups
  6 version(s):     1 groups
  7 version(s):     1 groups
  9 version(s):     1 groups

  Average versions per parent: 1.15

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
EdArXiv                                         2077
Thesis Commons                                    11
EdArXiv; ResearchGate                              9
SSRN                                               9
arXiv                                              8
Open Science Framework                             8
EdArXiv; arXiv         

# EGUsphere

In [69]:
EGUsphere_df, EGUsphere_summary = get_server_data("EGUsphere")


 SERVER ANALYSIS: EGUSPHERE
  > Files found:    1
  > Raw records:    15253
  > Cleaned shape:  (15253, 90)
  > Unique DOIs:    15253
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.5194/egusphere-    15249
10.5194/amt-              3
10.5194/hess-             1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
egusphere.copernicus.org          15252
oscar-egusphere.copernicus.org        1
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.5194    15253
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
3145    15253
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Copernicus GmbH    15253
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None

In [70]:
pattern = r'10.5194/amt-|10.5194/hess-'
# pattern = r'v\d+$'
mask = EGUsphere_df['doi'].str.contains(pattern, regex=True, na=False)
result = EGUsphere_df[mask]
result

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
263,crossref::10.5194/amt-2022-295,EGUsphere,crossref,10.5194/amt-2022-295,10.5194/amt-2022-295,https://doi.org/10.5194/amt-2022-295,https://egusphere.copernicus.org/preprints/202...,https://egusphere.copernicus.org/preprints/202...,10.5194,3145,None,None,crossref,Copernicus GmbH,None,None,Gases/In Situ Measurement/Instruments and Plat...,None,Temperature dependent sensitivity of iodide ch...,None,None,None,None,posted-content,preprint,preprint,None,True,2022-05-11,2022-05-11,2023-03-21,2026-02-28,None,2022-05-11,None,2022-05-11,None,2022,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<jats:p>Abstract. Iodide chemical ionization m...,Abstract. Iodide chemical ionization mass spec...,None,None,"Robinson, Michael A.; Neuman, J. Andrew; Huey,...",None,None,"[{""ORCID"": ""https://orcid.org/0000-0003-0977-9...",None,None,"[{""DOI"": ""10.13039/100004800"", ""award"": [""20RD...",California Air Resources Board; NOAA Center fo...,2.0,None,None,None,1,None,None,1,0,None,"{""has-comment"": [{""asserted-by"": ""subject"", ""i...",,10.5194/amt-15-4295-2022,True,None,,None,None,10.5194/amt-2022-295-rc1;10.5194/amt-2022-295-rc2,None,NaN,None,prefix/primary_domain,2,None,"{""DOI"": ""10.5194/amt-2022-295"", ""URL"": ""https:...",10.5194/amt-2022-295,10.5194,10.5194,amt-2022-295,10.5194/amt-,10.5194/amt-,egusphere.copernicus.org,egusphere.copernicus.org/preprints
264,crossref::10.5194/amt-2022-295-supplement,EGUsphere,crossref,10.5194/amt-2022-295-supplement,10.5194/amt-2022-295-supplement,https://doi.org/10.5194/amt-2022-295-supplement,https://egusphere.copernicus.org/preprints/202...,https://egusphere.copernicus.org/preprints/202...,10.5194,3145,None,None,crossref,Copernicus GmbH,None,None,Gases/In Situ Measurement/Instruments and Plat...,None,"Supplementary material to ""Temperature depende...",None,None,None,None,posted-content,other,other,None,True,2022-05-11,2022-05-11,2023-03-21,2026-02-28,None,2022-05-11,None,2022-05-11,None,2022,issued_date,posted_date,None,None,None,None,None,None,None,None,"Robinson, Michael A.; Neuman, J. Andrew; Huey,...",None,None,"[{""ORCID"": ""https://orcid.org/0000-0003-0977-9...",None,None,None,None,NaN,None,None,None,1,None,None,1,0,None,"{""is-supplement-to"": [{""asserted-by"": ""subject...",,,False,None,,None,None,,None,NaN,None,prefix/primary_domain,2,None,"{""DOI"": ""10.5194/amt-2022-295-supplement"", ""UR...",10.5194/amt-2022-295-supplement,10.5194,10.5194,amt-2022-295-supplement,10.5194/amt-,10.5194/amt-,egusphere.copernicus.org,egusphere.copernicus.org/preprints
9390,crossref::10.5194/amt-2024-3967,EGUsphere,crossref,10.5194/amt-2024-3967,10.519

https://doi.org/10.5194/amt-2024-3967 it seems like when publish, the doi change to have the same patterns as the journal doi

In [71]:
EGUsphere_parent_df, EGUsphere_parent_summary = analyze_parent_data("EGUsphere", full_parent_df)


 CONSOLIDATED ANALYSIS: EGUSPHERE
  > Total Parent Groups:     10,181
  > Total Versions Found:    10,300
  > Unique Parent DOIs:      10,181
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for EGUsphere:
-----------------------------------
  1 version(s):     10,069 groups
  2 version(s):     106 groups
  3 version(s):     5 groups
  4 version(s):     1 groups

  Average versions per parent: 1.01

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
EGUsphere                               10144
arXiv                                      20
Research Square                             6
Earth and Space Science Open Archive        3
EarthArXiv                                  3
ResearchGate                                1
SSRN                                        1
Preprints.org                               1
EGUsphere; arXiv                            1
Zenodo                              

# Electron Colloquium Comput Complex

In [72]:
Electron_df, Electron_summary = get_server_data("Electron_Colloquium_Comput_Complex")


 SERVER ANALYSIS: ELECTRON_COLLOQUIUM_COMPUT_COMPLEX
  > Files found:    1
  > Raw records:    227
  > Cleaned shape:  (227, 90)
  > Unique DOIs:    0
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
<NA>    227
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
dblp.uni-trier.de      221
eccc.weizmann.ac.il      6
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
None    227
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
None    227
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
None    227
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    227
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------


In [73]:
Electron_parent_df, Electron_parent_summary = analyze_parent_data("Electron Colloquium Comput Complex", full_parent_df)


 CONSOLIDATED ANALYSIS: ELECTRON COLLOQUIUM COMPUT COMPLEX
  > Total Parent Groups:     139
  > Total Versions Found:    170
  > Unique Parent DOIs:      0
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Electron Colloquium Comput Complex:
-----------------------------------
  1 version(s):     109 groups
  2 version(s):     29 groups
  3 version(s):     1 groups

  Average versions per parent: 1.22

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Electron Colloquium Comput Complex    110
arXiv                                  29

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
<NA>    139



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
dblp.uni-trier.de      134
eccc.weizmann.ac.il      5


 COMPLETED: ELECTRON COLLOQUIUM COMPUT COMPLEX



# eLife

In [74]:
elife_df, elife_summary = get_server_data("eLife")


 SERVER ANALYSIS: ELIFE
  > Files found:    2
  > Raw records:    29901
  > Cleaned shape:  (29901, 90)
  > Unique DOIs:    29901
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.7554/elife    29901
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
elifesciences.org    29901
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.7554    29901
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
4374    29901
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
eLife Sciences Publications, Ltd    29901
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
eLife    21412
None      8489
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
-------------

In [75]:
elife_df[elife_df['type_backend_raw']=='journal-article']

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
0,crossref::10.7554/elife.02516,eLife,crossref,10.7554/elife.02516,10.7554/elife.02516,https://doi.org/10.7554/elife.02516,http://elifesciences.org/lookup/doi/10.7554/eL...,http://elifesciences.org/lookup/doi/10.7554/eL...,10.7554,4374,None,None,crossref,"eLife Sciences Publications, Ltd",eLife,None,None,2050-084X,Correction: A diversity of localized timescale...,None,None,None,en,journal-article,None,journal-article,None,False,2014-08-22,None,2014-08-22,2026-03-17,None,2014-02-18,None,2014-02-18,2014-02-18,2014.0,issued_date,None,None,None,http://creativecommons.org/licenses/by/3.0/,http://creativecommons.org/licenses/by/3.0/,None,None,None,None,"Chaudhuri, Rishidev; Bernacchia, Alberto; Wang...",None,None,"[{""affiliation"": [], ""family"": ""Chaudhuri"", ""g...",None,None,None,None,NaN,None,None,None,18,None,None,18,0,None,None,,,False,None,,None,None,,None,None,None,issn,0,None,"{""DOI"": ""10.7554/elife.02516"", ""ISSN"": [""2050-...",10.7554/elife.02516,10.7554,10.7554,elife.02516,10.7554/elife,10.7554/elife,elifesciences.org,elifesciences.org/lookup
1,crossref::10.7554/elife.08172,eLife,crossref,10.7554/elife.08172,10.7554/elife.08172,https://doi.org/10.7554/elife.08172,http://elifesciences.org/content/4/e08127,http://elifesciences.org/content/4/e08127,10.7554,4374,None,None,crossref,"eLife Sciences Publications, Ltd",eLife,None,None,2050-084X,The number of olfactory stimuli that humans ca...,None,None,None,None,journal-article,None,journal-article,None,False,2015-07-07,None,2015-07-07,2022-04-05,None,2015-07-07,None,2015-07-07,2015-07-07,2015.0,issued_date,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,None,None,issn,0,None,"{""DOI"": ""10.7554/elife.08172"", ""ISSN"": [""2050-...",10.7554/elife.08172,10.7554,10.7554,elife.08172,10.7554/elife,10.7554/elife,elifesciences.org,elifesciences.org/content
3,crossref::10.7554/elife.43558,eLife,crossref,10.7554/elife.43558,10.7554/elife.43558,https://doi.org/10.7554/elife.43558,https://elifesciences.org/articles/43558,https://elifesciences.org/articles/43558,10.7554,4374,None,None,crossref,"eLife Sciences Publications, Ltd",eLife,None,None,2050-084X,Rapid task-dependent tuning of the mouse olfac...,None,None,None,en,journal-article,None,journal-article,None,False,2019-02-06,None,2019-02-06,2026-04-14,None,2019-02-06,None,2019-02-06,2019-02-06,2019.0,issued_date,None,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<jats:p>Adapting neural representation to rapi...,Adapting neural representation to rapidly chan...,"[{""URL"": ""https://cdn.elife

In [76]:
eLife_parent_df, eLife_parent_summary = analyze_parent_data("eLife", full_parent_df)


 CONSOLIDATED ANALYSIS: ELIFE
  > Total Parent Groups:     9,439
  > Total Versions Found:    9,710
  > Unique Parent DOIs:      9,439
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for eLife:
-----------------------------------
  1 version(s):     9,183 groups
  2 version(s):     246 groups
  3 version(s):     7 groups
  4 version(s):     1 groups
  5 version(s):     2 groups

  Average versions per parent: 1.03

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
eLife                                                                   9424
arXiv                                                                      7
medRxiv                                                                    1
Digital Access to Scholarship at Harvard (DASH) (Harvard University)       1
HAL; eLife                                                                 1
arXiv; eLife                                    

# ELPUB (Universitat Wuppertal)

In [77]:
ELPUB_df, ELPUB_summary = get_server_data("ELPUB_(Universitat_Wuppertal)")


 SERVER ANALYSIS: ELPUB_(UNIVERSITAT_WUPPERTAL)
  > Files found:    2
  > Raw records:    41
  > Cleaned shape:  (41, 90)
  > Unique DOIs:    41
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.25926/x          2
10.25926/7qdf       1
10.25926/wrn        1
10.25926/xdz        1
10.25926/cfrc-ad    1
10.25926/fjtg-wr    1
10.25926/fjdf-ae    1
10.25926/790c       1
10.25926/9bys       1
10.25926/2bhg       1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
elpub.bib.uni-wuppertal.de    41
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.25926    41
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
None    41
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
University of Wuppertal            39
Berg

/tmp/ipykernel_3012/1196196344.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  raw_df = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)


In [78]:
ELPUB_parent_df, ELPUB_parent_summary = analyze_parent_data("ELPUB (Universitat Wuppertal)", full_parent_df)


 CONSOLIDATED ANALYSIS: ELPUB (UNIVERSITAT WUPPERTAL)
  > Total Parent Groups:     41
  > Total Versions Found:    41
  > Unique Parent DOIs:      41
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for ELPUB (Universitat Wuppertal):
-----------------------------------
  1 version(s):     41 groups

  Average versions per parent: 1.00

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
ELPUB (Universitat Wuppertal)    41

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.25926/x          2
10.25926/0qp3       1
10.25926/nrcz-jr    1
10.25926/nf         1
10.25926/z          1
10.25926/pdf        1
10.25926/9k6n       1
10.25926/cfrc-ad    1
10.25926/hk         1
10.25926/ysd        1



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
elpub.bib.uni-wuppertal.de    41


 COMPLETED: ELPUB (UNIVERSITAT WUPPERT

# EmeRI

In [79]:
EmeRI_df, EmeRI_summary = get_server_data("EmeRI")


 SERVER ANALYSIS: EMERI
  > Files found:    1
  > Raw records:    8
  > Cleaned shape:  (8, 90)
  > Unique DOIs:    8
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.21452/15    6
10.21452/23    2
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
preprints.ibict.br    8
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.21452    8
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
8875    8
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
ABEC Publicacoes    8
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    8
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
institution_name
EMERI    8
Name:

In [80]:
EmeRI_parent_df, EmeRI_parent_summary = analyze_parent_data("EmeRI", full_parent_df)


 CONSOLIDATED ANALYSIS: EMERI
  > Total Parent Groups:     8
  > Total Versions Found:    8
  > Unique Parent DOIs:      8
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for EmeRI:
-----------------------------------
  1 version(s):     8 groups

  Average versions per parent: 1.00

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
EmeRI    8

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.21452/15    6
10.21452/23    2



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
preprints.ibict.br    8


 COMPLETED: EMERI



# Encyclopedia

In [81]:
Encyclopedia_df, Encyclopedia_summary = get_server_data("Encyclopedia")


 SERVER ANALYSIS: ENCYCLOPEDIA
  > Files found:    1
  > Raw records:    166
  > Cleaned shape:  (166, 90)
  > Unique DOIs:    166
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.32545/encyclopedia    166
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
encyclopedia.pub    166
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.32545    166
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
1968    166
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
MDPI AG    166
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    166
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
institution_name
None    1

In [82]:
Encyclopedia_parent_df, Encyclopedia_parent_summary = analyze_parent_data("Encyclopedia", full_parent_df)


 CONSOLIDATED ANALYSIS: ENCYCLOPEDIA
  > Total Parent Groups:     161
  > Total Versions Found:    167
  > Unique Parent DOIs:      161
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Encyclopedia:
-----------------------------------
  1 version(s):     155 groups
  2 version(s):     6 groups

  Average versions per parent: 1.04

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Encyclopedia     160
Preprints.org      1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.32545/encyclopedia    161



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
encyclopedia.pub    161


 COMPLETED: ENCYCLOPEDIA



# EnerarXiv

In [83]:
EnerarXiv_df, EnerarXiv_summary = get_server_data("EnerarXiv")


 SERVER ANALYSIS: ENERARXIV
  > Files found:    1
  > Raw records:    204
  > Cleaned shape:  (204, 90)
  > Unique DOIs:    204
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.46855/20    204
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
enerarxiv.org    204
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.46855    204
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
26239    204
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Applied Energy Innovation Institute (AEii)    204
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    204
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
instit

In [84]:
EnerarXiv_parent_df, EnerarXiv_parent_summary = analyze_parent_data("EnerarXiv", full_parent_df)


 CONSOLIDATED ANALYSIS: ENERARXIV
  > Total Parent Groups:     191
  > Total Versions Found:    196
  > Unique Parent DOIs:      191
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for EnerarXiv:
-----------------------------------
  1 version(s):     187 groups
  2 version(s):     3 groups
  3 version(s):     1 groups

  Average versions per parent: 1.03

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
EnerarXiv                  188
arXiv                        1
SSRN                         1
EnerarXiv; ResearchGate      1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.46855/20    191



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
enerarxiv.org    191


 COMPLETED: ENERARXIV



# engrXiv

In [85]:
engrXiv_df, engrXiv_summary = get_server_data("engrXiv")


 SERVER ANALYSIS: ENGRXIV
  > Files found:    2
  > Raw records:    4929
  > Cleaned shape:  (4929, 90)
  > Unique DOIs:    4929
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31224/osf    1909
10.31224/43       85
10.31224/30       82
10.31224/36       81
10.31224/40       80
10.31224/34       80
10.31224/42       79
10.31224/31       78
10.31224/25       78
10.31224/38       77
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
engrxiv.org    4839
osf.io           90
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31224    4642
10.31219      73
10.31234      68
10.31227      53
10.31235      39
10.31228      26
10.31223       9
10.31230       6
10.31220       5
10.31225       4
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
33966    4621

In [86]:
engrXiv_parent_df, engrXiv_parent_summary = analyze_parent_data("engrXiv", full_parent_df)


 CONSOLIDATED ANALYSIS: ENGRXIV
  > Total Parent Groups:     4,519
  > Total Versions Found:    5,028
  > Unique Parent DOIs:      4,519
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for engrXiv:
-----------------------------------
  1 version(s):     4,077 groups
  2 version(s):     394 groups
  3 version(s):     37 groups
  4 version(s):     6 groups
  5 version(s):     3 groups
  6 version(s):     1 groups
  7 version(s):     1 groups

  Average versions per parent: 1.11

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
engrXiv                            4238
Open Science Framework               66
SSRN                                 48
Open Science Framework; engrXiv      41
arXiv                                26
TechRxiv                             24
arXiv; engrXiv                       16
Preprints.org                        12
Research Square                      11
ResearchGa

# F1000Research

In [87]:
F1000Research_df, F1000Research_summary = get_server_data("F1000Research")


 SERVER ANALYSIS: F1000RESEARCH
  > Files found:    1
  > Raw records:    16873
  > Cleaned shape:  (16873, 90)
  > Unique DOIs:    16873
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.12688/f      16859
10.3410/f           7
<NA>                4
10.3410/10          1
10.3410/12          1
10.3410/http        1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
f1000research.com        16867
                             2
someurl.com                  2
xy.net                       1
researchdev.f1000.com        1
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.12688    16863
10.3410        10
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
2560    16863
4950       10
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------

In [88]:
F1000Research_parent_df, F1000Research_parent_summary = analyze_parent_data("F1000Research", full_parent_df)


 CONSOLIDATED ANALYSIS: F1000RESEARCH
  > Total Parent Groups:     10,640
  > Total Versions Found:    16,273
  > Unique Parent DOIs:      10,640
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for F1000Research:
-----------------------------------
  1 version(s):     6,408 groups
  2 version(s):     3,177 groups
  3 version(s):     812 groups
  4 version(s):     176 groups
  5 version(s):     48 groups
  6 version(s):     10 groups
  7 version(s):     5 groups
  8 version(s):     2 groups
  9 version(s):     1 groups
  11 version(s):    1 groups

  Average versions per parent: 1.53

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
F1000Research           10622
F1000Research; arXiv        4
ResearchGate                3
SSRN                        2
arXiv                       2
Research Square             2
searchRxiv                  1
Authorea Inc.               1
F1000Research; SSRN  

# FocUS Archive

In [89]:
FocUS_df, FocUS_summary = get_server_data("FocUS_Archive")


 SERVER ANALYSIS: FOCUS_ARCHIVE
  > Files found:    1
  > Raw records:    83
  > Cleaned shape:  (83, 90)
  > Unique DOIs:    83
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31225/osf    70
10.31227/osf     9
10.31219/osf     3
10.31234/osf     1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    83
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31225    70
10.31227     9
10.31219     3
10.31234     1
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    83
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Center for Open Science    83
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    83
Name: count, dtype: int64



In [90]:
FocUS_parent_df, FocUS_parent_summary = analyze_parent_data("FocUS Archive", full_parent_df)


 CONSOLIDATED ANALYSIS: FOCUS ARCHIVE
  > Total Parent Groups:     79
  > Total Versions Found:    119
  > Unique Parent DOIs:      79
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for FocUS Archive:
-----------------------------------
  1 version(s):     43 groups
  2 version(s):     34 groups
  3 version(s):     1 groups
  5 version(s):     1 groups

  Average versions per parent: 1.51

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
FocUS Archive                            43
FocUS Archive; Open Science Framework    36

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.31225/osf    79



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
osf.io    79


 COMPLETED: FOCUS ARCHIVE



# Frenxiv

In [91]:
Frenxiv_df, Frenxiv_summary = get_server_data("Frenxiv")


 SERVER ANALYSIS: FRENXIV
  > Files found:    1
  > Raw records:    179
  > Cleaned shape:  (179, 90)
  > Unique DOIs:    179
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31226/osf    178
10.31227/osf      1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    179
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31226    178
10.31227      1
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    179
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Center for Open Science    179
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    179
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
----------------------------

In [92]:
Frenxiv_parent_df, Frenxiv_parent_summary = analyze_parent_data("Frenxiv", full_parent_df)


 CONSOLIDATED ANALYSIS: FRENXIV
  > Total Parent Groups:     102
  > Total Versions Found:    116
  > Unique Parent DOIs:      102
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Frenxiv:
-----------------------------------
  1 version(s):     88 groups
  2 version(s):     14 groups

  Average versions per parent: 1.14

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Frenxiv                   94
SSRN                       4
Open Science Framework     2
Frenxiv; ResearchGate      1
EdArXiv                    1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.31226/osf    102



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
osf.io    102


 COMPLETED: FRENXIV



# Gates Open Research

In [93]:
Gates_df, Gates_summary = get_server_data("Gates_Open_Research")


 SERVER ANALYSIS: GATES_OPEN_RESEARCH
  > Files found:    1
  > Raw records:    863
  > Cleaned shape:  (863, 90)
  > Unique DOIs:    863
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.12688/gatesopenres    863
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
gatesopenresearch.org    863
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.12688    863
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
2560    863
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
F1000 Research Ltd    863
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
Gates Open Research    862
None                     1
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTI

In [94]:
Gates_parent_df, Gates_parent_summary = analyze_parent_data("Gates Open Research", full_parent_df)


 CONSOLIDATED ANALYSIS: GATES OPEN RESEARCH
  > Total Parent Groups:     502
  > Total Versions Found:    817
  > Unique Parent DOIs:      502
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Gates Open Research:
-----------------------------------
  1 version(s):     236 groups
  2 version(s):     222 groups
  3 version(s):     39 groups
  4 version(s):     5 groups

  Average versions per parent: 1.63

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Gates Open Research    501
VeriXiv                  1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.12688/gatesopenres    502



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
gatesopenresearch.org    502


 COMPLETED: GATES OPEN RESEARCH



In [95]:
Gates_parent_df

,dup_group_id_full,parent_record_id,parent_server_name,parent_doi,parent_url,parent_title,parent_authors,parent_date_first_seen,parent_year_first_seen,version_record_ids,version_dois,servers_with_counts,total_versions,first_version_date,last_version_date,most_recent_version_servers,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
24274,crossref::10.12688/gatesopenres.13123.1,crossref::10.12688/gatesopenres.13123.1,Gates Open Research,10.12688/gatesopenres.13123.1,https://gatesopenresearch.org/articles/4-71/v1,Introducing a drift and diffusion framework fo...,"Lewis, Fraser I; Guga, Godfrey; Mdoe, Paschal;...",2020-06-29,2020,crossref::10.12688/gatesopenres.13123.1; cross...,10.12688/gatesopenres.13123.1; 10.12688/gateso...,Gates Open Research (2),2,2020-06-29,2020-11-26,Gates Open Research,10.12688/gatesopenres.13123.1,10.12688,10.12688,gatesopenres.13123.1,10.12688/gatesopenres,10.12688/gatesopenres,gatesopenresearch.org,gatesopenresearch.org/articles
39601,crossref::10.12688/gatesopenres.13457.1,crossref::10.12688/gatesopenres.13457.1,Gates Open Research,10.12688/gatesopenres.13457.1,https://gatesopenresearch.org/articles/5-174/v1,A Randomized controlled trial of the Effect of...,"Harding, Rebecca; Ataide, Ricardo; Mwangi, Mar...",2021-12-07,2021,crossref::10.12688/gatesopenres.13457.1; cross...,10.12688/gatesopenres.13457.1; 10.12688/gateso...,Gates Open Research (2),2,2021-12-07,2022-04-14,Gates Open Research,10.12688/gatesopenres.13457.1,10.12688,10.12688,gatesopenres.13457.1,10.12688/gatesopenres,10.12688/gatesopenres,gatesopenresearch.org,gatesopenresearch.org/articles
61653,crossref::10.12688/gatesopenres.12799.2,crossref::10.12688/gatesopenres.12799.2,Gates Open Research,10.12688/gatesopenres.12799.2,https://gatesopenresearch.org/articles/2-13/v2,A sulfur-free peptide mimic of surfactant prot...,"Walther, Frans J.; Gupta, Monik; Gordon, Larry...",2018-07-10,2018,crossref::10.12688/gatesopenres.12799.2,10.12688/gatesopenres.12799.2,Gates Open Research (1),1,2018-07-10,2018-07-10,Gates Open Research,10.12688/gatesopenres.12799.2,10.12688,10.12688,gatesopenres.12799.2,10.12688/gatesopenres,10.12688/gatesopenres,gatesopenresearch.org,gatesopenresearch.org/articles
68519,crossref::10.12688/gatesopenres.14591.1,crossref::10.12688/gatesopenres.14591.1,Gates Open Research,10.12688/gatesopenres.14591.1,https://gatesopenresearch.org/articles/7-75/v1,Using responsive feedback from routine monitor...,"Meekers, Dominique; Olutola, Olaniyi; Abu Turk...",2023-05-19,2023,crossref::10.12688/gatesopenres.14591.1; cross...,10.12688/gatesopenres.14591.1; 10.12688/gateso...,Gates Open Research (2),2,2023-05-19,2023-11-16,Gates Open Research,10.12688/gatesopenres.14591.1,10.12688,10.12688,gatesopenres.14591.1,10.12688/gatesopenres,10.12688/gatesopenres,gatesopenresearch.org,gatesopenresearch.org/articles
75609,crossref::10.12688/gatesopenres.12854.1,crossref::10.12688/gatesopenres.12854.1,Gates Open Research,10.12688/gatesopenres.12854.1,https://gatesopenresearch.org/articles/2-38/v1,Evaluation of a multi-level intervention to im...,"Ingabire, Rosine; Nyombayire, Julien; Hoagland...",2018-08-20,2018,crossref::10.12688/gatesopenres.12854.1; cross...,10.12688/gatesopenres.12854.1; 10.12688/gateso...,Gates Open Research (3),3,2018-08-20,2019-02-04,Gates Open Research,10.12688/gatesopenres.12854.1,10.12688,10.12688,gatesopenres.12854.1,10.12688/gatesopenres,10.12688/gatesopenres,gatesopenresearch.org,gatesopenresearch.org/articles
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7265036,crossref::10.12688/gatesopenres.12954.1,crossref::10.12688/gatesopenres.12954.1,Gates Open Research,10.12688/gatesopenres.12954.1,https://gatesopenresearch.org/articles/3-1466/v1,"Knowledge, perception and experience of sexual...","Sule, Aisha I.; Titiloye, Musibau A.; Arulogun...",2019-05-15,2019,crossref::10.12688/gatesopenres.12

In [96]:
Gates_parent_df[Gates_parent_df['most_recent_version_servers']=='VeriXiv']

,dup_group_id_full,parent_record_id,parent_server_name,parent_doi,parent_url,parent_title,parent_authors,parent_date_first_seen,parent_year_first_seen,version_record_ids,version_dois,servers_with_counts,total_versions,first_version_date,last_version_date,most_recent_version_servers,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
7235707,crossref::10.12688/gatesopenres.16360.1,crossref::10.12688/gatesopenres.16360.1,Gates Open Research,10.12688/gatesopenres.16360.1,https://gatesopenresearch.org/articles/9-65/v1,Driving innovation from discovery to access: M...,"Palmer, Shaun; Clark, Rebecca A.; Connell, Bri...",2025-01-01,2025,crossref::10.12688/gatesopenres.16360.1; cross...,10.12688/gatesopenres.16360.1; 10.12688/verixi...,Gates Open Research (1); VeriXiv (1),2,2025-01-01,2025-05-23,VeriXiv,10.12688/gatesopenres.16360.1,10.12688,10.12688,gatesopenres.16360.1,10.12688/gatesopenres,10.12688/gatesopenres,gatesopenresearch.org,gatesopenresearch.org/articles


In [97]:
Gates_df.drop_duplicates('record_id')

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
0,crossref::10.12688/gatesopenres.12803.2,Gates Open Research,crossref,10.12688/gatesopenres.12803.2,10.12688/gatesopenres.12803.2,https://doi.org/10.12688/gatesopenres.12803.2,https://gatesopenresearch.org/articles/2-11/v2,https://gatesopenresearch.org/articles/2-11/v2,10.12688,2560,None,None,crossref,F1000 Research Ltd,Gates Open Research,None,None,2572-4754,A demographic dividend of the FP2020 Initiativ...,None,None,None,en,journal-article,None,journal-article,None,False,2018-07-12,None,2018-07-12,2025-02-21,None,2018-07-12,None,2018-07-12,2018-07-12,2018.0,issued_date,None,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<ns4:p><ns4:bold>Background:</ns4:bold> The de...,"Background: The demographic dividend, defined ...","[{""URL"": ""https://gatesopenresearch.org/articl...",None,"Li, Qingfeng; Rimon, Jose G.",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-6390-6...",None,None,"[{""DOI"": ""10.13039/100000865"", ""award"": [""OPP1...",Bill and Melinda Gates Foundation,1.0,None,None,None,2,None,None,2,22,"[{""DOI"": ""10.1016/S0140-6736(06)69480-4"", ""art...",None,,,False,None,,None,New version,,"[{""DOI"": ""10.12688/gatesopenres.12803.1"", ""lab...",None,https://doi.org/10.12688/gatesopenres.crossmar...,issn,49,None,"{""DOI"": ""10.12688/gatesopenres.12803.2"", ""ISSN...",10.12688/gatesopenres.12803.2,10.12688,10.12688,gatesopenres.12803.2,10.12688/gatesopenres,10.12688/gatesopenres,gatesopenresearch.org,gatesopenresearch.org/articles
1,crossref::10.12688/gatesopenres.12803.1,Gates Open Research,crossref,10.12688/gatesopenres.12803.1,10.12688/gatesopenres.12803.1,https://doi.org/10.12688/gatesopenres.12803.1,https://gatesopenresearch.org/articles/2-11/v1,https://gatesopenresearch.org/articles/2-11/v1,10.12688,2560,None,None,crossref,F1000 Research Ltd,Gates Open Research,None,None,2572-4754,A demographic dividend of the FP2020 Initiativ...,None,None,None,en,journal-article,None,journal-article,None,False,2018-02-22,None,2018-07-12,2025-02-21,None,2018-02-22,None,2018-02-22,2018-02-22,2018.0,issued_date,None,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<ns4:p><ns4:bold>Background:</ns4:bold> The de...,"Background: The demographic dividend, defined ...","[{""URL"": ""https://gatesopenresearch.org/articl...",None,"Li, Qingfeng; Rimon, Jose G.",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-6390-6...",None,None,"[{""DOI"": ""10.13039/100000865"", ""award"": [""OPP1...",Bill and Melinda Gates Foundation,1.0,None,None,None,3,None,None,3,22,"[{""DOI"": ""10.1016/S0140-6736(06)69480-4"", ""art...","{""has-review"": [{""asserted-by"": ""subj

In [98]:
dup_ti = Gates_df.drop_duplicates('title')

In [99]:
duplicates = dup_ti[dup_ti.duplicated('authors_flat')].sort_values(by='authors_flat')
print(len(duplicates))
duplicates

43


,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
128,crossref::10.12688/gatesopenres.12755.1,Gates Open Research,crossref,10.12688/gatesopenres.12755.1,10.12688/gatesopenres.12755.1,https://doi.org/10.12688/gatesopenres.12755.1,https://gatesopenresearch.org/articles/1-6/v1,https://gatesopenresearch.org/articles/1-6/v1,10.12688,2560,None,None,crossref,F1000 Research Ltd,Gates Open Research,None,None,2572-4754,Evaluating integrated development: are we aski...,None,None,None,en,journal-article,None,journal-article,None,False,2017-11-06,None,2019-10-10,2025-11-16,None,2017-11-06,None,2017-11-06,2017-11-06,2017.0,issued_date,None,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<ns4:p><ns4:bold>Background:</ns4:bold> Emergi...,Background: Emerging global transformations - ...,"[{""URL"": ""https://gatesopenresearch.org/articl...",https://gatesopenresearch.org/articles/1-6/v1/pdf,"Ahner-McHaffie, Tessa W; Guest, Greg; Petruney...",None,None,"[{""affiliation"": [], ""family"": ""Ahner-McHaffie...",None,None,"[{""DOI"": ""10.13039/100000865"", ""award"": [""OPP1...",Bill and Melinda Gates Foundation; FHI Foundation,2.0,None,None,None,2,None,None,2,32,"[{""DOI"": ""10.1136/bmj.g5785"", ""article-title"":...","{""has-review"": [{""asserted-by"": ""subject"", ""id...",,,False,None,,None,None,10.21956/gatesopenres.13815.r26084;10.21956/ga...,None,10.12688/gatesopenres.12755.2,https://doi.org/10.12688/gatesopenres.crossmar...,issn,49,None,"{""DOI"": ""10.12688/gatesopenres.12755.1"", ""ISSN...",10.12688/gatesopenres.12755.1,10.12688,10.12688,gatesopenres.12755.1,10.12688/gatesopenres,10.12688/gatesopenres,gatesopenresearch.org,gatesopenresearch.org/articles
245,crossref::10.12688/gatesopenres.12870.2,Gates Open Research,crossref,10.12688/gatesopenres.12870.2,10.12688/gatesopenres.12870.2,https://doi.org/10.12688/gatesopenres.12870.2,https://gatesopenresearch.org/articles/2-52/v2,https://gatesopenresearch.org/articles/2-52/v2,10.12688,2560,None,None,crossref,F1000 Research Ltd,Gates Open Research,None,None,2572-4754,Characterization of fecal sludge as biomass fe...,None,None,None,en,journal-article,None,journal-article,None,False,2020-01-24,None,2020-07-23,2025-06-13,None,2020-01-24,None,2020-01-24,2020-01-24,2020.0,issued_date,None,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<ns5:p><ns5:bold>Background</ns5:bold>: Transf...,Background : Transformative sanitation technol...,"[{""URL"": ""https://gatesopenresearch.org/articl...",https://gatesopenresearch.org/articles/2-52/v2...,"Barani, Viswa; Hegarty-Craver, Meghan; Rosario...",None,None,"[{""affiliation"": [], ""family"": ""Barani"", ""give...",None,None,"[{""DO

In [100]:
Gates_df

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
0,crossref::10.12688/gatesopenres.12803.2,Gates Open Research,crossref,10.12688/gatesopenres.12803.2,10.12688/gatesopenres.12803.2,https://doi.org/10.12688/gatesopenres.12803.2,https://gatesopenresearch.org/articles/2-11/v2,https://gatesopenresearch.org/articles/2-11/v2,10.12688,2560,None,None,crossref,F1000 Research Ltd,Gates Open Research,None,None,2572-4754,A demographic dividend of the FP2020 Initiativ...,None,None,None,en,journal-article,None,journal-article,None,False,2018-07-12,None,2018-07-12,2025-02-21,None,2018-07-12,None,2018-07-12,2018-07-12,2018.0,issued_date,None,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<ns4:p><ns4:bold>Background:</ns4:bold> The de...,"Background: The demographic dividend, defined ...","[{""URL"": ""https://gatesopenresearch.org/articl...",None,"Li, Qingfeng; Rimon, Jose G.",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-6390-6...",None,None,"[{""DOI"": ""10.13039/100000865"", ""award"": [""OPP1...",Bill and Melinda Gates Foundation,1.0,None,None,None,2,None,None,2,22,"[{""DOI"": ""10.1016/S0140-6736(06)69480-4"", ""art...",None,,,False,None,,None,New version,,"[{""DOI"": ""10.12688/gatesopenres.12803.1"", ""lab...",None,https://doi.org/10.12688/gatesopenres.crossmar...,issn,49,None,"{""DOI"": ""10.12688/gatesopenres.12803.2"", ""ISSN...",10.12688/gatesopenres.12803.2,10.12688,10.12688,gatesopenres.12803.2,10.12688/gatesopenres,10.12688/gatesopenres,gatesopenresearch.org,gatesopenresearch.org/articles
1,crossref::10.12688/gatesopenres.12803.1,Gates Open Research,crossref,10.12688/gatesopenres.12803.1,10.12688/gatesopenres.12803.1,https://doi.org/10.12688/gatesopenres.12803.1,https://gatesopenresearch.org/articles/2-11/v1,https://gatesopenresearch.org/articles/2-11/v1,10.12688,2560,None,None,crossref,F1000 Research Ltd,Gates Open Research,None,None,2572-4754,A demographic dividend of the FP2020 Initiativ...,None,None,None,en,journal-article,None,journal-article,None,False,2018-02-22,None,2018-07-12,2025-02-21,None,2018-02-22,None,2018-02-22,2018-02-22,2018.0,issued_date,None,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<ns4:p><ns4:bold>Background:</ns4:bold> The de...,"Background: The demographic dividend, defined ...","[{""URL"": ""https://gatesopenresearch.org/articl...",None,"Li, Qingfeng; Rimon, Jose G.",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-6390-6...",None,None,"[{""DOI"": ""10.13039/100000865"", ""award"": [""OPP1...",Bill and Melinda Gates Foundation,1.0,None,None,None,3,None,None,3,22,"[{""DOI"": ""10.1016/S0140-6736(06)69480-4"", ""art...","{""has-review"": [{""asserted-by"": ""subj

In [101]:
Gates_df

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
0,crossref::10.12688/gatesopenres.12803.2,Gates Open Research,crossref,10.12688/gatesopenres.12803.2,10.12688/gatesopenres.12803.2,https://doi.org/10.12688/gatesopenres.12803.2,https://gatesopenresearch.org/articles/2-11/v2,https://gatesopenresearch.org/articles/2-11/v2,10.12688,2560,None,None,crossref,F1000 Research Ltd,Gates Open Research,None,None,2572-4754,A demographic dividend of the FP2020 Initiativ...,None,None,None,en,journal-article,None,journal-article,None,False,2018-07-12,None,2018-07-12,2025-02-21,None,2018-07-12,None,2018-07-12,2018-07-12,2018.0,issued_date,None,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<ns4:p><ns4:bold>Background:</ns4:bold> The de...,"Background: The demographic dividend, defined ...","[{""URL"": ""https://gatesopenresearch.org/articl...",None,"Li, Qingfeng; Rimon, Jose G.",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-6390-6...",None,None,"[{""DOI"": ""10.13039/100000865"", ""award"": [""OPP1...",Bill and Melinda Gates Foundation,1.0,None,None,None,2,None,None,2,22,"[{""DOI"": ""10.1016/S0140-6736(06)69480-4"", ""art...",None,,,False,None,,None,New version,,"[{""DOI"": ""10.12688/gatesopenres.12803.1"", ""lab...",None,https://doi.org/10.12688/gatesopenres.crossmar...,issn,49,None,"{""DOI"": ""10.12688/gatesopenres.12803.2"", ""ISSN...",10.12688/gatesopenres.12803.2,10.12688,10.12688,gatesopenres.12803.2,10.12688/gatesopenres,10.12688/gatesopenres,gatesopenresearch.org,gatesopenresearch.org/articles
1,crossref::10.12688/gatesopenres.12803.1,Gates Open Research,crossref,10.12688/gatesopenres.12803.1,10.12688/gatesopenres.12803.1,https://doi.org/10.12688/gatesopenres.12803.1,https://gatesopenresearch.org/articles/2-11/v1,https://gatesopenresearch.org/articles/2-11/v1,10.12688,2560,None,None,crossref,F1000 Research Ltd,Gates Open Research,None,None,2572-4754,A demographic dividend of the FP2020 Initiativ...,None,None,None,en,journal-article,None,journal-article,None,False,2018-02-22,None,2018-07-12,2025-02-21,None,2018-02-22,None,2018-02-22,2018-02-22,2018.0,issued_date,None,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<ns4:p><ns4:bold>Background:</ns4:bold> The de...,"Background: The demographic dividend, defined ...","[{""URL"": ""https://gatesopenresearch.org/articl...",None,"Li, Qingfeng; Rimon, Jose G.",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-6390-6...",None,None,"[{""DOI"": ""10.13039/100000865"", ""award"": [""OPP1...",Bill and Melinda Gates Foundation,1.0,None,None,None,3,None,None,3,22,"[{""DOI"": ""10.1016/S0140-6736(06)69480-4"", ""art...","{""has-review"": [{""asserted-by"": ""subj

In [102]:
pattern = "has-preprint"

mask = Gates_df['relations_json'].str.contains(pattern, regex=False, na=False)
result = Gates_df[mask]
print(result.shape)
result

(28, 90)


,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
788,crossref::10.12688/gatesopenres.16311.1,Gates Open Research,crossref,10.12688/gatesopenres.16311.1,10.12688/gatesopenres.16311.1,https://doi.org/10.12688/gatesopenres.16311.1,https://gatesopenresearch.org/articles/8-143/v1,https://gatesopenresearch.org/articles/8-143/v1,10.12688,2560,None,None,crossref,F1000 Research Ltd,Gates Open Research,None,None,2572-4754,An exploration of unusual antimicrobial resist...,None,None,None,en,journal-article,None,journal-article,None,False,2024-12-23,None,2024-12-23,2025-11-21,None,2024-12-23,None,2024-12-23,2024-12-23,2024.0,issued_date,None,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<ns3:p>\n Typhoid fever is ...,Typhoid fever is a significant public health p...,"[{""URL"": ""https://gatesopenresearch.org/articl...",https://gatesopenresearch.org/articles/8-143/v...,"Zuza, Allan; Wailan, Alexander M.; Anscombe, C...",None,None,"[{""affiliation"": [], ""family"": ""Zuza"", ""given""...",None,None,"[{""DOI"": ""10.13039/100010269"", ""award"": [""2173...",Wellcome Trust; Biotechnology and Biological S...,4.0,None,None,None,0,None,None,0,47,"[{""DOI"": ""10.1016/S1473-3099(18)30685-6"", ""art...","{""has-preprint"": [{""asserted-by"": ""subject"", ""...",10.12688/verixiv.77.2,,False,None,,None,None,,None,None,https://doi.org/10.12688/gatesopenres.crossmar...,issn,49,None,"{""DOI"": ""10.12688/gatesopenres.16311.1"", ""ISSN...",10.12688/gatesopenres.16311.1,10.12688,10.12688,gatesopenres.16311.1,10.12688/gatesopenres,10.12688/gatesopenres,gatesopenresearch.org,gatesopenresearch.org/articles
799,crossref::10.12688/gatesopenres.16313.1,Gates Open Research,crossref,10.12688/gatesopenres.16313.1,10.12688/gatesopenres.16313.1,https://doi.org/10.12688/gatesopenres.16313.1,https://gatesopenresearch.org/articles/9-1/v1,https://gatesopenresearch.org/articles/9-1/v1,10.12688,2560,None,None,crossref,F1000 Research Ltd,Gates Open Research,None,None,2572-4754,Automated post-run analysis of arrayed quantit...,None,None,None,en,journal-article,None,journal-article,None,False,2025-01-20,None,2025-01-20,2026-01-05,None,2025-01-20,None,2025-01-20,2025-01-20,2025.0,issued_date,None,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<ns4:p>Background The TaqMan Array Card (TAC) ...,Background The TaqMan Array Card (TAC) is an a...,"[{""URL"": ""https://gatesopenresearch.org/articl...",https://gatesopenresearch.org/articles/9-1/v1/pdf,"Brintz, Ben J.; Operario, Darwin J.; Brown, Da...",None,None,"[{""ORCID"": ""https://orcid.org/0000-0003-4695-0...",None,None,"[{""DOI"": ""10.13039/100000865"", ""award"": [""INV-...",Bill and Melinda

In [103]:
Gates_df['has_preprint'].value_counts().reset_index().sort_values(by='has_preprint')

,has_preprint,count
0,,835
18,10.12688/verixiv.1078.1,1
26,10.12688/verixiv.1155.1,1
25,10.12688/verixiv.1161.1,1
15,10.12688/verixiv.1179.2,1
2,10.12688/verixiv.123.2,1
20,10.12688/verixiv.1323.2,1
8,10.12688/verixiv.15.2,1
19,10.12688/verixiv.1752.1,1
17,10.12688/verixiv.1808.1,1


In [104]:
pattern = "has-review"

mask = Gates_df['relations_json'].str.contains(pattern, regex=False, na=False)
result = Gates_df[mask]
print(result.shape)
result

(735, 90)


,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
1,crossref::10.12688/gatesopenres.12803.1,Gates Open Research,crossref,10.12688/gatesopenres.12803.1,10.12688/gatesopenres.12803.1,https://doi.org/10.12688/gatesopenres.12803.1,https://gatesopenresearch.org/articles/2-11/v1,https://gatesopenresearch.org/articles/2-11/v1,10.12688,2560,None,None,crossref,F1000 Research Ltd,Gates Open Research,None,None,2572-4754,A demographic dividend of the FP2020 Initiativ...,None,None,None,en,journal-article,None,journal-article,None,False,2018-02-22,None,2018-07-12,2025-02-21,None,2018-02-22,None,2018-02-22,2018-02-22,2018.0,issued_date,None,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<ns4:p><ns4:bold>Background:</ns4:bold> The de...,"Background: The demographic dividend, defined ...","[{""URL"": ""https://gatesopenresearch.org/articl...",None,"Li, Qingfeng; Rimon, Jose G.",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-6390-6...",None,None,"[{""DOI"": ""10.13039/100000865"", ""award"": [""OPP1...",Bill and Melinda Gates Foundation,1.0,None,None,None,3,None,None,3,22,"[{""DOI"": ""10.1016/S0140-6736(06)69480-4"", ""art...","{""has-review"": [{""asserted-by"": ""subject"", ""id...",,,False,None,,None,None,10.21956/gatesopenres.13867.r26331;10.21956/ga...,None,10.12688/gatesopenres.12803.2,https://doi.org/10.12688/gatesopenres.crossmar...,issn,49,None,"{""DOI"": ""10.12688/gatesopenres.12803.1"", ""ISSN...",10.12688/gatesopenres.12803.1,10.12688,10.12688,gatesopenres.12803.1,10.12688/gatesopenres,10.12688/gatesopenres,gatesopenresearch.org,gatesopenresearch.org/articles
2,crossref::10.12688/gatesopenres.12816.1,Gates Open Research,crossref,10.12688/gatesopenres.12816.1,10.12688/gatesopenres.12816.1,https://doi.org/10.12688/gatesopenres.12816.1,https://gatesopenresearch.org/articles/2-24/v1,https://gatesopenresearch.org/articles/2-24/v1,10.12688,2560,None,None,crossref,F1000 Research Ltd,Gates Open Research,None,None,2572-4754,Funding global health product R&amp;D: the Por...,None,None,None,en,journal-article,None,journal-article,None,False,2018-04-26,None,2018-07-19,2025-02-21,None,2018-04-26,None,2018-04-26,2018-04-26,2018.0,issued_date,None,None,None,https://creativecommons.org/licenses/by/3.0/igo/,https://creativecommons.org/licenses/by/3.0/igo/,<ns4:p><ns4:bold>Background</ns4:bold>: the Po...,Background : the Portfolio-To-Impact (P2I) mod...,"[{""URL"": ""https://gatesopenresearch.org/articl...",None,"Terry, Robert F; Yamey, Gavin; Miyazaki-Krause...",None,None,"[{""ORCID"": ""https://orcid.org/0000-0003-3849-7...",None,None,"[{""award"": [""OPP1151682""], ""award-info"": [{""aw...",Bill and Melinda Gates Foundation; Swiss Agenc...,3.0,None,None,None,2,N

In [105]:
pattern = "has-review"

mask = ~Gates_df['relations_json'].str.contains(pattern, regex=False, na=False)
result = Gates_df[mask]
print(result.shape)
result

(128, 90)


,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
0,crossref::10.12688/gatesopenres.12803.2,Gates Open Research,crossref,10.12688/gatesopenres.12803.2,10.12688/gatesopenres.12803.2,https://doi.org/10.12688/gatesopenres.12803.2,https://gatesopenresearch.org/articles/2-11/v2,https://gatesopenresearch.org/articles/2-11/v2,10.12688,2560,None,None,crossref,F1000 Research Ltd,Gates Open Research,None,None,2572-4754,A demographic dividend of the FP2020 Initiativ...,None,None,None,en,journal-article,None,journal-article,None,False,2018-07-12,None,2018-07-12,2025-02-21,None,2018-07-12,None,2018-07-12,2018-07-12,2018.0,issued_date,None,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<ns4:p><ns4:bold>Background:</ns4:bold> The de...,"Background: The demographic dividend, defined ...","[{""URL"": ""https://gatesopenresearch.org/articl...",None,"Li, Qingfeng; Rimon, Jose G.",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-6390-6...",None,None,"[{""DOI"": ""10.13039/100000865"", ""award"": [""OPP1...",Bill and Melinda Gates Foundation,1.0,None,None,None,2,None,None,2,22,"[{""DOI"": ""10.1016/S0140-6736(06)69480-4"", ""art...",None,,,False,None,,None,New version,,"[{""DOI"": ""10.12688/gatesopenres.12803.1"", ""lab...",None,https://doi.org/10.12688/gatesopenres.crossmar...,issn,49,None,"{""DOI"": ""10.12688/gatesopenres.12803.2"", ""ISSN...",10.12688/gatesopenres.12803.2,10.12688,10.12688,gatesopenres.12803.2,10.12688/gatesopenres,10.12688/gatesopenres,gatesopenresearch.org,gatesopenresearch.org/articles
6,crossref::10.12688/gatesopenres.12817.2,Gates Open Research,crossref,10.12688/gatesopenres.12817.2,10.12688/gatesopenres.12817.2,https://doi.org/10.12688/gatesopenres.12817.2,https://gatesopenresearch.org/articles/2-23/v2,https://gatesopenresearch.org/articles/2-23/v2,10.12688,2560,None,None,crossref,F1000 Research Ltd,Gates Open Research,None,None,2572-4754,Developing new health technologies for neglect...,None,None,None,en,journal-article,None,journal-article,None,False,2018-08-22,None,2018-08-22,2025-11-01,None,2018-08-22,None,2018-08-22,2018-08-22,2018.0,issued_date,None,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<ns4:p><ns4:bold>Background</ns4:bold>: Fundin...,Background : Funding for neglected disease pro...,"[{""URL"": ""https://gatesopenresearch.org/articl...",None,"Young, Ruth; Bekele, Tewodros; Gunn, Alexander...",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-8787-2...",None,None,"[{""DOI"": ""10.13039/100000865"", ""award"": [""OPP1...",Bill and Melinda Gates Foundation,1.0,None,None,None,4,None,None,4,22,"[{""DOI"": ""10.1016/S0140-6736(13)62105-4"", ""art...",None,,,False,None,,None,

In [106]:
Gates_df

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
0,crossref::10.12688/gatesopenres.12803.2,Gates Open Research,crossref,10.12688/gatesopenres.12803.2,10.12688/gatesopenres.12803.2,https://doi.org/10.12688/gatesopenres.12803.2,https://gatesopenresearch.org/articles/2-11/v2,https://gatesopenresearch.org/articles/2-11/v2,10.12688,2560,None,None,crossref,F1000 Research Ltd,Gates Open Research,None,None,2572-4754,A demographic dividend of the FP2020 Initiativ...,None,None,None,en,journal-article,None,journal-article,None,False,2018-07-12,None,2018-07-12,2025-02-21,None,2018-07-12,None,2018-07-12,2018-07-12,2018.0,issued_date,None,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<ns4:p><ns4:bold>Background:</ns4:bold> The de...,"Background: The demographic dividend, defined ...","[{""URL"": ""https://gatesopenresearch.org/articl...",None,"Li, Qingfeng; Rimon, Jose G.",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-6390-6...",None,None,"[{""DOI"": ""10.13039/100000865"", ""award"": [""OPP1...",Bill and Melinda Gates Foundation,1.0,None,None,None,2,None,None,2,22,"[{""DOI"": ""10.1016/S0140-6736(06)69480-4"", ""art...",None,,,False,None,,None,New version,,"[{""DOI"": ""10.12688/gatesopenres.12803.1"", ""lab...",None,https://doi.org/10.12688/gatesopenres.crossmar...,issn,49,None,"{""DOI"": ""10.12688/gatesopenres.12803.2"", ""ISSN...",10.12688/gatesopenres.12803.2,10.12688,10.12688,gatesopenres.12803.2,10.12688/gatesopenres,10.12688/gatesopenres,gatesopenresearch.org,gatesopenresearch.org/articles
1,crossref::10.12688/gatesopenres.12803.1,Gates Open Research,crossref,10.12688/gatesopenres.12803.1,10.12688/gatesopenres.12803.1,https://doi.org/10.12688/gatesopenres.12803.1,https://gatesopenresearch.org/articles/2-11/v1,https://gatesopenresearch.org/articles/2-11/v1,10.12688,2560,None,None,crossref,F1000 Research Ltd,Gates Open Research,None,None,2572-4754,A demographic dividend of the FP2020 Initiativ...,None,None,None,en,journal-article,None,journal-article,None,False,2018-02-22,None,2018-07-12,2025-02-21,None,2018-02-22,None,2018-02-22,2018-02-22,2018.0,issued_date,None,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<ns4:p><ns4:bold>Background:</ns4:bold> The de...,"Background: The demographic dividend, defined ...","[{""URL"": ""https://gatesopenresearch.org/articl...",None,"Li, Qingfeng; Rimon, Jose G.",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-6390-6...",None,None,"[{""DOI"": ""10.13039/100000865"", ""award"": [""OPP1...",Bill and Melinda Gates Foundation,1.0,None,None,None,3,None,None,3,22,"[{""DOI"": ""10.1016/S0140-6736(06)69480-4"", ""art...","{""has-review"": [{""asserted-by"": ""subj

In [107]:
Gates_df['funders_flat']

0                      Bill and Melinda Gates Foundation
1                      Bill and Melinda Gates Foundation
2      Bill and Melinda Gates Foundation; Swiss Agenc...
3      The Special Programme for Research and Trainin...
4      National Institute of General Medical Sciences...
                             ...                        
858    National Institute of Allergy and Infectious D...
859                                     Gates Foundation
860    Programa grand challenges explorations-brazil:...
861                                                 None
862                                     Gates Foundation
Name: funders_flat, Length: 863, dtype: object

In [108]:
# Optional: Group similar names together
df_funders = Gates_df['funders_flat'].dropna().str.split(';').explode().str.strip()

# # Normalize common variants
# df_funders = df_funders.replace({
#     "Gates Foundation": "Bill and Melinda Gates Foundation"
# })

final_counts = df_funders.value_counts().to_frame()
final_counts.head(10)

,count
funders_flat,
Bill and Melinda Gates Foundation,703
Gates Foundation,52
National Institutes of Health,42
United States Agency for International Development,36
Wellcome Trust,28
Medical Research Council,26
South African Medical Research Council,17
"Department for International Development, UK Government",14
Bill & Melinda Gates Foundation,11


# HAL

In [109]:
# HAL_df, HAL_summary = get_server_data("HAL")

In [110]:
HAL_parent_df, HAL_parent_summary = analyze_parent_data("HAL", full_parent_df)


 CONSOLIDATED ANALYSIS: HAL
  > Total Parent Groups:     1,030,229
  > Total Versions Found:    1,052,746
  > Unique Parent DOIs:      27,498
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for HAL:
-----------------------------------
  1 version(s):     1,009,984 groups
  2 version(s):     18,605 groups
  3 version(s):     1,302 groups
  4 version(s):     220 groups
  5 version(s):     58 groups
  6 version(s):     24 groups
  7 version(s):     17 groups
  8 version(s):     4 groups
  9 version(s):     3 groups
  10 version(s):    3 groups
  11 version(s):    3 groups
  12 version(s):    2 groups
  14 version(s):    1 groups
  17 version(s):    1 groups
  18 version(s):    2 groups

  Average versions per parent: 1.02

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
HAL                                         1026395
arXiv                                          1582
ResearchGate      

# HANS Publication PrePrints

In [111]:
HANS_df, HANS_summary = get_server_data("HANS_Publication_PrePrints")


 SERVER ANALYSIS: HANS_PUBLICATION_PREPRINTS
  > Files found:    1
  > Raw records:    75
  > Cleaned shape:  (75, 90)
  > Unique DOIs:    75
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.12677/hanspreprints    75
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
hanspub.org    75
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.12677    75
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
4945    75
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Hans Publishers    75
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
HANS Publication PrePrints    75
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
--------------------------

In [112]:
HANS_parent_df, HANS_parent_summary = analyze_parent_data("HANS Publication PrePrints", full_parent_df)


 CONSOLIDATED ANALYSIS: HANS PUBLICATION PREPRINTS
  > Total Parent Groups:     75
  > Total Versions Found:    75
  > Unique Parent DOIs:      75
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for HANS Publication PrePrints:
-----------------------------------
  1 version(s):     75 groups

  Average versions per parent: 1.00

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
HANS Publication PrePrints    75

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.12677/hanspreprints    75



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
hanspub.org    75


 COMPLETED: HANS PUBLICATION PREPRINTS



# HRB Open Research

In [113]:
HRB_df, HRB_summary = get_server_data("HRB_Open_Research")


 SERVER ANALYSIS: HRB_OPEN_RESEARCH
  > Files found:    1
  > Raw records:    1012
  > Cleaned shape:  (1012, 90)
  > Unique DOIs:    1012
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.12688/hrbopenres    1012
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
hrbopenresearch.org    1012
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.12688    1012
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
2560    1012
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
F1000 Research Ltd    1012
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
HRB Open Research    1012
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------

In [114]:
HRB_parent_df, HRB_parent_summary = analyze_parent_data("HRB Open Research", full_parent_df)


 CONSOLIDATED ANALYSIS: HRB OPEN RESEARCH
  > Total Parent Groups:     647
  > Total Versions Found:    1,007
  > Unique Parent DOIs:      647
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for HRB Open Research:
-----------------------------------
  1 version(s):     342 groups
  2 version(s):     254 groups
  3 version(s):     47 groups
  4 version(s):     4 groups

  Average versions per parent: 1.56

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
HRB Open Research    646
SSRN                   1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.12688/hrbopenres    647



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
hrbopenresearch.org    647


 COMPLETED: HRB OPEN RESEARCH



# Humanities Commons CORE

In [115]:
CORE_df, CORE_summary = get_server_data("Humanities_Commons_CORE")


 SERVER ANALYSIS: HUMANITIES_COMMONS_CORE
  > Files found:    2
  > Raw records:    29584
  > Cleaned shape:  (29584, 90)
  > Unique DOIs:    29584
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.17613/m    2950
10.17613/j     289
10.17613/s     279
10.17613/e     273
10.17613/b     272
10.17613/r     270
10.17613/z     270
10.17613/y     270
10.17613/w     267
10.17613/t     266
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
works.hcommons.org    20522
hcommons.org           9062
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.17613    29584
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
None    29584
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
unknown                                 

In [116]:
CORE_parent_df, CORE_parent_summary = analyze_parent_data("Humanities Commons CORE", full_parent_df)


 CONSOLIDATED ANALYSIS: HUMANITIES COMMONS CORE
  > Total Parent Groups:     16,685
  > Total Versions Found:    29,313
  > Unique Parent DOIs:      16,685
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Humanities Commons CORE:
-----------------------------------
  1 version(s):     5,298 groups
  2 version(s):     10,710 groups
  3 version(s):     318 groups
  4 version(s):     288 groups
  5 version(s):     21 groups
  6 version(s):     26 groups
  7 version(s):     4 groups
  8 version(s):     9 groups
  9 version(s):     2 groups
  10 version(s):    3 groups
  12 version(s):    3 groups
  14 version(s):    2 groups
  16 version(s):    1 groups

  Average versions per parent: 1.76

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Humanities Commons CORE            16631
SSRN                                  16
HAL; Humanities Commons CORE           8
Open Science Framework        

# IACR Cryptology ePrint Archive

In [117]:
IACR_df, IACR_summary = get_server_data("IACR_Cryptology_ePrint_Archive")


 SERVER ANALYSIS: IACR_CRYPTOLOGY_EPRINT_ARCHIVE
  > Files found:    1
  > Raw records:    11904
  > Cleaned shape:  (11904, 90)
  > Unique DOIs:    208
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
<NA>                  11696
10.48550/arxiv           72
10.13140/rg              28
10.60882/cispa           25
10.5281/zenodo           18
10.1007/97               11
10.3929/ethz-b-           5
10.7916/d                 3
10.4230/dagsemproc        3
10.3217/jucs-             3
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
eprint.iacr.org         8228
dblp.uni-trier.de       3493
csrc.nist.gov             15
cacr.uwaterloo.ca         12
researchgate.net           7
caislab.kaist.ac.kr        6
131002.net                 6
cr.yp.to                   6
apps.dtic.mil              5
people.csail.mit.edu       5
Name: count, dtype: int64

In [118]:
IACR_parent_df, IACR_parent_summary = analyze_parent_data("IACR Cryptology ePrint Archive", full_parent_df)


 CONSOLIDATED ANALYSIS: IACR CRYPTOLOGY EPRINT ARCHIVE
  > Total Parent Groups:     10,430
  > Total Versions Found:    10,679
  > Unique Parent DOIs:      0
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for IACR Cryptology ePrint Archive:
-----------------------------------
  1 version(s):     10,196 groups
  2 version(s):     224 groups
  3 version(s):     8 groups
  5 version(s):     1 groups
  6 version(s):     1 groups

  Average versions per parent: 1.02

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
IACR Cryptology ePrint Archive                                   10307
arXiv                                                              110
HAL                                                                  5
ResearchGate                                                         3
SSRN                                                                 2
IACR Cryptology ePrint Archiv

# INA-Rxiv

In [119]:
INA_df, INA_summary = get_server_data("INA-Rxiv")


 SERVER ANALYSIS: INA-RXIV
  > Files found:    1
  > Raw records:    17837
  > Cleaned shape:  (17837, 90)
  > Unique DOIs:    17837
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31227/osf    15764
10.31219/osf      784
10.31234/osf      448
10.31228/osf      227
10.31235/osf      219
10.31223/osf      183
10.31230/osf       60
10.31220/osf       55
10.31229/osf       48
10.31225/osf       44
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    17837
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31227    15764
10.31219      784
10.31234      448
10.31228      227
10.31235      219
10.31223      183
10.31230       60
10.31220       55
10.31229       48
10.31225       44
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    1759

In [120]:
INA_parent_df, INA_parent_summary = analyze_parent_data("INA-Rxiv", full_parent_df)


 CONSOLIDATED ANALYSIS: INA-RXIV
  > Total Parent Groups:     15,055
  > Total Versions Found:    20,750
  > Unique Parent DOIs:      15,055
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for INA-Rxiv:
-----------------------------------
  1 version(s):     11,033 groups
  2 version(s):     3,072 groups
  3 version(s):     711 groups
  4 version(s):     134 groups
  5 version(s):     42 groups
  6 version(s):     23 groups
  7 version(s):     7 groups
  8 version(s):     6 groups
  9 version(s):     5 groups
  10 version(s):    5 groups
  12 version(s):    3 groups
  13 version(s):    2 groups
  14 version(s):    1 groups
  17 version(s):    1 groups
  18 version(s):    2 groups
  19 version(s):    1 groups
  20 version(s):    2 groups
  21 version(s):    1 groups
  22 version(s):    1 groups
  27 version(s):    1 groups
  50 version(s):    1 groups
  56 version(s):    1 groups

  Average versions per parent: 1.38

 MOST RECENT DESTINATIONS

# IndiaRxiv

In [121]:
IndiaRxiv_df, IndiaRxiv_summary = get_server_data("IndiaRxiv")


 SERVER ANALYSIS: INDIARXIV
  > Files found:    2
  > Raw records:    142
  > Cleaned shape:  (142, 90)
  > Unique DOIs:    142
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.35543/osf          135
10.35543/indiarxiv      7
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io             135
ops.iihr.res.in      7
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.35543    142
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
34961    98
15934    44
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Open Access India          98
Center for Open Science    44
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None                            

In [122]:
IndiaRxiv_parent_df, IndiaRxiv_parent_summary = analyze_parent_data("IndiaRxiv", full_parent_df)


 CONSOLIDATED ANALYSIS: INDIARXIV
  > Total Parent Groups:     103
  > Total Versions Found:    110
  > Unique Parent DOIs:      103
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for IndiaRxiv:
-----------------------------------
  1 version(s):     96 groups
  2 version(s):     7 groups

  Average versions per parent: 1.07

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
IndiaRxiv          98
ChemRxiv            1
arXiv               1
engrXiv             1
Research Square     1
Thesis Commons      1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.35543/osf          97
10.35543/indiarxiv     6



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
osf.io             97
ops.iihr.res.in     6


 COMPLETED: INDIARXIV



# JMIR Preprints

In [17]:
JMIR_df, JMIR_summary = get_server_data("JMIR_Preprints")


 SERVER ANALYSIS: JMIR_PREPRINTS
  > Files found:    1
  > Raw records:    37631
  > Cleaned shape:  (37631, 90)
  > Unique DOIs:    37631
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.2196/preprints    37630
10.2196/iproc            1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
preprints.jmir.org    37631
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.2196    37631
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
1010    37631
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
JMIR Publications Inc.    37631
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    37631
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAM

In [18]:
JMIR_parent_df, JMIR_parent_summary = analyze_parent_data("JMIR Preprints", full_parent_df)


 CONSOLIDATED ANALYSIS: JMIR PREPRINTS
  > Total Parent Groups:     35,807
  > Total Versions Found:    37,218
  > Unique Parent DOIs:      35,807
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for JMIR Preprints:
-----------------------------------
  1 version(s):     34,506 groups
  2 version(s):     1,234 groups
  3 version(s):     59 groups
  4 version(s):     5 groups
  5 version(s):     2 groups
  37 version(s):    1 groups

  Average versions per parent: 1.04

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
JMIR Preprints              35450
Research Square               107
medRxiv                        86
arXiv                          53
SSRN                           35
Preprints.org                  16
PsyArXiv                       13
JMIR Preprints; PsyArXiv        7
JMIR Preprints; arXiv           5
bioRxiv                         4

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKE

# Jxiv

In [19]:
Jxiv_df, Jxiv_summary = get_server_data("Jxiv")


 SERVER ANALYSIS: JXIV
  > Files found:    1
  > Raw records:    902
  > Cleaned shape:  (902, 90)
  > Unique DOIs:    902
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.51094/jxiv    902
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
jxiv.jst.go.jp    902
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.51094    902
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
None    902
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Jxiv, JST Preprint Server    902
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    902
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
institution_name
None    9

In [20]:
Jxiv_parent_df, Jxiv_parent_summary = analyze_parent_data("Jxiv", full_parent_df)


 CONSOLIDATED ANALYSIS: JXIV
  > Total Parent Groups:     757
  > Total Versions Found:    772
  > Unique Parent DOIs:      757
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Jxiv:
-----------------------------------
  1 version(s):     746 groups
  2 version(s):     10 groups
  6 version(s):     1 groups

  Average versions per parent: 1.02

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Jxiv               753
SSRN                 2
Research Square      2

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.51094/jxiv    757



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
jxiv.jst.go.jp    757


 COMPLETED: JXIV



# Keldysh Institute Preprints

In [21]:
Keldysh_df, Keldysh_summary = get_server_data("Keldysh_Institute_Preprints")


 SERVER ANALYSIS: KELDYSH_INSTITUTE_PREPRINTS
  > Files found:    1
  > Raw records:    1258
  > Cleaned shape:  (1258, 90)
  > Unique DOIs:    1258
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.20948/prepr-       1257
10.20948/preprints       1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
keldysh.ru            1257
library.keldysh.ru       1
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.20948    1258
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
8521    1258
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Keldysh Institute of Applied Mathematics    1258
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
Keldysh Institute Prep

In [22]:
Keldysh_parent_df, Keldysh_parent_summary = analyze_parent_data("Keldysh Institute Preprints", full_parent_df)


 CONSOLIDATED ANALYSIS: KELDYSH INSTITUTE PREPRINTS
  > Total Parent Groups:     1,193
  > Total Versions Found:    1,257
  > Unique Parent DOIs:      1,193
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Keldysh Institute Preprints:
-----------------------------------
  1 version(s):     1,135 groups
  2 version(s):     54 groups
  3 version(s):     3 groups
  5 version(s):     1 groups

  Average versions per parent: 1.05

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Keldysh Institute Preprints    1185
arXiv                             8

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.20948/prepr-       1192
10.20948/preprints       1



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
keldysh.ru            1192
library.keldysh.ru       1


 COMPLETED: KELDYSH INSTITUTE PREPRINTS



# LatArXiv

In [23]:
LatArXiv_df, LatArXiv_summary = get_server_data("LatArXiv")


 SERVER ANALYSIS: LATARXIV
  > Files found:    1
  > Raw records:    125
  > Cleaned shape:  (125, 90)
  > Unique DOIs:    125
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.62059/latarxiv    125
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
preprints.latarxiv.org    125
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.62059    125
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
48409    125
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Paideia Studio (publications)    125
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    125
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
insti

In [24]:
LatArXiv_parent_df, LatArXiv_parent_summary = analyze_parent_data("LatArXiv", full_parent_df)


 CONSOLIDATED ANALYSIS: LATARXIV
  > Total Parent Groups:     115
  > Total Versions Found:    118
  > Unique Parent DOIs:      115
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for LatArXiv:
-----------------------------------
  1 version(s):     112 groups
  2 version(s):     3 groups

  Average versions per parent: 1.03

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
LatArXiv            114
SciELO Preprints      1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.62059/latarxiv    115



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
preprints.latarxiv.org    115


 COMPLETED: LATARXIV



# Law Archive

In [25]:
Law_Archive_df, Law_Archive_summary = get_server_data("Law_Archive")


 SERVER ANALYSIS: LAW_ARCHIVE
  > Files found:    1
  > Raw records:    1808
  > Cleaned shape:  (1808, 90)
  > Unique DOIs:    1808
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31228/osf    837
10.31219/osf    470
10.31227/osf    241
10.31234/osf    175
10.31235/osf     38
10.31231/osf     17
10.31223/osf     12
10.31224/osf      6
10.31230/osf      6
10.31229/osf      3
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    1808
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31228    837
10.31219    470
10.31227    241
10.31234    175
10.31235     38
10.31231     17
10.31223     12
10.31224      6
10.31230      6
10.31229      3
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    1794
29705      12
33966       2
Name: count,

In [26]:
Law_Archive_parent_df, Law_Archive_parent_summary = analyze_parent_data("Law Archive", full_parent_df)


 CONSOLIDATED ANALYSIS: LAW ARCHIVE
  > Total Parent Groups:     1,344
  > Total Versions Found:    2,212
  > Unique Parent DOIs:      1,344
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Law Archive:
-----------------------------------
  1 version(s):     634 groups
  2 version(s):     578 groups
  3 version(s):     124 groups
  4 version(s):     4 groups
  5 version(s):     2 groups
  11 version(s):    1 groups
  13 version(s):    1 groups

  Average versions per parent: 1.65

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Law Archive                            681
Law Archive; Open Science Framework    471
Open Science Framework                 144
SSRN                                    32
arXiv                                    5
EdArXiv                                  3
CrimRxiv                                 2
Law Archive; arXiv                       2
EdArXiv; Law Archiv

# LIS Scholarship Archive

In [27]:
LIS_df, LIS_summary = get_server_data("LIS_Scholarship_Archive")


 SERVER ANALYSIS: LIS_SCHOLARSHIP_ARCHIVE
  > Files found:    1
  > Raw records:    397
  > Cleaned shape:  (397, 90)
  > Unique DOIs:    397
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31229/osf    290
10.31219/osf     44
10.31227/osf     21
10.31228/osf      9
10.31235/osf      7
10.31234/osf      7
10.31223/osf      7
10.31230/osf      5
10.31225/osf      3
10.31220/osf      2
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    397
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31229    290
10.31219     44
10.31227     21
10.31228      9
10.31235      7
10.31234      7
10.31223      7
10.31230      5
10.31225      3
10.31220      2
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    388
29705      7
242        2
Name: c

In [28]:
LIS_parent_df, LIS_parent_summary = analyze_parent_data("LIS Scholarship Archive", full_parent_df)


 CONSOLIDATED ANALYSIS: LIS SCHOLARSHIP ARCHIVE
  > Total Parent Groups:     328
  > Total Versions Found:    488
  > Unique Parent DOIs:      328
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for LIS Scholarship Archive:
-----------------------------------
  1 version(s):     201 groups
  2 version(s):     97 groups
  3 version(s):     27 groups
  4 version(s):     3 groups

  Average versions per parent: 1.49

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
LIS Scholarship Archive                            217
LIS Scholarship Archive; Open Science Framework     96
Open Science Framework                               3
Thesis Commons                                       3
Frenxiv                                              2
LIS Scholarship Archive; ResearchGate                1
SSRN                                                 1
Zenodo                                           

# LSE Research Online Documents on Economics

In [29]:
LSE_df, LSE_summary = get_server_data("LSE_Research_Online_Documents_on_Economics")


 SERVER ANALYSIS: LSE_RESEARCH_ONLINE_DOCUMENTS_ON_ECONOMICS
  > Files found:    1
  > Raw records:    119
  > Cleaned shape:  (119, 90)
  > Unique DOIs:    12
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
<NA>           107
10.5089/97       3
10.1429/85       2
10.7208/97       1
10.2760/27       1
10.1425/84       1
10.60692/b       1
10.13140/rg      1
10.7916/d        1
10.1108/s        1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
ideas.repec.org          74
eprints.lse.ac.uk        24
cep.lse.ac.uk             5
sticerd.lse.ac.uk         3
elibrary.imf.org          2
foundation.org.uk         1
cris.unibo.it             1
europepmc.org             1
carnegieendowment.org     1
dialnet.unirioja.es       1
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
None        107
10.5089     

In [30]:
LSE_parent_df, LSE_parent_summary = analyze_parent_data("LSE Research Online Documents on Economics", full_parent_df)


 CONSOLIDATED ANALYSIS: LSE RESEARCH ONLINE DOCUMENTS ON ECONOMICS
  > Total Parent Groups:     110
  > Total Versions Found:    144
  > Unique Parent DOIs:      11
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for LSE Research Online Documents on Economics:
-----------------------------------
  1 version(s):     86 groups
  2 version(s):     15 groups
  3 version(s):     8 groups
  4 version(s):     1 groups

  Average versions per parent: 1.31

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
LSE Research Online Documents on Economics                        86
SSRN                                                              14
RePEc: Research Papers in Economics                                9
EconStor Preprints; LSE Research Online Documents on Economics     1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
<NA>          99
10.508

# MarXiv

In [31]:
MarXiv_df, MarXiv_summary = get_server_data("MarXiv")


 SERVER ANALYSIS: MARXIV
  > Files found:    1
  > Raw records:    508
  > Cleaned shape:  (508, 90)
  > Unique DOIs:    508
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31230/osf    317
10.31227/osf     60
10.31219/osf     57
10.31235/osf     23
10.31228/osf     19
10.31234/osf     13
10.31223/osf      9
10.31220/osf      5
10.31225/osf      5
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    508
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31230    317
10.31227     60
10.31219     57
10.31235     23
10.31228     19
10.31234     13
10.31223      9
10.31220      5
10.31225      5
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    494
29705      9
242        5
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
-----

In [32]:
MarXiv_parent_df, MarXiv_parent_summary = analyze_parent_data("MarXiv", full_parent_df)


 CONSOLIDATED ANALYSIS: MARXIV
  > Total Parent Groups:     509
  > Total Versions Found:    672
  > Unique Parent DOIs:      509
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for MarXiv:
-----------------------------------
  1 version(s):     360 groups
  2 version(s):     135 groups
  3 version(s):     14 groups

  Average versions per parent: 1.32

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
MarXiv                            489
MarXiv; Open Science Framework     17
INA-Rxiv                            1
Open Science Framework              1
EdArXiv                             1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.31230/osf    476
10.17605/osf     33



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
osf.io        476
marxiv.org     33


 COMPLETED: MARXIV



# MediArXiv

In [33]:
MediArXiv_df, MediArXiv_summary = get_server_data("MediArXiv")


 SERVER ANALYSIS: MEDIARXIV
  > Files found:    1
  > Raw records:    309
  > Cleaned shape:  (309, 90)
  > Unique DOIs:    309
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.33767/osf    306
10.31219/osf      3
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    309
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.33767    306
10.31219      3
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    309
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Center for Open Science    309
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    309
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
--------------------------

In [34]:
MediArXiv_parent_df, MediArXiv_parent_summary = analyze_parent_data("MediArXiv", full_parent_df)


 CONSOLIDATED ANALYSIS: MEDIARXIV
  > Total Parent Groups:     277
  > Total Versions Found:    299
  > Unique Parent DOIs:      277
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for MediArXiv:
-----------------------------------
  1 version(s):     260 groups
  2 version(s):     13 groups
  3 version(s):     3 groups
  4 version(s):     1 groups

  Average versions per parent: 1.08

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
MediArXiv                  270
SSRN                         2
Zenodo                       1
SocArXiv                     1
Humanities Commons CORE      1
arXiv                        1
APSA Preprints               1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.33767/osf    274
10.31219/osf      3



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
osf.io    277


 COM

# medRxiv

In [35]:
medRxiv_df, medRxiv_summary = get_server_data("medRxiv")


 SERVER ANALYSIS: MEDRXIV
  > Files found:    1
  > Raw records:    75743
  > Cleaned shape:  (75743, 90)
  > Unique DOIs:    75743
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.1101/20    74951
10.1101/19      792
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
medrxiv.org    75743
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.1101    75743
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
246    75743
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Cold Spring Harbor Laboratory    75743
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    75743
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
--------------------

In [36]:
medRxiv_parent_df, medRxiv_parent_summary = analyze_parent_data("medRxiv", full_parent_df)


 CONSOLIDATED ANALYSIS: MEDRXIV
  > Total Parent Groups:     74,055
  > Total Versions Found:    78,040
  > Unique Parent DOIs:      74,055
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for medRxiv:
-----------------------------------
  1 version(s):     70,734 groups
  2 version(s):     2,928 groups
  3 version(s):     246 groups
  4 version(s):     38 groups
  5 version(s):     94 groups
  6 version(s):     15 groups

  Average versions per parent: 1.05

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
medRxiv                   70925
Research Square            1194
SSRN                        693
eLife                       388
JMIR Preprints              305
Authorea Inc.                94
arXiv                        62
Wellcome Open Research       53
Preprints.org                39
F1000Research                38

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
----------------------------

# MetaArXiv

In [37]:
MetaArXiv_df, MetaArXiv_summary = get_server_data("MetaArXiv")


 SERVER ANALYSIS: METAARXIV
  > Files found:    1
  > Raw records:    880
  > Cleaned shape:  (880, 90)
  > Unique DOIs:    880
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31222/osf    851
10.31219/osf     11
10.31227/osf      7
10.31234/osf      4
10.31235/osf      4
10.31231/osf      1
10.31223/osf      1
10.31228/osf      1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    880
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31222    851
10.31219     11
10.31227      7
10.31234      4
10.31235      4
10.31231      1
10.31223      1
10.31228      1
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    879
29705      1
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Center for

In [38]:
MetaArXiv_parent_df, MetaArXiv_parent_summary = analyze_parent_data("MetaArXiv", full_parent_df)


 CONSOLIDATED ANALYSIS: METAARXIV
  > Total Parent Groups:     705
  > Total Versions Found:    907
  > Unique Parent DOIs:      705
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for MetaArXiv:
-----------------------------------
  1 version(s):     547 groups
  2 version(s):     124 groups
  3 version(s):     26 groups
  4 version(s):     6 groups
  5 version(s):     2 groups

  Average versions per parent: 1.29

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
MetaArXiv                            633
SSRN                                  14
MetaArXiv; Open Science Framework     10
Open Science Framework                10
SocArXiv                               6
eLife                                  5
Frenxiv                                4
Research Square                        3
Thesis Commons                         3
medRxiv                                2

 TOP VALUES FOR: DOI_

# MindRxiv

In [39]:
MindRxiv_df, MindRxiv_summary = get_server_data("MindRxiv")


 SERVER ANALYSIS: MINDRXIV
  > Files found:    1
  > Raw records:    335
  > Cleaned shape:  (335, 90)
  > Unique DOIs:    335
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31231/osf    258
10.31227/osf     38
10.31219/osf     15
10.31234/osf      7
10.31228/osf      5
10.31235/osf      5
10.31229/osf      4
10.31223/osf      2
10.31225/osf      1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    335
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31231    258
10.31227     38
10.31219     15
10.31234      7
10.31228      5
10.31235      5
10.31229      4
10.31223      2
10.31225      1
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    333
29705      2
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
----------------

In [40]:
MindRxiv_parent_df, MindRxiv_parent_summary = analyze_parent_data("MindRxiv", full_parent_df)


 CONSOLIDATED ANALYSIS: MINDRXIV
  > Total Parent Groups:     281
  > Total Versions Found:    402
  > Unique Parent DOIs:      281
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for MindRxiv:
-----------------------------------
  1 version(s):     198 groups
  2 version(s):     45 groups
  3 version(s):     38 groups

  Average versions per parent: 1.43

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
MindRxiv                            245
MindRxiv; Open Science Framework     32
MindRxiv; PsyArXiv                    1
SSRN                                  1
EdArXiv; MindRxiv                     1
Open Science Framework                1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.31231/osf    281



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
osf.io    281


 COMPLETED: MINDRXIV



# MNI Open Research

In [41]:
MNI_df, MNI_summary = get_server_data("MNI_Open_Research")


 SERVER ANALYSIS: MNI_OPEN_RESEARCH
  > Files found:    1
  > Raw records:    20
  > Cleaned shape:  (20, 90)
  > Unique DOIs:    20
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.12688/mniopenres    20
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
mniopenresearch.org    20
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.12688    20
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
2560    20
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
F1000 Research Ltd    20
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
MNI Open Research    19
None                  1
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------

In [42]:
MNI_parent_df, MNI_parent_summary = analyze_parent_data("MNI Open Research", full_parent_df)


 CONSOLIDATED ANALYSIS: MNI OPEN RESEARCH
  > Total Parent Groups:     13
  > Total Versions Found:    19
  > Unique Parent DOIs:      13
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for MNI Open Research:
-----------------------------------
  1 version(s):     7 groups
  2 version(s):     6 groups

  Average versions per parent: 1.46

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
MNI Open Research    13

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.12688/mniopenres    13



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
mniopenresearch.org    13


 COMPLETED: MNI OPEN RESEARCH



# Munich Personal RePEc Archive

In [43]:
Munich_df, Munich_summary = get_server_data("Munich_Personal_RePEc_Archive")

/tmp/ipykernel_2758/1196196344.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  raw_df = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)



 SERVER ANALYSIS: MUNICH_PERSONAL_REPEC_ARCHIVE
  > Files found:    7
  > Raw records:    68692
  > Cleaned shape:  (68692, 90)
  > Unique DOIs:    1427
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
<NA>              67264
10.13140/rg         476
10.48550/arxiv      333
10.5281/zenodo      100
10.13140/2           60
10.22004/ag          25
10.6084/m            15
10.17605/osf         12
10.17192/es           7
10.1108/ijse-         7
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
theses.fr                    50259
None                         16501
doi.org                        899
escholarship.org               449
arxiv.org                      191
library.tue.nl                 120
hdl.handle.net                  70
eprints.iisc.ac.in              48
eref.uni-bayreuth.de            30
repositorio.banrep.gov.co       24
Name:

In [44]:
Munich_parent_df, Munich_parent_summary = analyze_parent_data("Munich Personal RePEc Archive", full_parent_df)


 CONSOLIDATED ANALYSIS: MUNICH PERSONAL REPEC ARCHIVE
  > Total Parent Groups:     66,342
  > Total Versions Found:    68,859
  > Unique Parent DOIs:      0
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Munich Personal RePEc Archive:
-----------------------------------
  1 version(s):     63,971 groups
  2 version(s):     2,255 groups
  3 version(s):     100 groups
  4 version(s):     10 groups
  5 version(s):     3 groups
  7 version(s):     1 groups
  8 version(s):     2 groups

  Average versions per parent: 1.04

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Munich Personal RePEc Archive                                         64032
SSRN                                                                   1784
HAL; Munich Personal RePEc Archive                                      219
arXiv                                                                    90
RePEc: Research Pap

# National Bureau of Economic Research

In [45]:
National_Bureau_df, National_Bureau_summary = get_server_data("National_Bureau_of_Economic_Research")


 SERVER ANALYSIS: NATIONAL_BUREAU_OF_ECONOMIC_RESEARCH
  > Files found:    5
  > Raw records:    1856
  > Cleaned shape:  (1856, 90)
  > Unique DOIs:    28
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
<NA>                      1828
10.7916/d                    9
10.7208/97                   5
10.5089/97                   3
10.1007/s                    3
10.17863/cam                 2
10.1002/10                   1
10.23668/psycharchives       1
10.13140/2                   1
10.57912/23                  1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
ideas.repec.org                  1337
eric.ed.gov                       485
econpapers.repec.org                7
ecsocman.hse.ru                     4
nber.org                            3
healthpolicy.fsi.stanford.edu       2
elibrary.imf.org                    2
europepmc.org   

In [46]:
National_Bureau_parent_df, National_Bureau_parent_summary = analyze_parent_data("National Bureau of Economic Research", full_parent_df)


 CONSOLIDATED ANALYSIS: NATIONAL BUREAU OF ECONOMIC RESEARCH
  > Total Parent Groups:     1,663
  > Total Versions Found:    2,265
  > Unique Parent DOIs:      0
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for National Bureau of Economic Research:
-----------------------------------
  1 version(s):     1,131 groups
  2 version(s):     474 groups
  3 version(s):     48 groups
  4 version(s):     8 groups
  5 version(s):     2 groups

  Average versions per parent: 1.36

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
National Bureau of Economic Research                                         1146
SSRN                                                                          381
RePEc: Research Papers in Economics                                            91
National Bureau of Economic Research; RePEc: Research Papers in Economics      28
AgEcon Search                                 

# Nature Precedings

In [47]:
Nature_df, Nature_summary = get_server_data("Nature_Precedings")


 SERVER ANALYSIS: NATURE_PRECEDINGS
  > Files found:    1
  > Raw records:    5210
  > Cleaned shape:  (5210, 90)
  > Unique DOIs:    5210
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.1038/npre    5210
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
nature.com               3312
precedings.nature.com    1898
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.1038    5210
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
297    5210
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Springer Science and Business Media LLC    5210
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
Nature Precedings    5172
None                   38
Name: coun

In [48]:
Nature_parent_df, Nature_parent_summary = analyze_parent_data("Nature Precedings", full_parent_df)


 CONSOLIDATED ANALYSIS: NATURE PRECEDINGS
  > Total Parent Groups:     3,153
  > Total Versions Found:    5,187
  > Unique Parent DOIs:      3,153
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Nature Precedings:
-----------------------------------
  1 version(s):     1,394 groups
  2 version(s):     1,576 groups
  3 version(s):     120 groups
  4 version(s):     51 groups
  5 version(s):     7 groups
  6 version(s):     2 groups
  10 version(s):    3 groups

  Average versions per parent: 1.65

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Nature Precedings                         3111
arXiv                                       23
Nature Precedings; arXiv                     8
SSRN                                         3
F1000Research                                2
bioRxiv                                      2
DSpace@MIT                                   1
viXra            

# NewAddictionsX

In [49]:
NewAddictionsX_df, NewAddictionsX_summary = get_server_data("NewAddictionsX")


 SERVER ANALYSIS: NEWADDICTIONSX
  > Files found:    1
  > Raw records:    7
  > Cleaned shape:  (7, 90)
  > Unique DOIs:    7
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31219/osf    7
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    7
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31219    7
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    7
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Center for Open Science    7
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    7
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
institution_name
None    7
Name: count, dtyp

In [50]:
NewAddictionsX_parent_df, NewAddictionsX_parent_summary = analyze_parent_data("NewAddictionsX", full_parent_df)


 CONSOLIDATED ANALYSIS: NEWADDICTIONSX
  > Total Parent Groups:     4
  > Total Versions Found:    7
  > Unique Parent DOIs:      4
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for NewAddictionsX:
-----------------------------------
  1 version(s):     2 groups
  2 version(s):     1 groups
  3 version(s):     1 groups

  Average versions per parent: 1.75

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
NewAddictionsX    4

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.31219/osf    4



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
osf.io    4


 COMPLETED: NEWADDICTIONSX



# NutriXiv

In [51]:
NutriXiv_df, NutriXiv_summary = get_server_data("NutriXiv")


 SERVER ANALYSIS: NUTRIXIV
  > Files found:    1
  > Raw records:    94
  > Cleaned shape:  (94, 90)
  > Unique DOIs:    94
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31232/osf    70
10.31219/osf    12
10.31228/osf     6
10.31227/osf     5
10.31235/osf     1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    94
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31232    70
10.31219    12
10.31228     6
10.31227     5
10.31235     1
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    94
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Center for Open Science    94
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    9

In [52]:
NutriXiv_parent_df, NutriXiv_parent_summary = analyze_parent_data("NutriXiv", full_parent_df)


 CONSOLIDATED ANALYSIS: NUTRIXIV
  > Total Parent Groups:     83
  > Total Versions Found:    111
  > Unique Parent DOIs:      83
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for NutriXiv:
-----------------------------------
  1 version(s):     63 groups
  2 version(s):     16 groups
  3 version(s):     2 groups
  4 version(s):     1 groups
  6 version(s):     1 groups

  Average versions per parent: 1.34

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
NutriXiv                            66
NutriXiv; Open Science Framework    14
SSRN                                 1
IndiaRxiv                            1
Research Square                      1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.31232/osf    83



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
osf.io    83


 COMPLETED: NUTRIXIV



# Open Research Africa

In [53]:
openra_df, openra_summary = get_server_data("Open_Research_Africa")


 SERVER ANALYSIS: OPEN_RESEARCH_AFRICA
  > Files found:    1
  > Raw records:    288
  > Cleaned shape:  (288, 90)
  > Unique DOIs:    288
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.12688/aasopenres       222
10.12688/openresafrica     66
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
aasopenresearch.org       176
openresearchafrica.org    112
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.12688    288
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
2560    288
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
F1000 Research Ltd    288
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
AAS Open Research       175
Open Research Afr

In [54]:
openra_parent_df, openra_parent_summary = analyze_parent_data("Open Research Africa", full_parent_df)


 CONSOLIDATED ANALYSIS: OPEN RESEARCH AFRICA
  > Total Parent Groups:     180
  > Total Versions Found:    276
  > Unique Parent DOIs:      180
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Open Research Africa:
-----------------------------------
  1 version(s):     99 groups
  2 version(s):     66 groups
  3 version(s):     15 groups

  Average versions per parent: 1.53

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Open Research Africa    179
F1000Research             1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.12688/aasopenres       134
10.12688/openresafrica     46



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
aasopenresearch.org       112
openresearchafrica.org     68


 COMPLETED: OPEN RESEARCH AFRICA



# Open Research Europe

In [55]:
openre_df, openre_summary = get_server_data("Open_Research_Europe")


 SERVER ANALYSIS: OPEN_RESEARCH_EUROPE
  > Files found:    1
  > Raw records:    1877
  > Cleaned shape:  (1877, 90)
  > Unique DOIs:    1877
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.12688/openreseurope    1877
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
open-research-europe.ec.europa.eu    1877
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.12688    1877
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
2560    1877
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
F1000 Research Ltd    1877
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
Open Research Europe    1877
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_

In [56]:
openre_parent_df, openre_parent_summary = analyze_parent_data("Open Research Europe", full_parent_df)


 CONSOLIDATED ANALYSIS: OPEN RESEARCH EUROPE
  > Total Parent Groups:     1,115
  > Total Versions Found:    1,776
  > Unique Parent DOIs:      1,115
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Open Research Europe:
-----------------------------------
  1 version(s):     563 groups
  2 version(s):     455 groups
  3 version(s):     86 groups
  4 version(s):     10 groups
  5 version(s):     1 groups

  Average versions per parent: 1.59

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Open Research Europe    1110
arXiv                      3
SSRN                       1
Zenodo                     1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.12688/openreseurope    1115



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
open-research-europe.ec.europa.eu    1115


 COMPLETED: OPEN RESEARCH 

# Open Science Framework

In [57]:
osf_df, osf_summary = get_server_data("Open_Science_Framework")

/tmp/ipykernel_2758/1196196344.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  raw_df = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)



 SERVER ANALYSIS: OPEN_SCIENCE_FRAMEWORK
  > Files found:    3
  > Raw records:    119481
  > Cleaned shape:  (119481, 90)
  > Unique DOIs:    119481
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31219/osf    101598
10.17605/osf     15829
10.31227/osf       649
10.31234/osf       434
10.31235/osf       421
10.31228/osf       213
10.31223/osf       110
10.31220/osf        56
10.31229/osf        55
10.31224/osf        47
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io               112545
doi.org                4615
psyarxiv.com           1327
eartharxiv.org          262
thesiscommons.org       216
marxiv.org              158
engrxiv.org             137
arabixiv.org             82
mindrxiv.org             46
agrixiv.org              40
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefi

In [58]:
osf_df['primary_domain'].value_counts()

primary_domain
osf.io               112545
doi.org                4615
psyarxiv.com           1327
eartharxiv.org          262
thesiscommons.org       216
marxiv.org              158
engrxiv.org             137
arabixiv.org             82
mindrxiv.org             46
agrixiv.org              40
paleorxiv.org            30
ecsarxiv.org             12
ezid.cdlib.org           11
Name: count, dtype: int64

In [59]:
osf_parent_df, osf_parent_summary = analyze_parent_data("Open Science Framework", full_parent_df)


 CONSOLIDATED ANALYSIS: OPEN SCIENCE FRAMEWORK
  > Total Parent Groups:     94,335
  > Total Versions Found:    110,556
  > Unique Parent DOIs:      94,335
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Open Science Framework:
-----------------------------------
  1 version(s):     83,245 groups
  2 version(s):     8,579 groups
  3 version(s):     1,634 groups
  4 version(s):     516 groups
  5 version(s):     158 groups
  6 version(s):     69 groups
  7 version(s):     42 groups
  8 version(s):     26 groups
  9 version(s):     20 groups
  10 version(s):    8 groups
  11 version(s):    7 groups
  12 version(s):    4 groups
  13 version(s):    3 groups
  14 version(s):    4 groups
  15 version(s):    3 groups
  17 version(s):    2 groups
  19 version(s):    1 groups
  20 version(s):    1 groups
  21 version(s):    1 groups
  22 version(s):    1 groups
  25 version(s):    1 groups
  28 version(s):    1 groups
  30 version(s):    1 groups

In [60]:
osf_parent_df[osf_parent_df['primary_domain']=='agrixiv.org']

,dup_group_id_full,parent_record_id,parent_server_name,parent_doi,parent_url,parent_title,parent_authors,parent_date_first_seen,parent_year_first_seen,version_record_ids,version_dois,servers_with_counts,total_versions,first_version_date,last_version_date,most_recent_version_servers,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend


In [61]:
osf_parent_df[osf_parent_df['primary_domain']=='engrxiv.org']

,dup_group_id_full,parent_record_id,parent_server_name,parent_doi,parent_url,parent_title,parent_authors,parent_date_first_seen,parent_year_first_seen,version_record_ids,version_dois,servers_with_counts,total_versions,first_version_date,last_version_date,most_recent_version_servers,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend


https://engrxiv.org/n7ajq  /  https://engrxiv.org/index.php/engrxiv/preprint/view/102 

10.17605/osf.io/n7ajq == https://doi.org/10.31224/osf.io/n7ajq

In [62]:
osf_parent_df[osf_parent_df['primary_domain']=='arabixiv.org']

,dup_group_id_full,parent_record_id,parent_server_name,parent_doi,parent_url,parent_title,parent_authors,parent_date_first_seen,parent_year_first_seen,version_record_ids,version_dois,servers_with_counts,total_versions,first_version_date,last_version_date,most_recent_version_servers,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend


In [63]:
osf_parent_df[osf_parent_df['primary_domain']=='thesiscommons.org']

,dup_group_id_full,parent_record_id,parent_server_name,parent_doi,parent_url,parent_title,parent_authors,parent_date_first_seen,parent_year_first_seen,version_record_ids,version_dois,servers_with_counts,total_versions,first_version_date,last_version_date,most_recent_version_servers,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend


# Organic Eprints

In [64]:
Organic_Eprints_df, Organic_Eprints_summary = get_server_data("Organic_Eprints")


 SERVER ANALYSIS: ORGANIC_EPRINTS
  > Files found:    1
  > Raw records:    13983
  > Cleaned shape:  (13983, 90)
  > Unique DOIs:    252
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
<NA>              13704
10.5281/zenodo       35
10.3220/rep          30
10.13140/rg          29
10.15454/1           22
10.5169/seals-       14
10.22004/ag          13
10.13140/2           10
10.3920/97            4
10.5073/jfk           3
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
None             12257
orgprints.org     1441
doi.org            260
                    24
bioaktuell.ch        1
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
None        13704
10.17180       71
10.13140       39
10.15454       36
10.5281        35
10.3220        32
10.5169        14
10.22004       13
10.3920         4
10.

In [65]:
Organic_parent_df, Organic_parent_summary = analyze_parent_data("Organic Eprints", full_parent_df)


 CONSOLIDATED ANALYSIS: ORGANIC EPRINTS
  > Total Parent Groups:     13,510
  > Total Versions Found:    13,731
  > Unique Parent DOIs:      0
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Organic Eprints:
-----------------------------------
  1 version(s):     13,304 groups
  2 version(s):     192 groups
  3 version(s):     13 groups
  4 version(s):     1 groups

  Average versions per parent: 1.02

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Organic Eprints         13471
AgEcon Search              22
HAL                        10
HAL; Organic Eprints        7

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
<NA>    13510



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
None             12079
orgprints.org     1406
                    24
bioaktuell.ch        1


 COMPLETED: ORGANIC EPRINTS


# Oroboros Instruments

In [66]:
Oroboros_Instruments_df, Oroboros_Instruments_summary = get_server_data("Oroboros_Instruments")


 SERVER ANALYSIS: OROBOROS_INSTRUMENTS
  > Files found:    2
  > Raw records:    95
  > Cleaned shape:  (95, 90)
  > Unique DOIs:    95
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.26124/mitofit    67
10.26124/bec        19
10.26124/becprep     9
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
wiki.oroboros.at                    37
bioenergetics-communications.org    28
mitofit.org                         27
mitoeagle.org                        3
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.26124    95
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
None    95
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
MitoFit Preprints                49
MitoFit Preprint Archives        18
Bioener

/tmp/ipykernel_2758/1196196344.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  raw_df = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)


In [67]:
Oroboros_parent_df, Oroboros_parent_summary = analyze_parent_data("Oroboros Instruments", full_parent_df)


 CONSOLIDATED ANALYSIS: OROBOROS INSTRUMENTS
  > Total Parent Groups:     68
  > Total Versions Found:    89
  > Unique Parent DOIs:      68
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Oroboros Instruments:
-----------------------------------
  1 version(s):     52 groups
  2 version(s):     14 groups
  3 version(s):     1 groups
  6 version(s):     1 groups

  Average versions per parent: 1.31

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Oroboros Instruments    68

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.26124/mitofit    45
10.26124/bec        15
10.26124/becprep     8



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
wiki.oroboros.at                    30
bioenergetics-communications.org    23
mitofit.org                         15


 COMPLETED: OROBOROS INSTRUMENTS



# PaleorXiv

In [68]:
PaleorXiv_df, PaleorXiv_summary = get_server_data("PaleorXiv")


 SERVER ANALYSIS: PALEORXIV
  > Files found:    1
  > Raw records:    287
  > Cleaned shape:  (287, 90)
  > Unique DOIs:    287
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31233/osf    202
10.31227/osf     23
10.31219/osf     18
10.31231/osf     17
10.31235/osf      8
10.31228/osf      5
10.31229/osf      5
10.31234/osf      3
10.31223/osf      3
10.31230/osf      2
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    287
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31233    202
10.31227     23
10.31219     18
10.31231     17
10.31235      8
10.31228      5
10.31229      5
10.31234      3
10.31223      3
10.31230      2
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    284
29705      3
Name: count, dtype: int64



TOP V

In [69]:
PaleorXiv_parent_df, PaleorXiv_parent_summary = analyze_parent_data("PaleorXiv", full_parent_df)


 CONSOLIDATED ANALYSIS: PALEORXIV
  > Total Parent Groups:     243
  > Total Versions Found:    362
  > Unique Parent DOIs:      243
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for PaleorXiv:
-----------------------------------
  1 version(s):     150 groups
  2 version(s):     72 groups
  3 version(s):     17 groups
  4 version(s):     3 groups
  5 version(s):     1 groups

  Average versions per parent: 1.49

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
PaleorXiv                            191
Open Science Framework; PaleorXiv     51
Preprints.org                          1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.31233/osf    243



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
osf.io    243


 COMPLETED: PALEORXIV



# PeerJ Preprints

In [70]:
PeerJ_df, PeerJ_summary = get_server_data("PeerJ_Preprints")


 SERVER ANALYSIS: PEERJ_PREPRINTS
  > Files found:    1
  > Raw records:    6446
  > Cleaned shape:  (6446, 90)
  > Unique DOIs:    6446
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.7287/peerj    6446
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
peerj.com    6446
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.7287    6446
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
4443    6446
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
PeerJ    6446
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    6446
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
institution_name
None    6446
Na

In [71]:
PeerJ_parent_df, PeerJ_parent_summary = analyze_parent_data("PeerJ Preprints", full_parent_df)


 CONSOLIDATED ANALYSIS: PEERJ PREPRINTS
  > Total Parent Groups:     5,141
  > Total Versions Found:    6,440
  > Unique Parent DOIs:      5,141
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for PeerJ Preprints:
-----------------------------------
  1 version(s):     4,216 groups
  2 version(s):     686 groups
  3 version(s):     161 groups
  4 version(s):     46 groups
  5 version(s):     22 groups
  6 version(s):     5 groups
  7 version(s):     3 groups
  12 version(s):    2 groups

  Average versions per parent: 1.25

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
PeerJ Preprints           5082
arXiv                       20
F1000Research               11
Open Science Framework       3
bioRxiv                      3
Zenodo                       3
MarXiv                       2
ResearchGate                 2
HAL                          2
PeerJ Preprints; arXiv       2

 TOP VALUES

# PhilSci-Archive

In [72]:
PhilSci_df, PhilSci_summary = get_server_data("PhilSci-Archive")


 SERVER ANALYSIS: PHILSCI-ARCHIVE
  > Files found:    1
  > Raw records:    2362
  > Cleaned shape:  (2362, 90)
  > Unique DOIs:    195
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
<NA>              2167
10.48550/arxiv     146
10.13140/rg         18
10.6084/m            5
10.5281/zenodo       3
10.17863/cam         2
10.2143/lea          2
10.22381/rcp         1
10.24338/abs-        1
10.25455/wgtn        1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
None                        1676
philsci-archive.pitt.edu     505
doi.org                      181
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
None        2167
10.48550     146
10.13140      19
10.6084        5
10.5281        3
10.17863       2
10.2143        2
10.22381       1
10.25455       1
10.24338       1
Name: count, dtype: int

In [73]:
PhilSci_parent_df, PhilSci_parent_summary = analyze_parent_data("PhilSci-Archive", full_parent_df)


 CONSOLIDATED ANALYSIS: PHILSCI-ARCHIVE
  > Total Parent Groups:     2,129
  > Total Versions Found:    2,165
  > Unique Parent DOIs:      0
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for PhilSci-Archive:
-----------------------------------
  1 version(s):     2,097 groups
  2 version(s):     29 groups
  3 version(s):     2 groups
  4 version(s):     1 groups

  Average versions per parent: 1.02

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
PhilSci-Archive    2108
arXiv                18
bioRxiv               1
ResearchGate          1
HAL                   1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
<NA>    2129



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
None                        1645
philsci-archive.pitt.edu     484


 COMPLETED: PHILSCI-ARCHIVE



# PoolText

In [74]:
PoolText_df, PoolText_summary = get_server_data("PoolText")


 SERVER ANALYSIS: POOLTEXT
  > Files found:    1
  > Raw records:    79
  > Cleaned shape:  (79, 90)
  > Unique DOIs:    79
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31923/pooltext-preprint-    78
10.31923/55                     1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
content.pooltext.com    79
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31923    79
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
16838    79
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
PoolText, Inc    79
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    79
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
-------------------

In [75]:
PoolText_parent_df, PoolText_parent_summary = analyze_parent_data("PoolText", full_parent_df)


 CONSOLIDATED ANALYSIS: POOLTEXT
  > Total Parent Groups:     79
  > Total Versions Found:    79
  > Unique Parent DOIs:      79
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for PoolText:
-----------------------------------
  1 version(s):     79 groups

  Average versions per parent: 1.00

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
PoolText    79

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.31923/pooltext-preprint-    78
10.31923/55                     1



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
content.pooltext.com    79


 COMPLETED: POOLTEXT



# prepare@u

In [76]:
prepare_df, prepare_summary = get_server_data("prepare@u")


 SERVER ANALYSIS: PREPARE@U
  > Files found:    1
  > Raw records:    227
  > Cleaned shape:  (227, 90)
  > Unique DOIs:    227
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.36375/prepare    227
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
preprint.prepare.org.in    223
prepare.enggtalks.com        2
prepare.org.in               2
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.36375    227
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
22141    227
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
CALNESTOR Knowledge Solutions Private Limited    227
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None                           

In [77]:
prepare_parent_df, prepare_parent_summary = analyze_parent_data("prepare@u", full_parent_df)


 CONSOLIDATED ANALYSIS: PREPARE@U
  > Total Parent Groups:     224
  > Total Versions Found:    227
  > Unique Parent DOIs:      224
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for prepare@u:
-----------------------------------
  1 version(s):     221 groups
  2 version(s):     3 groups

  Average versions per parent: 1.01

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
prepare@u          223
Research Square      1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.36375/prepare    224



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
preprint.prepare.org.in    220
prepare.org.in               2
prepare.enggtalks.com        2


 COMPLETED: PREPARE@U



# Preprints.org

In [78]:
Preprints_df, Preprints_summary = get_server_data("Preprints.org")


 SERVER ANALYSIS: PREPRINTS.ORG
  > Files found:    1
  > Raw records:    115815
  > Cleaned shape:  (115815, 90)
  > Unique DOIs:    115815
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.20944/preprints    115815
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
preprints.org    115815
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.20944    115815
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
1968    115815
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
MDPI AG    115815
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    115815
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
inst

In [79]:
Preprints_parent_df, Preprints_parent_summary = analyze_parent_data("Preprints.org", full_parent_df)


 CONSOLIDATED ANALYSIS: PREPRINTS.ORG
  > Total Parent Groups:     103,649
  > Total Versions Found:    115,070
  > Unique Parent DOIs:      103,649
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Preprints.org:
-----------------------------------
  1 version(s):     95,681 groups
  2 version(s):     6,404 groups
  3 version(s):     1,011 groups
  4 version(s):     276 groups
  5 version(s):     105 groups
  6 version(s):     53 groups
  7 version(s):     28 groups
  8 version(s):     16 groups
  9 version(s):     14 groups
  10 version(s):    8 groups
  11 version(s):    12 groups
  12 version(s):    9 groups
  13 version(s):    4 groups
  14 version(s):    3 groups
  15 version(s):    2 groups
  16 version(s):    1 groups
  17 version(s):    3 groups
  18 version(s):    2 groups
  20 version(s):    1 groups
  21 version(s):    2 groups
  22 version(s):    1 groups
  23 version(s):    5 groups
  26 version(s):    3 groups
  36 version(s

# PREPRINTS.RU

In [80]:
PREPRINTS_df, PREPRINTS_summary = get_server_data("PREPRINTS.RU")


 SERVER ANALYSIS: PREPRINTS.RU
  > Files found:    1
  > Raw records:    1415
  > Cleaned shape:  (1415, 90)
  > Unique DOIs:    1415
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.24108/preprints-    1415
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
preprints.ru    1415
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.24108    1415
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
10196    1415
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
NPG Publishing    1415
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    1415
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
institution_nam

In [81]:
PREPRINTS_parent_df, PREPRINTS_parent_summary = analyze_parent_data("PREPRINTS.RU", full_parent_df)


 CONSOLIDATED ANALYSIS: PREPRINTS.RU
  > Total Parent Groups:     1,346
  > Total Versions Found:    1,415
  > Unique Parent DOIs:      1,346
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for PREPRINTS.RU:
-----------------------------------
  1 version(s):     1,295 groups
  2 version(s):     42 groups
  3 version(s):     4 groups
  4 version(s):     2 groups
  5 version(s):     2 groups
  6 version(s):     1 groups

  Average versions per parent: 1.05

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
PREPRINTS.RU                  1334
PREPRINTS.RU; Zenodo             3
SSRN                             2
PREPRINTS.RU; ResearchGate       1
engrXiv                          1
AfricArXiv; PREPRINTS.RU         1
Advance                          1
Open Science Framework           1
Covid-19 Preprints               1
Cambridge Open Engage            1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN


# Prepublicaciones OpenCiencia

In [82]:
OpenCiencia_df, OpenCiencia_summary = get_server_data("Prepublicaciones_OpenCiencia")


 SERVER ANALYSIS: PREPUBLICACIONES_OPENCIENCIA
  > Files found:    1
  > Raw records:    8
  > Cleaned shape:  (8, 90)
  > Unique DOIs:    8
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.47073/preprints    8
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
prepublicaciones.org    8
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.47073    8
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
28319    8
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Centro de Investigacion sobre Desarrollo Humano y Sociedad (Coideso)    8
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    8
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_N

In [83]:
OpenCiencia_parent_df, OpenCiencia_parent_summary = analyze_parent_data("Prepublicaciones OpenCiencia", full_parent_df)


 CONSOLIDATED ANALYSIS: PREPUBLICACIONES OPENCIENCIA
  > Total Parent Groups:     7
  > Total Versions Found:    8
  > Unique Parent DOIs:      7
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Prepublicaciones OpenCiencia:
-----------------------------------
  1 version(s):     6 groups
  2 version(s):     1 groups

  Average versions per parent: 1.14

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Prepublicaciones OpenCiencia    7

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.47073/preprints    7



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
prepublicaciones.org    7


 COMPLETED: PREPUBLICACIONES OPENCIENCIA



# PropylaeumDok

In [8]:
PropylaeumDok_df, PropylaeumDok_summary = get_server_data("PropylaeumDok")


 SERVER ANALYSIS: PROPYLAEUMDOK
  > Files found:    2
  > Raw records:    6750
  > Cleaned shape:  (6750, 90)
  > Unique DOIs:    6750
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.11588/propylaeumdok    6750
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
archiv.ub.uni-heidelberg.de    6750
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.11588    6750
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
None    6750
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Heidelberg University Library    6749
None                                1
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    6750
Name: count, dtype: int64



TOP VALU

In [9]:
PropylaeumDok_parent_df, PropylaeumDok_parent_summary = analyze_parent_data("PropylaeumDok", full_parent_df)


 CONSOLIDATED ANALYSIS: PROPYLAEUMDOK
  > Total Parent Groups:     6,664
  > Total Versions Found:    6,748
  > Unique Parent DOIs:      6,664
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for PropylaeumDok:
-----------------------------------
  1 version(s):     6,608 groups
  2 version(s):     46 groups
  3 version(s):     5 groups
  4 version(s):     2 groups
  5 version(s):     1 groups
  6 version(s):     1 groups
  14 version(s):    1 groups

  Average versions per parent: 1.01

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
PropylaeumDok    6664

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.11588/propylaeumdok    6664



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
archiv.ub.uni-heidelberg.de    6664


 COMPLETED: PROPYLAEUMDOK



In [10]:
PropylaeumDok_df[PropylaeumDok_df['type_backend_raw']=='Film']

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
757,datacite::10.11588/propylaeumdok.00000379,PropylaeumDok,datacite,10.11588/propylaeumdok.00000379,10.11588/propylaeumdok.00000379,https://doi.org/10.11588/propylaeumdok.00000379,http://archiv.ub.uni-heidelberg.de/propylaeumd...,http://archiv.ub.uni-heidelberg.de/propylaeumd...,10.11588,None,gesis.ubhd,gesis,datacite,Heidelberg University Library,None,None,None,None,Experimente zur Keramikherstellung: Bau und Te...,None,None,None,None,Film,Video,Film,None,None,2017-02-23,None,None,None,2020-12-10,None,2017-02-23,None,None,2009.0,published_year,None,None,None,None,None,[],None,None,None,"Hampe, Roland; Winter, Adam",None,None,"[{""affiliation"": [], ""familyName"": ""Hampe"", ""g...",[],None,[],None,None,"[{""subject"": ""Plastic arts Sculpture""}]",None,None,0,0,None,None,0,None,[],,,False,None,,None,None,,None,None,None,client_id/doi_prefix_first_token,8,"{""client"": {""data"": {""id"": ""gesis.ubhd"", ""type...","{""citationCount"": 0, ""container"": {}, ""content...",10.11588/propylaeumdok.00000379,10.11588,10.11588,propylaeumdok.00000379,10.11588/propylaeumdok,10.11588/propylaeumdok,archiv.ub.uni-heidelberg.de,archiv.ub.uni-heidelberg.de/propylaeumdok
758,datacite::10.11588/propylaeumdok.00000380,PropylaeumDok,datacite,10.11588/propylaeumdok.00000380,10.11588/propylaeumdok.00000380,https://doi.org/10.11588/propylaeumdok.00000380,http://archiv.ub.uni-heidelberg.de/propylaeumd...,http://archiv.ub.uni-heidelberg.de/propylaeumd...,10.11588,None,gesis.ubhd,gesis,datacite,Heidelberg University Library,None,None,None,None,Experimente zur Keramikherstellung: Becher aus...,None,None,None,None,Film,Video,Film,None,None,2017-02-23,None,None,None,2020-12-10,None,2017-02-23,None,None,2009.0,published_year,None,None,None,None,None,[],None,None,None,"Hampe, Roland; Winter, Adam",None,None,"[{""affiliation"": [], ""familyName"": ""Hampe"", ""g...",[],None,[],None,None,"[{""subject"": ""Plastic arts Sculpture""}]",None,None,0,0,None,None,0,None,[],,,False,None,,None,None,,None,None,None,client_id/doi_prefix_first_token,8,"{""client"": {""data"": {""id"": ""gesis.ubhd"", ""type...","{""citationCount"": 0, ""container"": {}, ""content...",10.11588/propylaeumdok.00000380,10.11588,10.11588,propylaeumdok.00000380,10.11588/propylaeumdok,10.11588/propylaeumdok,archiv.ub.uni-heidelberg.de,archiv.ub.uni-heidelberg.de/propylaeumdok
763,datacite::10.11588/propylaeumdok.00000381,PropylaeumDok,datacite,10.11588/propylaeumdok.00000381,10.11588/propylaeumdok.00000381,https://doi.org/10.11588/propylaeumdok.00000381,http://archiv.ub.uni-heidelberg.de/propylaeumd...,http://archiv.ub.uni-heidelberg.de/propylaeumd...,10.11588,None,gesis.ubhd,gesis,datacite,Heidelberg University Lib

In [11]:
PropylaeumDok_df[PropylaeumDok_df['type_backend_raw']=='Collection']

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
523,datacite::10.11588/propylaeumdok.00000139,PropylaeumDok,datacite,10.11588/propylaeumdok.00000139,10.11588/propylaeumdok.00000139,https://doi.org/10.11588/propylaeumdok.00000139,http://archiv.ub.uni-heidelberg.de/propylaeumd...,http://archiv.ub.uni-heidelberg.de/propylaeumd...,10.11588,None,gesis.ubhd,gesis,datacite,Heidelberg University Library,None,None,None,None,OCCIDENT & ORIENT: Newsletter of the German Pr...,None,None,None,None,Collection,Periodical,Collection,None,None,2017-02-23,None,None,None,2020-12-10,None,2017-02-23,None,None,2008.0,published_year,None,None,None,None,None,[],None,None,None,Deutsches Evangelisches Institut Für Altertums...,None,None,"[{""affiliation"": [], ""name"": ""Deutsches Evange...",[],None,[],None,None,"[{""subject"": ""Palästina, Israel (Altertum)""}]",None,None,0,0,None,None,0,None,[],,,False,None,,None,None,,None,None,None,client_id/doi_prefix_first_token,8,"{""client"": {""data"": {""id"": ""gesis.ubhd"", ""type...","{""citationCount"": 0, ""container"": {}, ""content...",10.11588/propylaeumdok.00000139,10.11588,10.11588,propylaeumdok.00000139,10.11588/propylaeumdok,10.11588/propylaeumdok,archiv.ub.uni-heidelberg.de,archiv.ub.uni-heidelberg.de/propylaeumdok
524,datacite::10.11588/propylaeumdok.00000140,PropylaeumDok,datacite,10.11588/propylaeumdok.00000140,10.11588/propylaeumdok.00000140,https://doi.org/10.11588/propylaeumdok.00000140,http://archiv.ub.uni-heidelberg.de/propylaeumd...,http://archiv.ub.uni-heidelberg.de/propylaeumd...,10.11588,None,gesis.ubhd,gesis,datacite,Heidelberg University Library,None,None,None,None,OCCIDENT & ORIENT: Newsletter of the German Pr...,None,None,None,None,Collection,Periodical,Collection,None,None,2017-02-23,None,None,None,2020-12-10,None,2017-02-23,None,None,2008.0,published_year,None,None,None,None,None,[],None,None,None,Deutsches Evangelisches Institut Für Altertums...,None,None,"[{""affiliation"": [], ""name"": ""Deutsches Evange...",[],None,[],None,None,"[{""subject"": ""Palästina, Israel (Altertum)""}]",None,None,0,0,None,None,0,None,[],,,False,None,,None,None,,None,None,None,client_id/doi_prefix_first_token,8,"{""client"": {""data"": {""id"": ""gesis.ubhd"", ""type...","{""citationCount"": 0, ""container"": {}, ""content...",10.11588/propylaeumdok.00000140,10.11588,10.11588,propylaeumdok.00000140,10.11588/propylaeumdok,10.11588/propylaeumdok,archiv.ub.uni-heidelberg.de,archiv.ub.uni-heidelberg.de/propylaeumdok
532,datacite::10.11588/propylaeumdok.00000147,PropylaeumDok,datacite,10.11588/propylaeumdok.00000147,10.11588/propylaeumdok.00000147,https://doi.org/10.11588/propylaeumdok.00000147,http://archiv.ub.uni-heidelberg.de/propylaeumd...,http://archiv.ub.uni-heidelberg.d

# PsyArXiv

In [12]:
PsyArXiv_df, PsyArXiv_summary = get_server_data("PsyArXiv")


 SERVER ANALYSIS: PSYARXIV
  > Files found:    1
  > Raw records:    56866
  > Cleaned shape:  (56866, 90)
  > Unique DOIs:    56866
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31234/osf    54938
10.31219/osf      568
10.31227/osf      531
10.31235/osf      339
10.31228/osf      181
10.31223/osf       52
10.31225/osf       47
10.31230/osf       46
10.31229/osf       46
10.31224/osf       42
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    56866
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31234    54938
10.31219      568
10.31227      531
10.31235      339
10.31228      181
10.31223       52
10.31225       47
10.31230       46
10.31229       46
10.31224       42
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    5676

In [13]:
PsyArXiv_parent_df, PsyArXiv_parent_summary = analyze_parent_data("PsyArXiv", full_parent_df)


 CONSOLIDATED ANALYSIS: PSYARXIV
  > Total Parent Groups:     48,701
  > Total Versions Found:    59,523
  > Unique Parent DOIs:      48,701
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for PsyArXiv:
-----------------------------------
  1 version(s):     40,523 groups
  2 version(s):     6,500 groups
  3 version(s):     1,243 groups
  4 version(s):     255 groups
  5 version(s):     99 groups
  6 version(s):     36 groups
  7 version(s):     16 groups
  8 version(s):     7 groups
  9 version(s):     3 groups
  10 version(s):    4 groups
  12 version(s):    4 groups
  14 version(s):    1 groups
  15 version(s):    1 groups
  16 version(s):    1 groups
  17 version(s):    1 groups
  18 version(s):    2 groups
  21 version(s):    1 groups
  24 version(s):    1 groups
  25 version(s):    1 groups
  37 version(s):    1 groups
  52 version(s):    1 groups

  Average versions per parent: 1.22

 MOST RECENT DESTINATIONS (Migration):
------------

# Qeios

In [14]:
Qeios_df, Qeios_summary = get_server_data("Qeios")


 SERVER ANALYSIS: QEIOS
  > Files found:    1
  > Raw records:    5650
  > Cleaned shape:  (5650, 90)
  > Unique DOIs:    5650
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.32388/i    66
10.32388/g    54
10.32388/h    51
10.32388/a    49
10.32388/p    48
10.32388/j    48
10.32388/r    48
10.32388/n    48
10.32388/z    47
10.32388/o    46
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
qeios.com    5650
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.32388    5650
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
17262    5650
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Qeios Ltd    5650
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_

In [15]:
Qeios_parent_df, Qeios_parent_summary = analyze_parent_data("Qeios", full_parent_df)


 CONSOLIDATED ANALYSIS: QEIOS
  > Total Parent Groups:     3,256
  > Total Versions Found:    5,135
  > Unique Parent DOIs:      3,256
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Qeios:
-----------------------------------
  1 version(s):     2,084 groups
  2 version(s):     829 groups
  3 version(s):     210 groups
  4 version(s):     54 groups
  5 version(s):     37 groups
  6 version(s):     12 groups
  7 version(s):     10 groups
  8 version(s):     4 groups
  9 version(s):     4 groups
  11 version(s):    3 groups
  12 version(s):    3 groups
  13 version(s):    3 groups
  14 version(s):    1 groups
  15 version(s):    2 groups

  Average versions per parent: 1.58

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Qeios                     3190
SSRN                        19
arXiv                        9
ResearchGate                 7
Research Square              6
Qeios; Rese

# RePEc: Research Papers in Economics

In [16]:
# RePEc_df, RePEc_summary = get_server_data("RePEc_Research_Papers_in_Economics")

In [17]:
RePEc_parent_df, RePEc_parent_summary = analyze_parent_data("RePEc: Research Papers in Economics", full_parent_df)


 CONSOLIDATED ANALYSIS: REPEC: RESEARCH PAPERS IN ECONOMICS
  > Total Parent Groups:     369,997
  > Total Versions Found:    430,371
  > Unique Parent DOIs:      20,673
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for RePEc: Research Papers in Economics:
-----------------------------------
  1 version(s):     315,721 groups
  2 version(s):     49,392 groups
  3 version(s):     4,096 groups
  4 version(s):     611 groups
  5 version(s):     116 groups
  6 version(s):     25 groups
  7 version(s):     12 groups
  8 version(s):     6 groups
  9 version(s):     4 groups
  10 version(s):    4 groups
  11 version(s):    4 groups
  14 version(s):    1 groups
  18 version(s):    1 groups
  22 version(s):    1 groups
  27 version(s):    1 groups
  28 version(s):    1 groups
  43 version(s):    1 groups

  Average versions per parent: 1.16

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
RePEc

# Research Square

In [18]:
# Research_Square_df, Research_Square_summary = get_server_data("Research_Square")

In [19]:
Research_Square_parent_df, Research_Square_parent_summary = analyze_parent_data("Research Square", full_parent_df)


 CONSOLIDATED ANALYSIS: RESEARCH SQUARE
  > Total Parent Groups:     389,983
  > Total Versions Found:    439,160
  > Unique Parent DOIs:      389,983
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Research Square:
-----------------------------------
  1 version(s):     355,085 groups
  2 version(s):     25,294 groups
  3 version(s):     6,449 groups
  4 version(s):     2,253 groups
  5 version(s):     610 groups
  6 version(s):     174 groups
  7 version(s):     66 groups
  8 version(s):     24 groups
  9 version(s):     11 groups
  10 version(s):    5 groups
  11 version(s):    2 groups
  12 version(s):    2 groups
  13 version(s):    1 groups
  14 version(s):    1 groups
  17 version(s):    1 groups
  21 version(s):    1 groups
  24 version(s):    2 groups
  25 version(s):    1 groups
  47 version(s):    1 groups

  Average versions per parent: 1.13

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_rece

# ResearchGate

In [20]:
ResearchGate_df, ResearchGate_summary = get_server_data("ResearchGate")


 SERVER ANALYSIS: RESEARCHGATE
  > Files found:    1
  > Raw records:    181231
  > Cleaned shape:  (181231, 90)
  > Unique DOIs:    181231
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.13140/rg    181188
10.13140/2         43
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
researchgate.net    178343
rgdoi.net             2888
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.13140    181231
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
None    181231
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Unpublished                                                180824
Plekhanov Russian University of Economics                      21
Flora Montiberica.org                                         

In [21]:
ResearchGate_parent_df, ResearchGate_parent_summary = analyze_parent_data("ResearchGate", full_parent_df)


 CONSOLIDATED ANALYSIS: RESEARCHGATE
  > Total Parent Groups:     167,312
  > Total Versions Found:    180,737
  > Unique Parent DOIs:      167,312
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for ResearchGate:
-----------------------------------
  1 version(s):     157,458 groups
  2 version(s):     8,116 groups
  3 version(s):     1,231 groups
  4 version(s):     241 groups
  5 version(s):     97 groups
  6 version(s):     43 groups
  7 version(s):     44 groups
  8 version(s):     16 groups
  9 version(s):     13 groups
  10 version(s):    9 groups
  11 version(s):    7 groups
  12 version(s):    4 groups
  13 version(s):    5 groups
  14 version(s):    1 groups
  15 version(s):    2 groups
  17 version(s):    4 groups
  18 version(s):    2 groups
  19 version(s):    1 groups
  20 version(s):    1 groups
  21 version(s):    3 groups
  22 version(s):    1 groups
  23 version(s):    1 groups
  25 version(s):    1 groups
  27 version(s): 

# ResearchHub

In [22]:
ResearchHub_df, ResearchHub_summary = get_server_data("ResearchHub")


 SERVER ANALYSIS: RESEARCHHUB
  > Files found:    1
  > Raw records:    1636
  > Cleaned shape:  (1636, 90)
  > Unique DOIs:    1636
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.55277/researchhub    1554
10.55277/rhj              81
10.55277/pvtglk            1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
researchhub.com                1513
staging.researchhub.com         100
staging-web.researchhub.com      23
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.55277    1636
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
33940    1636
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
ResearchHub Technologies, Inc.    1636
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
-------

In [23]:
ResearchHub_parent_df, ResearchHub_parent_summary = analyze_parent_data("ResearchHub", full_parent_df)


 CONSOLIDATED ANALYSIS: RESEARCHHUB
  > Total Parent Groups:     1,375
  > Total Versions Found:    1,619
  > Unique Parent DOIs:      1,375
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for ResearchHub:
-----------------------------------
  1 version(s):     1,260 groups
  2 version(s):     71 groups
  3 version(s):     17 groups
  4 version(s):     8 groups
  5 version(s):     4 groups
  6 version(s):     7 groups
  7 version(s):     3 groups
  8 version(s):     3 groups
  10 version(s):    1 groups
  17 version(s):    1 groups

  Average versions per parent: 1.18

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
ResearchHub                  1356
ResearchHub; Zenodo             9
Zenodo                          4
Authorea Inc.                   2
ResearchGate; ResearchHub       2
SocArXiv                        1
EarthArXiv                      1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOK

# SAE Mobilus®

In [24]:
SAE_df, SAE_summary = get_server_data("SAE_Mobilus®")


 SERVER ANALYSIS: SAE_MOBILUS®
  > Files found:    1
  > Raw records:    105
  > Cleaned shape:  (105, 90)
  > Unique DOIs:    105
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.47953/sae-pp-    105
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
saemobilus.sae.org    105
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.47953    105
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
2796    105
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
SAE International    105
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    105
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
institution_name
No

In [25]:
SAE_parent_df, SAE_parent_summary = analyze_parent_data("SAE Mobilus®", full_parent_df)


 CONSOLIDATED ANALYSIS: SAE MOBILUS®
  > Total Parent Groups:     104
  > Total Versions Found:    108
  > Unique Parent DOIs:      104
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for SAE Mobilus®:
-----------------------------------
  1 version(s):     100 groups
  2 version(s):     4 groups

  Average versions per parent: 1.04

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
SAE Mobilus®    101
arXiv             2
SSRN              1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.47953/sae-pp-    104



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
saemobilus.sae.org    104


 COMPLETED: SAE MOBILUS®



# SciELO Preprints

In [26]:
SciELO_df, SciELO_summary = get_server_data("SciELO_Preprints")


 SERVER ANALYSIS: SCIELO_PREPRINTS
  > Files found:    1
  > Raw records:    4141
  > Cleaned shape:  (4141, 90)
  > Unique DOIs:    4141
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.1590/scielopreprints        3939
10.1590/16                       87
10.1590/s                        39
10.1590/22                       29
10.1590/25                       22
10.1590/01                        9
10.1590/23                        3
10.1590/19                        3
10.1590/26                        3
10.1590/scielopreprintstest       2
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
preprints.scielo.org            4136
homolog-preprints.scielo.org       4
preprints-bolha.scielo.org         1
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.1590    4141
Name: count, dtype: int64



TOP 

In [27]:
SciELO_parent_df, SciELO_parent_summary = analyze_parent_data("SciELO Preprints", full_parent_df)


 CONSOLIDATED ANALYSIS: SCIELO PREPRINTS
  > Total Parent Groups:     4,084
  > Total Versions Found:    4,142
  > Unique Parent DOIs:      4,084
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for SciELO Preprints:
-----------------------------------
  1 version(s):     4,038 groups
  2 version(s):     37 groups
  3 version(s):     7 groups
  4 version(s):     1 groups
  5 version(s):     1 groups

  Average versions per parent: 1.01

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
SciELO Preprints                  4066
ResearchGate; SciELO Preprints       3
Research Square                      3
Zenodo                               2
medRxiv                              2
SciELO Preprints; Zenodo             2
Qeios; SciELO Preprints              1
SSRN                                 1
PsyArXiv                             1
Preprints.org                        1

 TOP VALUES FOR: DOI_

# ScienceOpen Preprints

In [28]:
ScienceOpen_df, ScienceOpen_summary = get_server_data("ScienceOpen_Preprints")


 SERVER ANALYSIS: SCIENCEOPEN_PREPRINTS
  > Files found:    1
  > Raw records:    2970
  > Cleaned shape:  (2970, 90)
  > Unique DOIs:    2970
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.14293/s     1386
10.14293/pr    1068
10.14293/p      516
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
scienceopen.com    2970
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.14293    2970
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
5403    2970
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
ScienceOpen    2970
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    2970
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------

In [29]:
ScienceOpen_df[ScienceOpen_df['subtype_backend_raw']=='other']

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
1,crossref::10.14293/s2199-1006.1.sor-.ppzm4he.v1,ScienceOpen Preprints,crossref,10.14293/s2199-1006.1.sor-.ppzm4he.v1,10.14293/s2199-1006.1.sor-.ppzm4he.v1,https://doi.org/10.14293/s2199-1006.1.sor-.ppz...,https://scienceopen.com/document?vid=e8537577-...,https://scienceopen.com/document?vid=e8537577-...,10.14293,5403,None,None,crossref,ScienceOpen,None,ScienceOpen,None,None,Towards exploring adaptive and associatively r...,None,None,None,None,posted-content,other,other,None,True,2020-08-04,2019-01-10,2020-09-08,2025-05-14,None,2019-01-10,None,2019-01-10,None,2019,issued_date,posted_date,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<ns7:p>This poster presents an initial set of ...,This poster presents an initial set of observa...,"[{""URL"": ""https://scienceopen.com/document?vid...",None,"Olteteanu, Ana-Maria; Dyer, Jonathan",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-0639-7...",None,None,None,None,NaN,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,1,None,"{""DOI"": ""10.14293/s2199-1006.1.sor-.ppzm4he.v1...",10.14293/s2199-1006.1.sor-.ppzm4he.v1,10.14293,10.14293,s2199-1006.1.sor-.ppzm4he.v1,10.14293/s,10.14293/s,scienceopen.com,scienceopen.com/document
6,crossref::10.14293/s2199-1006.1.sor-.ppsvskm.v1,ScienceOpen Preprints,crossref,10.14293/s2199-1006.1.sor-.ppsvskm.v1,10.14293/s2199-1006.1.sor-.ppsvskm.v1,https://doi.org/10.14293/s2199-1006.1.sor-.pps...,https://scienceopen.com/hosted-document?doi=10...,https://scienceopen.com/hosted-document?doi=10...,10.14293,5403,None,None,crossref,ScienceOpen,None,ScienceOpen,None,None,Stealth Adapted Coronaviruses Resulting from t...,None,None,None,None,posted-content,other,other,None,True,2021-05-10,2021-05-08,2023-03-17,2023-03-17,None,2021-05-08,None,2021-05-08,None,2021,issued_date,posted_date,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<jats:p>The continuing emergence of variant fo...,The continuing emergence of variant forms of t...,"[{""URL"": ""https://scienceopen.com/hosted-docum...",None,"Martin, W John","Institute of Progressive Medicine, 1634 Spruce...",None,"[{""affiliation"": [{""name"": ""Institute of Progr...",None,None,None,None,NaN,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,1,None,"{""DOI"": ""10.14293/s2199-1006.1.sor-.ppsvskm.v1...",10.14293/s2199-1006.1.sor-.ppsvskm.v1,10.14293,10.14293,s2199-1006.1.sor-.ppsvskm.v1,10.14293/s,10.14293/s,scienceopen.com,scienceopen.com/hosted-document
72,crossref::10.14293/s2199-1006.1.sor-.pptzsif.v1,ScienceOpen

In [30]:
ScienceOpen_parent_df, ScienceOpen_parent_summary = analyze_parent_data("ScienceOpen Preprints", full_parent_df)


 CONSOLIDATED ANALYSIS: SCIENCEOPEN PREPRINTS
  > Total Parent Groups:     1,624
  > Total Versions Found:    2,027
  > Unique Parent DOIs:      1,624
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for ScienceOpen Preprints:
-----------------------------------
  1 version(s):     1,342 groups
  2 version(s):     207 groups
  3 version(s):     49 groups
  4 version(s):     16 groups
  5 version(s):     5 groups
  6 version(s):     2 groups
  7 version(s):     2 groups
  9 version(s):     1 groups

  Average versions per parent: 1.25

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
ScienceOpen Preprints                   1474
SSRN                                      33
Preprints.org                             17
Open Science Framework                    13
ResearchGate                              10
Authorea Inc.                             10
Research Square                           

# Sciencepaper Online

In [31]:
Sciencepaper_df, Sciencepaper_summary = get_server_data("Sciencepaper_Online")


 SERVER ANALYSIS: SCIENCEPAPER_ONLINE
  > Files found:    1
  > Raw records:    99
  > Cleaned shape:  (99, 90)
  > Unique DOIs:    99
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.61951/sciencepaperonline    99
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
paper.edu.cn    99
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.61951    99
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
48263    99
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Center for Scientific Research and Development in Higher Education Institutes, Ministry of Education    99
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    99
Name: count, dtype: int64

In [32]:
Sciencepaper_parent_df, Sciencepaper_parent_summary = analyze_parent_data("Sciencepaper Online", full_parent_df)


 CONSOLIDATED ANALYSIS: SCIENCEPAPER ONLINE
  > Total Parent Groups:     97
  > Total Versions Found:    97
  > Unique Parent DOIs:      97
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Sciencepaper Online:
-----------------------------------
  1 version(s):     97 groups

  Average versions per parent: 1.00

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Sciencepaper Online    97

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.61951/sciencepaperonline    97



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
paper.edu.cn    97


 COMPLETED: SCIENCEPAPER ONLINE



# searchRxiv

In [33]:
searchRxiv_df, searchRxiv_summary = get_server_data("searchRxiv")


 SERVER ANALYSIS: SEARCHRXIV
  > Files found:    2
  > Raw records:    1234
  > Cleaned shape:  (1234, 90)
  > Unique DOIs:    1234
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.1079/searchrxiv    1234
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
cabidigitallibrary.org    1234
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.1079    1234
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
242    1234
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
CABI Publishing    1234
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None          1232
searchRxiv       2
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
----------------

In [34]:
searchRxiv_parent_df, searchRxiv_parent_summary = analyze_parent_data("searchRxiv", full_parent_df)


 CONSOLIDATED ANALYSIS: SEARCHRXIV
  > Total Parent Groups:     529
  > Total Versions Found:    1,221
  > Unique Parent DOIs:      529
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for searchRxiv:
-----------------------------------
  1 version(s):     368 groups
  2 version(s):     28 groups
  3 version(s):     26 groups
  4 version(s):     27 groups
  5 version(s):     20 groups
  6 version(s):     18 groups
  7 version(s):     16 groups
  8 version(s):     9 groups
  9 version(s):     6 groups
  10 version(s):    2 groups
  11 version(s):    4 groups
  14 version(s):    2 groups
  15 version(s):    1 groups
  21 version(s):    1 groups
  37 version(s):    1 groups

  Average versions per parent: 2.31

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
searchRxiv         518
Research Square      6
JMIR Preprints       3
PsyArXiv             1
Zenodo               1

 TOP VALUES FOR: DO

# SocArXiv

In [35]:
SocArXiv_df, SocArXiv_summary = get_server_data("SocArXiv")


 SERVER ANALYSIS: SOCARXIV
  > Files found:    1
  > Raw records:    21541
  > Cleaned shape:  (21541, 90)
  > Unique DOIs:    21541
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31235/osf    19755
10.31219/osf      594
10.31234/osf      385
10.31227/osf      357
10.31228/osf      133
10.31223/osf      103
10.31224/osf       95
10.31231/osf       35
10.31229/osf       30
10.31225/osf       29
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    21541
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31235    19755
10.31219      594
10.31234      385
10.31227      357
10.31228      133
10.31223      103
10.31224       95
10.31231       35
10.31229       30
10.31225       29
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    2139

In [36]:
SocArXiv_parent_df, SocArXiv_parent_summary = analyze_parent_data("SocArXiv", full_parent_df)


 CONSOLIDATED ANALYSIS: SOCARXIV
  > Total Parent Groups:     18,026
  > Total Versions Found:    23,686
  > Unique Parent DOIs:      18,026
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for SocArXiv:
-----------------------------------
  1 version(s):     13,586 groups
  2 version(s):     3,439 groups
  3 version(s):     846 groups
  4 version(s):     117 groups
  5 version(s):     27 groups
  6 version(s):     4 groups
  7 version(s):     3 groups
  8 version(s):     2 groups
  10 version(s):    2 groups

  Average versions per parent: 1.31

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
SocArXiv                            14741
SSRN                                 1108
Open Science Framework; SocArXiv     1026
Open Science Framework                868
CrimRxiv                               43
arXiv                                  38
SocArXiv; arXiv                        26
Resear

# Social Science Open Access Repository

In [37]:
Social_Science_Open_Access_Repository_df, Social_Science_Open_Access_Repository_summary = get_server_data("Social_Science_Open_Access_Repository")


 SERVER ANALYSIS: SOCIAL_SCIENCE_OPEN_ACCESS_REPOSITORY
  > Files found:    1
  > Raw records:    27201
  > Cleaned shape:  (27201, 90)
  > Unique DOIs:    6679
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
<NA>                      20522
10.12759/hsr               1411
10.21241/ssoar              815
10.23668/psycharchives      437
10.14765/zzf                365
10.17169/fqs-               296
10.5281/zenodo              216
10.15464/isi                202
10.14764/10                 194
10.11588/iqas               189
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
ssoar.info                     26545
wbv.de                           267
publikationen.soziologie.de      135
doi.org                          111
hdl.handle.net                    31
journals.sub.uni-hamburg.de       21
uni-graz.at                       12
verbrauc

In [38]:
ssoar_parent_df, ssoar_parent_summary = analyze_parent_data("Social Science Open Access Repository", full_parent_df)


 CONSOLIDATED ANALYSIS: SOCIAL SCIENCE OPEN ACCESS REPOSITORY
  > Total Parent Groups:     26,905
  > Total Versions Found:    27,266
  > Unique Parent DOIs:      6,581
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Social Science Open Access Repository:
-----------------------------------
  1 version(s):     26,597 groups
  2 version(s):     274 groups
  3 version(s):     24 groups
  4 version(s):     2 groups
  5 version(s):     7 groups
  6 version(s):     1 groups

  Average versions per parent: 1.01

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Social Science Open Access Repository                                         26741
SSRN                                                                             80
RePEc: Research Papers in Economics                                              17
RePEc: Research Papers in Economics; Social Science Open Access Repository       13


In [39]:
Social_Science_Open_Access_Repository_df['doi_prefix_first_token'].value_counts()

doi_prefix_first_token
10.12759/hsr              1411
10.21241/ssoar             815
10.23668/psycharchives     437
10.14765/zzf               365
10.17169/fqs-              296
                          ... 
10.18449/2020s26             1
10.18449/2021c14             1
10.18449/2020c44             1
10.18449/2019s15             1
10.60922/226h                1
Name: count, Length: 722, dtype: int64

In [40]:
Social_Science_Open_Access_Repository_df[Social_Science_Open_Access_Repository_df['doi_prefix_first_token']=='10.60922/226h']

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
11749,openalex::W3136189946,Social Science Open Access Repository,openalex,https://openalex.org/W3136189946,10.60922/226h-hc50,https://doi.org/10.60922/226h-hc50,https://www.ssoar.info/ssoar/handle/document/7...,https://www.ssoar.info/ssoar/handle/document/7...,10.60922,None,None,None,openalex,None,None,None,None,None,Einsamkeit steigt in der Corona-Pandemie bei M...,None,None,None,de,article,None,article,False,False,2025-10-10T00:00:00,None,None,None,2025-11-06T06:51:31.235846,None,None,2021-02-01,None,2021,openalex.publication_date,None,True,green,None,None,"{""2014"": [51], ""2017."": [53], ""2020"": [2], ""46...",Seit März 2020 haben die Maßnahmen zur Eindämm...,None,https://www.ssoar.info/ssoar/handle/document/7...,Oliver Huxhold; Clemens Tesch‐Römer,None,None,"[{""affiliations"": [], ""author"": {""display_name...",None,None,[],None,NaN,None,"[{""display_name"": ""Gynecology"", ""id"": ""https:/...","[{""display_name"": ""COVID-19 and Mental Health""...",2,None,2,None,0,[],None,None,None,None,None,None,None,None,None,None,None,None,source_id,16,None,"{""abstract_inverted_index"": {""2014"": [51], ""20...",10.60922/226h-hc50,10.60922,10.60922,226h-hc50,10.60922/226h,10.60922/226h,ssoar.info,ssoar.info/ssoar


In [41]:
Social_Science_Open_Access_Repository_df[Social_Science_Open_Access_Repository_df['doi_prefix_first_token']=='10.23668/psycharchives']

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
162,openalex::W2247431608,Social Science Open Access Repository,openalex,https://openalex.org/W2247431608,10.23668/psycharchives.13780,https://doi.org/10.23668/psycharchives.13780,http://www.ssoar.info/ssoar/handle/document/2824,http://www.ssoar.info/ssoar/handle/document/2824,10.23668,None,None,None,openalex,None,None,None,None,None,Die Psychologie der Metapher und die (vermitte...,None,None,None,de,article,None,article,False,False,2025-10-10T00:00:00,None,None,None,2025-11-06T06:51:31.235846,None,None,2001-01-01,None,2001,openalex.publication_date,None,True,green,None,None,"{""\""hervorzubringen\"""": [46], ""Allerdings"": [2...","Es werden Überlegungen angestellt zur Frage, w...",None,http://www.ssoar.info/ssoar/handle/document/2824,Anil Jain,None,None,"[{""affiliations"": [], ""author"": {""display_name...",None,None,[],None,NaN,None,"[{""display_name"": ""Philosophy"", ""id"": ""https:/...","[{""display_name"": ""Language, Metaphor, and Cog...",31,None,31,None,0,[],None,None,None,None,None,None,None,None,None,None,None,None,source_id,16,None,"{""abstract_inverted_index"": {""\""hervorzubringe...",10.23668/psycharchives.13780,10.23668,10.23668,psycharchives.13780,10.23668/psycharchives,10.23668/psycharchives,ssoar.info,ssoar.info/ssoar
232,openalex::W965965652,Social Science Open Access Repository,openalex,https://openalex.org/W965965652,10.23668/psycharchives.13449,https://doi.org/10.23668/psycharchives.13449,http://www.ssoar.info/ssoar/handle/document/3377,http://www.ssoar.info/ssoar/handle/document/3377,10.23668,None,None,None,openalex,None,None,None,None,None,Buchbesprechung zu: Gerhard Kleining: Lehrbuch...,None,None,None,de,article,None,article,False,False,2025-10-10T00:00:00,None,None,None,2025-11-06T06:51:31.235846,None,None,2000-01-01,None,2000,openalex.publication_date,None,True,green,None,None,"{""None."": [0]}",None.,None,http://www.ssoar.info/ssoar/handle/document/3377,Christoph Klotter,None,None,"[{""affiliations"": [], ""author"": {""display_name...",None,None,[],None,NaN,None,"[{""display_name"": ""Philosophy"", ""id"": ""https:/...","[{""display_name"": ""Sociology and Education Stu...",81,None,81,None,0,[],None,None,None,None,None,None,None,None,None,None,None,None,source_id,16,None,"{""abstract_inverted_index"": {""None."": [0]}, ""a...",10.23668/psycharchives.13449,10.23668,10.23668,psycharchives.13449,10.23668/psycharchives,10.23668/psycharchives,ssoar.info,ssoar.info/ssoar
420,openalex::W1521498110,Social Science Open Access Repository,openalex,https://openalex.org/W1521498110,10.23668/psycharchives.8801,https://doi.org/10.23668/psycharchives.8801,http://www.ssoar.info/ssoar/handle/document/4977,http://www.ssoar.info/ssoar/han

# SportRxiv

In [42]:
SportRxiv_df, SportRxiv_summary = get_server_data("SportRxiv")


 SERVER ANALYSIS: SPORTRXIV
  > Files found:    2
  > Raw records:    878
  > Cleaned shape:  (878, 90)
  > Unique DOIs:    878
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.51224/srxiv    492
10.31236/osf      350
10.31227/osf       12
10.31234/osf        6
10.31231/osf        6
10.31219/osf        5
10.31223/osf        4
10.31235/osf        2
10.31229/osf        1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
sportrxiv.org    492
osf.io           386
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.51224    492
10.31236    350
10.31227     12
10.31234      6
10.31231      6
10.31219      5
10.31223      4
10.31235      2
10.31229      1
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
7569     492
15934    382
29705      4
Name: count

In [43]:
SportRxiv_parent_df, SportRxiv_parent_summary = analyze_parent_data("SportRxiv", full_parent_df)


 CONSOLIDATED ANALYSIS: SPORTRXIV
  > Total Parent Groups:     847
  > Total Versions Found:    898
  > Unique Parent DOIs:      847
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for SportRxiv:
-----------------------------------
  1 version(s):     800 groups
  2 version(s):     43 groups
  3 version(s):     4 groups

  Average versions per parent: 1.06

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
SportRxiv                            810
Open Science Framework; SportRxiv     28
JMIR Preprints                         2
Research Square                        2
arXiv                                  2
Open Science Framework                 1
ResearchGate; SportRxiv                1
SSRN                                   1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.51224/srxiv    480
10.31236/osf      366
10.31236/srxiv      

# SSRN

In [44]:
# SSRN_df, SSRN_summary = get_server_data("SSRN")

In [45]:
SSRN_parent_df, SSRN_parent_summary = analyze_parent_data("SSRN", full_parent_df)


 CONSOLIDATED ANALYSIS: SSRN
  > Total Parent Groups:     1,120,944
  > Total Versions Found:    1,235,560
  > Unique Parent DOIs:      1,120,944
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for SSRN:
-----------------------------------
  1 version(s):     1,022,779 groups
  2 version(s):     85,658 groups
  3 version(s):     10,164 groups
  4 version(s):     1,749 groups
  5 version(s):     349 groups
  6 version(s):     104 groups
  7 version(s):     53 groups
  8 version(s):     29 groups
  9 version(s):     15 groups
  10 version(s):    8 groups
  11 version(s):    3 groups
  12 version(s):    4 groups
  13 version(s):    7 groups
  14 version(s):    3 groups
  15 version(s):    1 groups
  16 version(s):    4 groups
  17 version(s):    1 groups
  19 version(s):    1 groups
  20 version(s):    1 groups
  21 version(s):    1 groups
  22 version(s):    2 groups
  24 version(s):    2 groups
  28 version(s):    1 groups
  30 version(s):   

# TechRxiv

In [46]:
TechRxiv_df, TechRxiv_summary = get_server_data("TechRxiv")


 SERVER ANALYSIS: TECHRXIV
  > Files found:    1
  > Raw records:    29418
  > Cleaned shape:  (29418, 90)
  > Unique DOIs:    29418
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.36227/techrxiv    29418
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
techrxiv.org             29088
techrxiv.figshare.com      324
essopenarchive.org           5
figshare.com                 1
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.36227    29418
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
263    29418
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Institute of Electrical and Electronics Engineers (IEEE)    29418
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------

In [47]:
TechRxiv_parent_df, TechRxiv_parent_summary = analyze_parent_data("TechRxiv", full_parent_df)


 CONSOLIDATED ANALYSIS: TECHRXIV
  > Total Parent Groups:     16,216
  > Total Versions Found:    27,311
  > Unique Parent DOIs:      16,216
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for TechRxiv:
-----------------------------------
  1 version(s):     7,677 groups
  2 version(s):     6,764 groups
  3 version(s):     1,291 groups
  4 version(s):     336 groups
  5 version(s):     79 groups
  6 version(s):     38 groups
  7 version(s):     9 groups
  8 version(s):     11 groups
  9 version(s):     4 groups
  10 version(s):    2 groups
  11 version(s):    2 groups
  12 version(s):    2 groups
  13 version(s):    1 groups

  Average versions per parent: 1.68

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
TechRxiv                  15748
arXiv                       209
SSRN                         93
Research Square              48
TechRxiv; arXiv              39
Authorea Inc.        

In [48]:
TechRxiv_parent_df[TechRxiv_parent_df['primary_domain']=='figshare.com']

,dup_group_id_full,parent_record_id,parent_server_name,parent_doi,parent_url,parent_title,parent_authors,parent_date_first_seen,parent_year_first_seen,version_record_ids,version_dois,servers_with_counts,total_versions,first_version_date,last_version_date,most_recent_version_servers,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend


https://doi.org/10.36227/techrxiv.12480425.v1 on techrxiv 

10.36227/techrxiv.12480425 on figshare


In [49]:
TechRxiv_parent_df[TechRxiv_parent_df['primary_domain']=='essopenarchive.org']

,dup_group_id_full,parent_record_id,parent_server_name,parent_doi,parent_url,parent_title,parent_authors,parent_date_first_seen,parent_year_first_seen,version_record_ids,version_dois,servers_with_counts,total_versions,first_version_date,last_version_date,most_recent_version_servers,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
2600119,crossref::10.36227/techrxiv.171207488.83541017/v1,crossref::10.36227/techrxiv.171207488.83541017/v1,TechRxiv,10.36227/techrxiv.171207488.83541017/v1,https://essopenarchive.org/users/556129/articl...,Effective media models for wave propagation in...,"Fu, Li-Yun; Yang, Haidi; Fu, Bo-Ye; Müller, To...",2024-04-02,2024,crossref::10.36227/techrxiv.171207488.83541017/v1,10.36227/techrxiv.171207488.83541017/v1,TechRxiv (1),1,2024-04-02,2024-04-02,TechRxiv,10.36227/techrxiv.171207488.83541017/v1,10.36227,10.36227,techrxiv.171207488.83541017/v1,10.36227/techrxiv,10.36227/techrxiv,essopenarchive.org,essopenarchive.org/users
4814340,crossref::10.36227/techrxiv.171043391.14447454/v1,crossref::10.36227/techrxiv.171043391.14447454/v1,TechRxiv,10.36227/techrxiv.171043391.14447454/v1,https://essopenarchive.org/users/524243/articl...,Geodetic anomaly detection and analysis in the...,"Giudicepietro, Flora; Casu, Francesco; Bonano,...",2024-03-14,2024,crossref::10.36227/techrxiv.171043391.14447454...,10.36227/techrxiv.171043391.14447454/v1; 10.36...,TechRxiv (2),2,2024-03-14,2024-04-08,TechRxiv,10.36227/techrxiv.171043391.14447454/v1,10.36227,10.36227,techrxiv.171043391.14447454/v1,10.36227/techrxiv,10.36227/techrxiv,essopenarchive.org,essopenarchive.org/users
5914615,crossref::10.36227/techrxiv.20128406.v1,crossref::10.36227/techrxiv.20128406.v1,TechRxiv,10.36227/techrxiv.20128406.v1,https://essopenarchive.org/users/535193/articl...,Automatic Detection of InSAR Surface Deformati...,"Staniewicz, Scott; Chen, Jingyi",2023-10-30,2023,crossref::10.36227/techrxiv.20128406.v1; cross...,10.36227/techrxiv.20128406.v1; 10.36227/techrx...,TechRxiv (2),2,2023-10-30,2024-03-18,TechRxiv,10.36227/techrxiv.20128406.v1,10.36227,10.36227,techrxiv.20128406.v1,10.36227/techrxiv,10.36227/techrxiv,essopenarchive.org,essopenarchive.org/users
6135961,crossref::10.36227/techrxiv.171470721.15780746/v1,crossref::10.36227/techrxiv.171470721.15780746/v1,TechRxiv,10.36227/techrxiv.171470721.15780746/v1,https://essopenarchive.org/users/523866/articl...,A Fast Three-dimensional Imaging Scheme of Air...,"Tang, Rongjiang; Gan, Lu; Li, Fusheng; Shen, F...",2024-05-03,2024,crossref::10.36227/techrxiv.171470721.15780746/v1,10.36227/techrxiv.171470721.15780746/v1,TechRxiv (1),1,2024-05-03,2024-05-03,TechRxiv,10.36227/techrxiv.171470721.15780746/v1,10.36227,10.36227,techrxiv.171470721.15780746/v1,10.36227/techrxiv,10.36227/techrxiv,essopenarchive.org,essopenarchive.org/users


all essopenarchive.org are on techrxiv

# Therapoid

In [50]:
Therapoid_df, Therapoid_summary = get_server_data("Therapoid")


 SERVER ANALYSIS: THERAPOID
  > Files found:    1
  > Raw records:    7
  > Cleaned shape:  (7, 90)
  > Unique DOIs:    7
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.24973/20    7
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
therapoid.net    7
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.24973    7
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
10597    7
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Therapoid            6
Open Therapeutics    1
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    7
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
institution_name
None    7

In [51]:
Therapoid_parent_df, Therapoid_parent_summary = analyze_parent_data("Therapoid", full_parent_df)


 CONSOLIDATED ANALYSIS: THERAPOID
  > Total Parent Groups:     6
  > Total Versions Found:    7
  > Unique Parent DOIs:      6
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Therapoid:
-----------------------------------
  1 version(s):     5 groups
  2 version(s):     1 groups

  Average versions per parent: 1.17

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Therapoid    6

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.24973/20    6



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
therapoid.net    6


 COMPLETED: THERAPOID



# Thesis Commons

In [52]:
Thesis_df, Thesis_summary = get_server_data("Thesis_Commons")


 SERVER ANALYSIS: THESIS_COMMONS
  > Files found:    1
  > Raw records:    3959
  > Cleaned shape:  (3959, 90)
  > Unique DOIs:    3959
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31237/osf    3663
10.31227/osf     103
10.31231/osf      45
10.31219/osf      38
10.31228/osf      30
10.31234/osf      18
10.31230/osf      18
10.31235/osf      14
10.31223/osf      10
10.31220/osf      10
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    3959
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31237    3663
10.31227     103
10.31231      45
10.31219      38
10.31228      30
10.31234      18
10.31230      18
10.31235      14
10.31223      10
10.31220      10
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    3939
242        10
297

In [53]:
Thesis_parent_df, Thesis_parent_summary = analyze_parent_data("Thesis Commons", full_parent_df)


 CONSOLIDATED ANALYSIS: THESIS COMMONS
  > Total Parent Groups:     3,220
  > Total Versions Found:    4,327
  > Unique Parent DOIs:      3,220
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Thesis Commons:
-----------------------------------
  1 version(s):     2,734 groups
  2 version(s):     393 groups
  3 version(s):     57 groups
  4 version(s):     14 groups
  5 version(s):     5 groups
  6 version(s):     1 groups
  7 version(s):     3 groups
  8 version(s):     2 groups
  9 version(s):     1 groups
  11 version(s):    1 groups
  13 version(s):    1 groups
  14 version(s):    1 groups
  16 version(s):    2 groups
  29 version(s):    1 groups
  46 version(s):    1 groups
  100 version(s):   1 groups
  108 version(s):   1 groups
  150 version(s):   1 groups

  Average versions per parent: 1.34

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Thesis Commons                      

# UCL Open Environment

In [54]:
UCL_df, UCL_summary = get_server_data("UCL_Open_Environment")


 SERVER ANALYSIS: UCL_OPEN_ENVIRONMENT
  > Files found:    2
  > Raw records:    369
  > Cleaned shape:  (369, 90)
  > Unique DOIs:    369
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.14324/11                317
10.14324/ucloepreprints     51
10.14324/ucloe               1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
journals.uclpress.co.uk    364
ucl.scienceopen.com          5
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.14324    369
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
5433    369
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
UCL Press    369
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None               

/tmp/ipykernel_3327/1196196344.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  raw_df = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)


In [55]:
UCL_parent_df, UCL_parent_summary = analyze_parent_data("UCL Open Environment", full_parent_df)


 CONSOLIDATED ANALYSIS: UCL OPEN ENVIRONMENT
  > Total Parent Groups:     173
  > Total Versions Found:    366
  > Unique Parent DOIs:      173
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for UCL Open Environment:
-----------------------------------
  1 version(s):     85 groups
  2 version(s):     15 groups
  3 version(s):     48 groups
  4 version(s):     22 groups
  5 version(s):     2 groups
  9 version(s):     1 groups

  Average versions per parent: 2.12

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
UCL Open Environment    173

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.14324/11                150
10.14324/ucloepreprints     23



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
journals.uclpress.co.uk    171
ucl.scienceopen.com          2


 COMPLETED: UCL OPEN ENVIRONMENT



# UnisaRxiv

In [56]:
UnisaRxiv_df, UnisaRxiv_summary = get_server_data("UnisaRxiv")


 SERVER ANALYSIS: UNISARXIV
  > Files found:    1
  > Raw records:    126
  > Cleaned shape:  (126, 90)
  > Unique DOIs:    126
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.25159/unisarxiv    126
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
scienceopen.com    126
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.25159    126
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
10792    126
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
UNISA Press    126
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    126
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
institution_name
ScienceOpen

In [57]:
UnisaRxiv_parent_df, UnisaRxiv_parent_summary = analyze_parent_data("UnisaRxiv", full_parent_df)


 CONSOLIDATED ANALYSIS: UNISARXIV
  > Total Parent Groups:     120
  > Total Versions Found:    125
  > Unique Parent DOIs:      120
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for UnisaRxiv:
-----------------------------------
  1 version(s):     116 groups
  2 version(s):     3 groups
  3 version(s):     1 groups

  Average versions per parent: 1.04

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
UnisaRxiv       117
SSRN              1
ResearchGate      1
AfricArXiv        1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.25159/unisarxiv    120



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
scienceopen.com    120


 COMPLETED: UNISARXIV



# VeriXiv

In [58]:
VeriXiv_df, VeriXiv_summary = get_server_data("VeriXiv")


 SERVER ANALYSIS: VERIXIV
  > Files found:    1
  > Raw records:    504
  > Cleaned shape:  (504, 90)
  > Unique DOIs:    504
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.12688/verixiv    504
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
verixiv.org    504
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.12688    504
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
2560    504
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
F1000 Research Ltd    504
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    504
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
institution_name
None    504
N

In [59]:
VeriXiv_parent_df, VeriXiv_parent_summary = analyze_parent_data("VeriXiv", full_parent_df)


 CONSOLIDATED ANALYSIS: VERIXIV
  > Total Parent Groups:     425
  > Total Versions Found:    507
  > Unique Parent DOIs:      425
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for VeriXiv:
-----------------------------------
  1 version(s):     368 groups
  2 version(s):     38 groups
  3 version(s):     13 groups
  4 version(s):     6 groups

  Average versions per parent: 1.19

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
VeriXiv                399
Gates Open Research     24
AgEcon Search            1
Research Square          1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.12688/verixiv    425



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
verixiv.org    425


 COMPLETED: VERIXIV



In [60]:
VeriXiv_parent_df[VeriXiv_parent_df['most_recent_version_servers']=='Research Square']

,dup_group_id_full,parent_record_id,parent_server_name,parent_doi,parent_url,parent_title,parent_authors,parent_date_first_seen,parent_year_first_seen,version_record_ids,version_dois,servers_with_counts,total_versions,first_version_date,last_version_date,most_recent_version_servers,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
7125739,crossref::10.12688/verixiv.2010.1,crossref::10.12688/verixiv.2010.1,VeriXiv,10.12688/verixiv.2010.1,https://verixiv.org/articles/2-286/v1,Maternal screening coverage and determinants d...,"Adelabu, Yusuf; Saalu, Tersur T.; Afolabi, Bos...",2025-09-15,2025,crossref::10.12688/verixiv.2010.1; crossref::1...,10.12688/verixiv.2010.1; 10.21203/rs.3.rs-7490...,Research Square (1); VeriXiv (1),2,2025-09-15,2025-09-17,Research Square,10.12688/verixiv.2010.1,10.12688,10.12688,verixiv.2010.1,10.12688/verixiv,10.12688/verixiv,verixiv.org,verixiv.org/articles


In [61]:
VeriXiv_df

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
0,crossref::10.12688/verixiv.83.1,VeriXiv,crossref,10.12688/verixiv.83.1,10.12688/verixiv.83.1,https://doi.org/10.12688/verixiv.83.1,https://verixiv.org/articles/1-14/v1,https://verixiv.org/articles/1-14/v1,10.12688,2560,None,None,crossref,F1000 Research Ltd,None,None,Gates Foundation,None,Developing a male-specific counselling curricu...,None,None,None,None,posted-content,preprint,preprint,None,True,2024-10-10,2024-10-10,2024-10-11,2025-11-21,None,2024-10-10,None,2024-10-10,None,2024,issued_date,posted_date,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,None,None,None,None,"Mphande, Misheck; Robson, Isabella; Hubbard, J...",None,None,"[{""affiliation"": [], ""family"": ""Mphande"", ""giv...",None,None,"[{""DOI"": ""10.13039/100000061"", ""award"": [""K01-...",Fogarty International Center; Bill and Melinda...,3.0,None,None,None,0,None,None,0,63,"[{""DOI"": ""10.1080/17441690902942464"", ""article...",None,,,False,None,,None,None,,None,None,None,prefix/primary_domain,40,None,"{""DOI"": ""10.12688/verixiv.83.1"", ""URL"": ""https...",10.12688/verixiv.83.1,10.12688,10.12688,verixiv.83.1,10.12688/verixiv,10.12688/verixiv,verixiv.org,verixiv.org/articles
1,crossref::10.12688/verixiv.197.1,VeriXiv,crossref,10.12688/verixiv.197.1,10.12688/verixiv.197.1,https://doi.org/10.12688/verixiv.197.1,https://verixiv.org/articles/1-15/v1,https://verixiv.org/articles/1-15/v1,10.12688,2560,None,None,crossref,F1000 Research Ltd,None,None,Gates Foundation,None,Discovery of a picomolar antiplasmodial pyrazo...,None,None,None,None,posted-content,preprint,preprint,None,True,2024-10-16,2024-10-16,2024-10-18,2025-07-23,None,2024-10-16,None,2024-10-16,None,2024,issued_date,posted_date,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,None,None,None,None,"Tchatat Tali, Mariscal Brice; Dize, Darline; Y...",None,None,"[{""affiliation"": [], ""family"": ""Tchatat Tali"",...",None,None,"[{""DOI"": ""10.13039/100000865"", ""award"": [""INV-...",Bill and Melinda Gates Foundation,1.0,None,None,None,1,None,None,1,65,"[{""DOI"": ""10.1021/jm990002y"", ""article-title"":...",None,,,False,None,,None,None,,None,None,None,prefix/primary_domain,40,None,"{""DOI"": ""10.12688/verixiv.197.1"", ""URL"": ""http...",10.12688/verixiv.197.1,10.12688,10.12688,verixiv.197.1,10.12688/verixiv,10.12688/verixiv,verixiv.org,verixiv.org/articles
2,crossref::10.12688/verixiv.77.1,VeriXiv,crossref,10.12688/verixiv.77.1,10.12688/verixiv.77.1,https://doi.org/10.12688/verixiv.77.1,https://verixiv.org/articles/1-9/v1,https://verixiv.org/articles/1-9/v1,10.12688,2560,None,None,crossref,F1000 Research Ltd,None,None,Gates

In [62]:
VeriXiv_df['funders_flat'].value_counts()

funders_flat
Bill and Melinda Gates Foundation                                                                                                                                                                                                                                                          261
Gates Foundation                                                                                                                                                                                                                                                                            93
Bill and Melinda Gates Foundation; African Economic Research Consortium                                                                                                                                                                                                                      4
Bill and Melinda Gates Foundation; Children’s Investment Fund Foundation                                                      

In [63]:
# Optional: Group similar names together
df_funders = VeriXiv_df['funders_flat'].dropna().str.split(';').explode().str.strip()

# # Normalize common variants
# df_funders = df_funders.replace({
#     "Gates Foundation": "Bill and Melinda Gates Foundation"
# })

final_counts = df_funders.value_counts().to_frame()
final_counts.head(10)

,count
funders_flat,
Bill and Melinda Gates Foundation,366
Gates Foundation,120
Wellcome Trust,9
Biotechnology and Biological Sciences Research Council,8
Children's Investment Fund Foundation,7
National Institute of Allergy and Infectious Diseases,7
European and Developing Countries Clinical Trials Partnership,6
Consortium pour la recherche économique en Afrique,6
Wellcome,6


# viXra

In [64]:
viXra_df, viXra_summary = get_server_data("viXra")


 SERVER ANALYSIS: VIXRA
  > Files found:    1
  > Raw records:    25570
  > Cleaned shape:  (25570, 90)
  > Unique DOIs:    2646
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
<NA>              22924
10.5281/zenodo      989
10.13140/rg         989
10.6084/m           225
10.48550/arxiv      136
10.13140/2           36
10.1016/j            23
10.17605/osf         13
10.18147/smn         11
10.1007/s            10
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
vixra.org                     25200
rxiv.org                         75
fs.gallup.unm.edu                67
philpapers.org                   25
gallup.unm.edu                   23
s3-eu-west-1.amazonaws.com       10
deepblue.lib.umich.edu           10
zenodo.org                       10
redshift.vif.com                  9
researchgate.net                  8
Name: count, dtype:

In [65]:
viXra_parent_df, viXra_parent_summary = analyze_parent_data("viXra", full_parent_df)


 CONSOLIDATED ANALYSIS: VIXRA
  > Total Parent Groups:     22,570
  > Total Versions Found:    23,011
  > Unique Parent DOIs:      0
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for viXra:
-----------------------------------
  1 version(s):     22,228 groups
  2 version(s):     287 groups
  3 version(s):     37 groups
  4 version(s):     8 groups
  5 version(s):     3 groups
  6 version(s):     3 groups
  7 version(s):     3 groups
  12 version(s):    1 groups

  Average versions per parent: 1.02

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
viXra                     22382
arXiv                        85
ResearchGate                 57
SSRN                         18
Zenodo                        8
HAL                           4
Open Science Framework        4
Preprints.org                 3
bioRxiv                       3
engrXiv                       1

 TOP VALUES FOR: DOI_PREF

In [66]:
viXra_df[viXra_df['doi_prefix_first_token']=='10.5281/zenodo']

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
1,openalex::W2868542249,viXra,openalex,https://openalex.org/W2868542249,10.5281/zenodo.3843259,https://doi.org/10.5281/zenodo.3843259,https://vixra.org/pdf/1807.0120v1.pdf,https://vixra.org/pdf/1807.0120v1.pdf,10.5281,None,None,None,openalex,None,None,None,None,None,"Putrescine, Cadaverine, Spermine and Spermidin...",None,None,None,en,preprint,None,preprint,False,True,2025-10-10T00:00:00,None,None,None,2025-11-06T06:51:31.235846,None,None,2018-07-01,None,2018,openalex.publication_date,None,True,green,None,None,"{""(EPPSI)"": [16], ""(NPM)"": [25], ""(NPME)"": [34...","In the current study, we study Putrescine, Cad...",None,None,Alireza Heidari; Ricardo Gobato,None,None,"[{""affiliations"": [], ""author"": {""display_name...",None,None,[],None,None,None,"[{""display_name"": ""Cadaverine"", ""id"": ""https:/...","[{""display_name"": ""Advanced biosensing and bio...",96,None,96,None,65,"[""https://openalex.org/W2919873033"", ""https://...",None,None,None,None,None,None,None,None,None,None,None,None,source_id,10,None,"{""abstract_inverted_index"": {""(EPPSI)"": [16], ...",10.5281/zenodo.3843259,10.5281,10.5281,zenodo.3843259,10.5281/zenodo,10.5281/zenodo,vixra.org,vixra.org/pdf
6,openalex::W3003001456,viXra,openalex,https://openalex.org/W3003001456,10.5281/zenodo.3871965,https://doi.org/10.5281/zenodo.3871965,https://vixra.org/pdf/2001.0191v1.pdf,https://vixra.org/pdf/2001.0191v1.pdf,10.5281,None,None,None,openalex,None,None,None,None,None,Pros and Cons of Livermorium Nanoparticles for...,None,None,None,en,preprint,None,preprint,False,True,2025-10-10T00:00:00,None,None,None,2025-11-06T06:51:31.235846,None,None,2020-01-11,None,2020,openalex.publication_date,None,True,green,None,None,"{""(emission"": [13], ""(non–emission"": [20], ""3D...",When Livermorium nanoparticles are subjected t...,None,None,Alireza Heidari; Katrina Schmitt; Maria Hender...,None,None,"[{""affiliations"": [], ""author"": {""display_name...",None,None,[],None,None,None,"[{""display_name"": ""Nanoparticle"", ""id"": ""https...","[{""display_name"": ""Spectroscopy Techniques in ...",31,None,31,None,0,[],None,None,None,None,None,None,None,None,None,None,None,None,source_id,10,None,"{""abstract_inverted_index"": {""(emission"": [13]...",10.5281/zenodo.3871965,10.5281,10.5281,zenodo.3871965,10.5281/zenodo,10.5281/zenodo,vixra.org,vixra.org/pdf
7,openalex::W2127088805,viXra,openalex,https://openalex.org/W2127088805,10.5281/zenodo.23153,https://doi.org/10.5281/zenodo.23153,https://vixra.org/abs/1411.0488,https://vixra.org/abs/1411.0488,10.5281,None,None,None,openalex,None,None,None,None,None,Neutrosophic Soft Relations And Some Properties,None,None,None,en,preprint,None,preprint,False,True,202

In [67]:
viXra_df[viXra_df['doi_prefix_first_token']=='10.13140/rg']

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
3,openalex::W2842105084,viXra,openalex,https://openalex.org/W2842105084,10.13140/rg.2.2.22315.75047,https://doi.org/10.13140/rg.2.2.22315.75047,https://vixra.org/pdf/1807.0123v1.pdf,https://vixra.org/pdf/1807.0123v1.pdf,10.13140,None,None,None,openalex,None,None,None,None,None,Using the Quantum Chemistry for Genesis of a N...,None,None,None,en,preprint,None,preprint,False,True,2025-10-10T00:00:00,None,None,None,2025-11-06T06:51:31.235846,None,None,2018-01-01,None,2018,openalex.publication_date,None,True,gold,None,None,"{""(3df,"": [238], ""(Hartree-Fock)"": [108], ""3pd...",Preliminary bibliographic studies did not reve...,None,None,Ricardo Gobato; Alireza Heidari; Abhijit Mitra,None,None,"[{""affiliations"": [], ""author"": {""display_name...",None,None,[],None,None,None,"[{""display_name"": ""Basis set"", ""id"": ""https://...","[{""display_name"": ""History and advancements in...",70,None,70,None,0,[],None,None,None,None,None,None,None,None,None,None,None,None,source_id,10,None,"{""abstract_inverted_index"": {""(3df,"": [238], ""...",10.13140/rg.2.2.22315.75047,10.13140,10.13140,rg.2.2.22315.75047,10.13140/rg,10.13140/rg,vixra.org,vixra.org/pdf
5,openalex::W3093092331,viXra,openalex,https://openalex.org/W3093092331,10.13140/rg.2.2.27971.84008,https://doi.org/10.13140/rg.2.2.27971.84008,https://vixra.org/pdf/2010.0063v1.pdf,https://vixra.org/pdf/2010.0063v1.pdf,10.13140,None,None,None,openalex,None,None,None,None,None,Vortex Cotes's spiral in an extratropical cycl...,None,None,None,en,preprint,None,preprint,False,True,2025-10-10T00:00:00,None,None,None,2025-11-06T06:51:31.235846,None,None,2020-01-01,None,2020,openalex.publication_date,None,True,gold,None,None,"{""(159.99"": [150], ""(31.998"": [159], ""-30◦C,"":...",Ae extratropical cyclone” is an at- mospheric ...,None,None,Ricardo Gobato; Alireza Heidari; Abhijit Mitra...,None,None,"[{""affiliations"": [], ""author"": {""display_name...",None,None,[],None,None,None,"[{""display_name"": ""Extratropical cyclone"", ""id...","[{""display_name"": ""Environmental and biologica...",36,None,36,None,0,[],None,None,None,None,None,None,None,None,None,None,None,None,source_id,10,None,"{""abstract_inverted_index"": {""(159.99"": [150],...",10.13140/rg.2.2.27971.84008,10.13140,10.13140,rg.2.2.27971.84008,10.13140/rg,10.13140/rg,vixra.org,vixra.org/pdf
21,openalex::W2900877152,viXra,openalex,https://openalex.org/W2900877152,10.13140/rg.2.2.22927.94885,https://doi.org/10.13140/rg.2.2.22927.94885,https://vixra.org/pdf/1811.0283v1.pdf,https://vixra.org/pdf/1811.0283v1.pdf,10.13140,None,None,None,openalex,None,None,None,None,None,To divide by zero is to multiply by zero,None,None,None,en,preprint,None,preprint,False,

In [68]:
viXra_df[viXra_df['doi_prefix_first_token']=='10.6084/m']

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
13,openalex::W1508236978,viXra,openalex,https://openalex.org/W1508236978,10.6084/m9.figshare.6205535,https://doi.org/10.6084/m9.figshare.6205535,https://vixra.org/pdf/1508.0285v1.pdf,https://vixra.org/pdf/1508.0285v1.pdf,10.6084,None,None,None,openalex,None,None,None,None,None,"Neutrosophic Sets and Systems, 20/2018",None,None,None,en,preprint,None,preprint,False,True,2025-10-10T00:00:00,None,None,None,2025-11-06T06:51:31.235846,None,None,2018-01-01,None,2018,openalex.publication_date,None,True,green,None,None,"{""1995"": [25], ""Sets"": [1], ""Systems”"": [3], ""...",“Neutrosophic Sets and Systems” has been creat...,None,None,Florentín Smarandache; Mumtaz Ali,None,None,"[{""affiliations"": [], ""author"": {""display_name...",None,None,[],None,None,None,"[{""display_name"": ""Indeterminacy (philosophy)""...","[{""display_name"": ""Advanced Mathematical Theor...",25,None,25,None,0,[],None,None,None,None,None,None,None,None,None,None,None,None,source_id,10,None,"{""abstract_inverted_index"": {""1995"": [25], ""Se...",10.6084/m9.figshare.6205535,10.6084,10.6084,m9.figshare.6205535,10.6084/m,10.6084/m,vixra.org,vixra.org/pdf
71,openalex::W2801841746,viXra,openalex,https://openalex.org/W2801841746,10.6084/m9.figshare.6205448,https://doi.org/10.6084/m9.figshare.6205448,https://vixra.org/pdf/1804.0434v1.pdf,https://vixra.org/pdf/1804.0434v1.pdf,10.6084,None,None,None,openalex,None,None,None,None,None,NC-VIKOR Based MAGDM Strategy under Neutrosoph...,None,None,None,en,preprint,None,preprint,False,True,2025-10-10T00:00:00,None,None,None,2025-11-06T06:51:31.235846,None,None,2018-01-01,None,2018,openalex.publication_date,None,True,green,None,None,"{""Neutrosophic"": [0], ""and"": [8], ""consists"": ...",Neutrosophic cubic set consists of interval ne...,None,None,Surapati Pramanik; Shyamal Dalapati; Shariful ...,Indian Institute of Engineering Science and Te...,IN,"[{""affiliations"": [{""institution_ids"": [], ""ra...",None,None,[],None,None,None,"[{""display_name"": ""Set (abstract data type)"", ...","[{""display_name"": ""Multi-Criteria Decision Mak...",7,None,7,None,0,[],None,None,None,None,None,None,None,None,None,None,None,None,source_id,10,None,"{""abstract_inverted_index"": {""Neutrosophic"": [...",10.6084/m9.figshare.6205448,10.6084,10.6084,m9.figshare.6205448,10.6084/m,10.6084/m,vixra.org,vixra.org/pdf
89,openalex::W2953040890,viXra,openalex,https://openalex.org/W2953040890,10.6084/m9.figshare.5028353,https://doi.org/10.6084/m9.figshare.5028353,https://vixra.org/pdf/1609.0062v1.pdf,https://vixra.org/pdf/1609.0062v1.pdf,10.6084,None,None,None,openalex,None,None,None,None,None,Optimization of Supercritical Fluid Consecutiv...,None,None,None,en,preprint,Non

In [69]:
viXra_df[viXra_df['doi_prefix_first_token']=='10.48550/arxiv']

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
20,openalex::W2951522206,viXra,openalex,https://openalex.org/W2951522206,10.48550/arxiv.1511.03095,https://doi.org/10.48550/arxiv.1511.03095,https://vixra.org/pdf/1511.0232v1.pdf,https://vixra.org/pdf/1511.0232v1.pdf,10.48550,None,None,None,openalex,None,None,None,None,None,Generalized Multiple Importance Sampling,None,None,None,en,preprint,None,preprint,False,True,2025-10-10T00:00:00,None,None,None,2025-11-06T06:51:31.235846,None,None,2015-11-10,None,2015,openalex.publication_date,None,True,green,None,None,"{""(MIS)"": [61], ""All"": [126], ""Finally,"": [143...",Importance Sampling methods are broadly used t...,None,None,Vı́ctor Elvira; Luca Martino; David Luengo; Mó...,None,None,"[{""affiliations"": [], ""author"": {""display_name...",None,None,[],None,None,None,"[{""display_name"": ""Weighting"", ""id"": ""https://...","[{""display_name"": ""Advanced Statistical Method...",17,None,17,None,21,"[""https://openalex.org/W3098203061"", ""https://...",None,None,None,None,None,None,None,None,None,None,None,None,source_id,10,None,"{""abstract_inverted_index"": {""(MIS)"": [61], ""A...",10.48550/arxiv.1511.03095,10.48550,10.48550,arxiv.1511.03095,10.48550/arxiv,10.48550/arxiv,vixra.org,vixra.org/pdf
29,openalex::W2951328659,viXra,openalex,https://openalex.org/W2951328659,10.48550/arxiv.1609.07731,https://doi.org/10.48550/arxiv.1609.07731,https://vixra.org/pdf/1512.0420v3.pdf,https://vixra.org/pdf/1512.0420v3.pdf,10.48550,None,None,None,openalex,None,None,None,None,None,Cooperative Parallel Particle Filters for onli...,None,None,None,en,preprint,None,preprint,False,True,2025-10-10T00:00:00,None,None,None,2025-11-06T06:51:31.235846,None,None,2016-09-25,None,2016,openalex.publication_date,None,True,green,None,None,"{""Bayesian"": [12], ""Carlo"": [5], ""For"": [52], ...",We design a sequential Monte Carlo scheme for ...,None,None,Luca Martino; Jesse Read; Vı́ctor Elvira; Fran...,Universidade de São Paulo; Aalto University; C...,BR; FI; FR; ES,"[{""affiliations"": [{""institution_ids"": [""https...",None,None,[],None,None,None,"[{""display_name"": ""Particle filter"", ""id"": ""ht...","[{""display_name"": ""Target Tracking and Data Fu...",9,None,9,None,35,"[""https://openalex.org/W2160337655"", ""https://...",None,None,None,None,None,None,None,None,None,None,None,None,source_id,10,None,"{""abstract_inverted_index"": {""Bayesian"": [12],...",10.48550/arxiv.1609.07731,10.48550,10.48550,arxiv.1609.07731,10.48550/arxiv,10.48550/arxiv,vixra.org,vixra.org/pdf
141,openalex::W2950625612,viXra,openalex,https://openalex.org/W2950625612,10.48550/arxiv.math/0404520,https://doi.org/10.48550/arxiv.math/0404520,https://vixra.org/pdf/1109.0041v1.pdf,https://vixra.org/pdf/1109.

# Wellcome Open Research

In [70]:
Wellcome_df, Wellcome_summary = get_server_data("Wellcome_Open_Research")


 SERVER ANALYSIS: WELLCOME_OPEN_RESEARCH
  > Files found:    1
  > Raw records:    4727
  > Cleaned shape:  (4727, 90)
  > Unique DOIs:    4727
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.12688/wellcomeopenres    4727
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
wellcomeopenresearch.org    4727
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.12688    4727
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
2560    4727
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
F1000 Research Ltd    4727
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
Wellcome Open Research    4727
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAM

In [71]:
Wellcome_parent_df, Wellcome_parent_summary = analyze_parent_data("Wellcome Open Research", full_parent_df)


 CONSOLIDATED ANALYSIS: WELLCOME OPEN RESEARCH
  > Total Parent Groups:     3,276
  > Total Versions Found:    4,480
  > Unique Parent DOIs:      3,276
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Wellcome Open Research:
-----------------------------------
  1 version(s):     2,264 groups
  2 version(s):     858 groups
  3 version(s):     122 groups
  4 version(s):     27 groups
  5 version(s):     4 groups
  6 version(s):     1 groups

  Average versions per parent: 1.37

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Wellcome Open Research    3274
Authorea Inc.                1
medRxiv                      1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.12688/wellcomeopenres    3276



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
wellcomeopenresearch.org    3276


 COMPLETED: WELLCOME

# Zenodo

In [72]:
Zenodo_df, Zenodo_summary = get_server_data("Zenodo")


 SERVER ANALYSIS: ZENODO
  > Files found:    1
  > Raw records:    166786
  > Cleaned shape:  (166786, 90)
  > Unique DOIs:    166786
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.5281/zenodo    166786
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
zenodo.org    166786
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.5281    166786
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
None    166786
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Zenodo                                                                            163441
The Five Principles of Organized Complexity.                                         385
Landon Puritz                                                              

In [73]:
Zenodo_parent_df, Zenodo_parent_summary = analyze_parent_data("Zenodo", full_parent_df)


 CONSOLIDATED ANALYSIS: ZENODO
  > Total Parent Groups:     67,908
  > Total Versions Found:    161,491
  > Unique Parent DOIs:      67,908
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Zenodo:
-----------------------------------
  1 version(s):     6,942 groups
  2 version(s):     48,920 groups
  3 version(s):     6,665 groups
  4 version(s):     2,732 groups
  5 version(s):     942 groups
  6 version(s):     539 groups
  7 version(s):     290 groups
  8 version(s):     204 groups
  9 version(s):     122 groups
  10 version(s):    109 groups
  11 version(s):    71 groups
  12 version(s):    66 groups
  13 version(s):    35 groups
  14 version(s):    41 groups
  15 version(s):    18 groups
  16 version(s):    22 groups
  17 version(s):    20 groups
  18 version(s):    14 groups
  19 version(s):    12 groups
  20 version(s):    15 groups
  21 version(s):    5 groups
  22 version(s):    10 groups
  23 version(s):    11 groups
  24 versio